In [14]:
import SparseArrays: SparseMatrixCSC
import LinearAlgebra: dot, Symmetric
import JuMP: VariableRef

function dot(A::SparseMatrixCSC{Float64, Int64}, B::Symmetric{VariableRef, Matrix{VariableRef}})
    return sum(A[i,j] * B[j,i] for i in 1:size(A,1), j in 1:size(A,2))
end

dot (generic function with 65 methods)

# Generating extremal behaviors with relaxed non-signaling and finding classical and quantum values of Bell-like inequalities in the cycle scenario

## Overview

In this notebook, we study multipartite Bell scenarios with **relaxed no-signaling constraints**, where some parties are allowed to communicate their inputs according to a fixed communication structure.

The main goal is to search for **linear inequalities** that are satisfied by all classical behaviours compatible with this relaxed communication model, but can still be violated by standard **quantum behaviours without communication**.

In this sense, we are interested in understanding whether quantum correlations can outperform classical strategies even when the classical model is strengthened by allowing a limited form of input communication.

We focus on the tripartite cyclic communication scenario and use tools from convex geometry, linear programming, and quantum optimization to derive and test candidate inequalities.

## The tripartite Cycle Scenario

We focus on the *tripartite cycle scenario*, a Bell scenario involving three parties: Alice, Bob, and Charlie. There are two distinct communication structures that define this scenario:

- **Clockwise cycle**:  
  - Alice is allowed to communicate her input to Bob.  
  - Bob is allowed to communicate his input to Charlie.  
  - Charlie is allowed to communicate his input to Alice.  
  Its classical model is given by:
  $$p(a,b,c|x,y,z) = \sum_{\lambda} p(\lambda) p(a|x,z,\lambda)p(b|y,x,\lambda)p(c|z,y,\lambda) $$

- **Counter-clockwise cycle**:  
  - Alice is allowed to communicate her input to Charlie.
  - Bob is allowed to communicate his input to Alice.
  - Charlie is allowed to communicate his input to Bob.  
  Its classical model is given by:
  $$p(a,b,c|x,y,z) = \sum_{\lambda} p(\lambda) p(a|x,y,\lambda)p(b|y,z,\lambda)p(c|z,x,\lambda) $$

The *cycle scenario* is defined as the **convex combination** of these two configurations. Effectively, in terms of extremal points, this results in a set of deterministic behaviours that is the **union** of the deterministic points corresponding to the clockwise and counter-clockwise cycle structures.


# Generation of Extremal Behaviours

## Introduction

This section construct multipartite extremal behaviours within the context of Bell scenarios with non-signalling relaxations. We will define Julia functions that generate all possible deterministic behaviours conditioned on subsets of inputs, respecting consistency conditions when input indices are shared between different parts.

## Function: Generate Single-Party Deterministic Behaviours (Sera que vale fazer geral assim e não já direto o tripartido?)

In a general causal scenarios, behaviours may be factorizable over different subsets of parties. For example, a behaviour may be expressed as:

$$
p(o_1 | x_{i_1}, ..., x_{i_r}) \cdot p(o_2 | x_{j_1}, ..., x_{j_s}) \cdot \ldots \cdot p(o_{n_p} | x_{k_1}, ..., x_{k_t}) 
$$

where $\{i_1, ..., i_r\}$, $\{j_1, ..., j_s\}$,..., $\{k_1, ..., k_t\}$ are subsets of the total set of parties $\{1, ..., n_p\}$. 

Here, we implement a function to generate all deterministic behaviours of the form:

$$
p(o_{1} | x_{i_1}, ..., x_{i_k})
$$

for a given subset of parties $\{x_{i_1}, ..., x_{i_k}\}$.

In [15]:
function generate_deterministic_behaviours(n_o::Int, n_p::Int, n_m::Int, subset_indices::Vector{Int})
""" generate_deterministic_behaviours(n_o, n_p, n_m, subset_indices)

Generates all deterministic behaviours for a subset of parties.

Arguments:
- n_o: Number of outcomes.
- n_p: Total number of parties.
- n_m: Number of measurements per party.
- subset_indices: Vector of integers indicating the subset of parties involved.

Returns:
- A list of vectors representing all deterministic behaviours.
"""
    k = length(subset_indices)
    n_inputs = n_m^k

    deterministic_behaviours = []

    # For each input combination
    for _ in 1:(n_o^n_inputs)
        push!(deterministic_behaviours, zeros(Int, n_o * n_inputs))
    end

    # Build all deterministic assignments
    assignments = Iterators.product(ntuple(_ -> 1:n_o, n_inputs)...)

    idx = 1
    for assignment in assignments
        behaviour = zeros(Int, n_o * n_inputs)
        for (input_idx, outcome) in enumerate(assignment)
            start_idx = (input_idx - 1) * n_o + 1
            behaviour[start_idx + outcome - 1] = 1
        end
        deterministic_behaviours[idx] = behaviour
        idx += 1
    end

    return deterministic_behaviours
end

generate_deterministic_behaviours (generic function with 1 method)

---

## Function: Generate Multipartite Deterministic Behaviours
This function constructs the full set of multipartite deterministic behaviours, ensuring that shared input indices are consistently assigned.


In [16]:
function generate_multipartite_deterministic_behaviours(n_o::Int, n_p::Int, n_m::Int, subsets::Vector{Vector{Int}})
    # Step 1: Identify all unique variables
    all_indices = unique(vcat(subsets...))
    total_inputs = n_m ^ length(all_indices)

    # Mapping each variable in subset -> position in all_indices
    index_map = Dict()
    for (pos, idx) in enumerate(all_indices)
        index_map[idx] = pos
    end

    # Step 2: Generate all global input combinations
    input_combinations = collect(Iterators.product(ntuple(_ -> 1:n_m, length(all_indices))...))

    # Step 3: Generate all conditional deterministic behaviours for each part
    behaviours_list = []
    for subset in subsets
        behaviours = generate_deterministic_behaviours(n_o, n_p, n_m, subset)
        push!(behaviours_list, behaviours)
    end

    # Step 4: Generate Cartesian product of all deterministic assignments
    all_assignments = Iterators.product(behaviours_list...)

    # List to store global behaviours
    multipartite_behaviours = []

    # Step 5: for each combination of deterministic behaviours
    for assignment in all_assignments
        # assignment is a tupla: (behaviour_1, behaviour_2, ..., behaviour_n_p)
        behaviour_vector = []
        
        # For each global combination of inputs
        for global_input in input_combinations
            # For each part, take the relevant input
            probs = []
            for (part_idx, subset) in enumerate(subsets)
                # Map subset to position in global_input
                subset_input_indices = [index_map[idx] for idx in subset]
                subset_input = global_input[subset_input_indices]

                # Determine what the position in the corresponding deterministic behavior is
                input_pos = 1
                for (k, val) in enumerate(subset_input)
                    input_pos += (val - 1) * n_m ^ (length(subset_input) - k)
                end

                # Get the deterministic behavior of the part
                behaviour = assignment[part_idx]
                
                # Determine which outcome is "active" (prob=1) for this combination of inputs
                prob_slice = behaviour[((input_pos - 1) * n_o + 1):(input_pos * n_o)]
                push!(probs, prob_slice)
            end

            # Now for each combination of possible outcomes, multiply the probabilities
            for outcome_tuple in Iterators.product(ntuple(_ -> 1:n_o, length(subsets))...)
                prob = 1.0
                for (part_idx, outcome) in enumerate(outcome_tuple)
                    prob *= probs[part_idx][outcome]
                end
                push!(behaviour_vector, prob)
            end
        end

        # Save this global behavior
        push!(multipartite_behaviours, behaviour_vector)
    end

    return multipartite_behaviours
end


generate_multipartite_deterministic_behaviours (generic function with 1 method)

## Example Usage (não sei se compensa esse aqui. Se pá deixa para quando for usar. Também acho que vale melhorar os nomes. E talvez fazer para o ciclico, não só o horario e anti-horario separadamente).

In [17]:
n_p = 3               # 3 parties
n_o = 2               # 2 outcomes for each part
n_m = 2               # 2 inputs per measurement

# Conditional dependencies for each party (clock-wise)
subsets_list = [[3, 1], [1, 2], [2, 3]]

global_behaviours = generate_multipartite_deterministic_behaviours(n_o, n_p, n_m, subsets_list)

println("Total multipartite deterministic behaviours generated: ", length(global_behaviours))


Total multipartite deterministic behaviours generated: 4096


In [18]:
behaviours_matrix = reduce(vcat, [Float64.(b)' for b in global_behaviours]);

# Deriving Bell-Type Inequalities from Extremal Nonsignalling Points

In this subsection, we derive Bell-type inequalities designed to detect correlations that cannot be reproduced by the classical set with cyclic communication. he approach is based on selecting extremal points of the nonsignalling (NS) politope that do **not admit a classical model with cyclic communication**.

For a fixed extremal NS behavior $\mathbf{q}$, we search for a coefficient vector $\mathbf{c}$ such that the linear functional

$$
I(\mathbf{p}) = \mathbf{c} \cdot \mathbf{p}
$$

satisfies the following properties:

1. $I(\mathbf{d}) \geq \beta_{\mathrm{cl}}$ for every deterministic behavior $\mathbf{d}$ of the classical cyclic polytope. 
2. $I(\mathbf{q})$ is **as small as possible**, making $\mathbf{q}$ the optimal point of the optimization;

The resulting inequality is therefore valid for every classical behavior compatible with cyclic communication, while being violated—or even maximally minimized—by the selected extremal NS behavior.

Since the extremal points of the NS polytope are available in correlator notation, our first task is to construct a function that converts them into full conditional probabilities.

### Converting Correlator Notation to Full Probabilities

In the tripartite case, assuming nonsignalling, the relation between **correlator notation** and **full probability notation** is given by:

$$
p(a,b,c|x,y,z) = \frac{1}{8} \left[ 1 + a \langle A_{x} \rangle + b \langle B_{y} \rangle + c \langle C_{z} \rangle + ab \langle AB_{xy} \rangle + ac \langle AC_{xz} \rangle + bc \langle BC_{yz} \rangle + abc \langle ABC_{xyz} \rangle \right]
$$

where $a,b,c \in \{-1,1\}$ and $x,y,z \in \{0,1\}$.

We will now implement a function that takes as input a vector of **26 correlators** and returns the corresponding vector of **64 conditional probabilities**.

In [19]:
"""
convert_correlators_to_probabilities(correlators::Vector{Float64}) -> Vector{Float64}

Given a non-signaling behavior expressed in correlator notation, this function returns
the equivalent full probability distribution.

# Arguments
- `correlators::Vector{Float64}`: A vector of length 26 representing:
    [ <A_0>, <A_1>, <B_0>, <B_1>, <C_0>, <C_1>,
    <AB_00>, <AB_01>, <AB_10>, <AB_11>,
    <AC_00>, <AC_01>, <AC_10>, <AC_11>,
    <BC_00>, <BC_01>, <BC_10>, <BC_11>,
    <ABC_000>, <ABC_001>, <ABC_010>, <ABC_011>,
    <ABC_100>, <ABC_101>, <ABC_110>, <ABC_111> ]

# Returns
- `probabilities::Vector{Float64}`: A vector of length 64 corresponding to
  p(a,b,c | x,y,z) for a,b,c ∈ {-1,1}, x,y,z ∈ {0,1}.
"""
function convert_correlators_to_probabilities(correlators::Vector{Float64})
    @assert length(correlators) == 26

    probabilities = zeros(Float64, 64)

    A = correlators[1:2]
    B = correlators[3:4]
    C = correlators[5:6]

    AB = reshape(correlators[7:10], 2, 2)
    AC = reshape(correlators[11:14], 2, 2)
    BC = reshape(correlators[15:18], 2, 2)
    ABC = reshape(correlators[19:26], 2, 2, 2)

    idx = 1
    for x in 0:1, y in 0:1, z in 0:1
        for a in [-1,1], b in [-1,1], c in [-1,1]
            probabilities[idx] = 1/8 * (
                1 + 
                a * A[x+1] + 
                b * B[y+1] + 
                c * C[z+1] +
                a*b * AB[y+1, x+1] +
                a*c * AC[z+1, x+1] +
                b*c * BC[z+1, y+1] +
                a*b*c * ABC[z+1, y+1, x+1]
            )
            idx += 1
        end
    end

    return probabilities
end


convert_correlators_to_probabilities

## Applying the Conversion to the Nonclassical Extremal Nonsignalling Behaviors that do not has a classical cyclic-signal model

We now apply the function `convert_correlators_to_probabilities` to the **extremal nonsignalling behaviors** that **do not admit any classical model with cyclic communication** (the BDAG-cyclic scenario).


In [20]:
# Correlator vector with 26 entries ordered as:
# [ <A_0>, <A_1>, <B_0>, <B_1>, <C_0>, <C_1>,
#   <AB_00>, <AB_01>, <AB_10>, <AB_11>,
#   <AC_00>, <AC_01>, <AC_10>, <AC_11>,
#   <BC_00>, <BC_01>, <BC_10>, <BC_11>,
#   <ABC_000>, <ABC_001>, <ABC_010>, <ABC_011>,
#   <ABC_100>, <ABC_101>, <ABC_110>, <ABC_111> ]

NS_class29_correlator_vector = [
    1/3, 1/3, 1/3, 1/3, 1/3, 1/3,         # <A_x>, <B_y>, <C_z>
    1, -1/3, -1/3, -1/3,                   # <AB_{xy}>
    1, -1/3, -1/3, -1/3,                   # <AC_{xz}>
    1, -1/3, -1/3, -1/3,                   # <BC_{yz}>
    1/3, 1/3, 1/3,                         # <ABC_{000}>, <ABC_{001}>, <ABC_{010}>
    -1, 1/3, -1, -1, -1                      # <ABC_{011}> ... <ABC_{111}>
];
NS_class29_fullprob_vector = convert_correlators_to_probabilities(NS_class29_correlator_vector);

In [25]:
function clean_vector(v::Vector{Float64}; tol::Float64 = 1e-12)
    return [abs(x) < tol ? 0.0 : x for x in v]
end

NS_class29_fullprob_vector = clean_vector(NS_class29_fullprob_vector);

## Constructing the Bell-like Inequality from the Extremal NS Point

Having fixed the target nonsignalling extremal behavior, we now construct a
linear inequality that is tailored to this point. The idea is to search for a
linear functional $c \cdot p$ that is **minimized** at the chosen NS
extremal point $p_{\text{NS}}$, while being **lower-bounded** on all
deterministic behaviors of the classical cyclic polytope.

To achieve this, we solve a linear optimization problem where the coefficients
$c$ are chosen so that every classical cyclic deterministic behavior
$p_{\text{CC}}^{(i)}$ satisfies
$$
c \cdot p_{\text{CC}}^{(i)} \ge 1,
$$
while simultaneously minimizing the value $c \cdot p_{\text{NS}}$. The
resulting hyperplane defines a Bell-like inequality that is violated (or
optimized) by the selected NS extremal point and satisfied by all classical
behaviors. This procedure yields a facet candidate of the corresponding
classical polytope.


## Acho que compensa fazer direto para todos os behaviours deterministicos do ciclico também.

In [22]:
using JuMP, GLPK

# Nonsignalling extremal behaviours that are not classical with this type of communication
p29 = NS_class29_fullprob_vector  # example p29 vector

psl = behaviours_matrix  # each row is a p_sl^i vector

# Example data: dimension and number of p_sl vectors
d = length(p29)  # dimension of the vectors
n = size(behaviours_matrix)[1]   # number of p_sl^i vectors

# Create optimization model
model = Model(GLPK.Optimizer)

# Define variables: c >= 0
@variable(model, c[1:d] >= 0)

# Objective: minimize dot(c, p29)
@objective(model, Min, sum(c[j] * p29[j] for j in 1:d))

# Constraints: dot(c, psl_i) >= 1 for all i
for i in 1:n
    @constraint(model, sum(c[j] * psl[i, j] for j in 1:d) >= 1)
end

# Printing the prepared optimization model
print(model)

# Solve the problem
optimize!(model)

# Check solver status
println("Solver status: ", termination_status(model))

# Print optimal solution
println("Optimal solution for c:")
for j in 1:d
    println("c[$j] = ", value(c[j]))
end

# Print optimal objective value
println("Optimal objective value: ", objective_value(model))

Solver status: OPTIMAL
Optimal solution for c:
c[1] = 0.0
c[2] = 0.9999999999999999
c[3] = 1.0000000000000002
c[4] = 0.4999999999999999
c[5] = 1.0
c[6] = 0.4999999999999999
c[7] = 0.5000000000000001
c[8] = 0.0
c[9] = 0.5000000000000006
c[10] = 0.0
c[11] = 0.49999999999999967
c[12] = 0.5000000000000007
c[13] = 0.0
c[14] = 1.0
c[15] = -2.220446049250313e-16
c[16] = 0.0
c[17] = 1.5543122344752192e-15
c[18] = 0.0
c[19] = 0.0
c[20] = 1.0
c[21] = 0.5000000000000003
c[22] = 0.0
c[23] = 0.4999999999999999
c[24] = 0.0
c[25] = 0.9999999999999984
c[26] = 1.0
c[27] = 1.0
c[28] = 0.0
c[29] = 0.0
c[30] = 0.4999999999999999
c[31] = 0.0
c[32] = 0.4999999999999999
c[33] = 3.3306690738754696e-16
c[34] = 0.5000000000000007
c[35] = 1.0000000000000004
c[36] = 0.0
c[37] = 0.0
c[38] = 0.5000000000000003
c[39] = 1.0
c[40] = 0.0
c[41] = 0.4999999999999995
c[42] = 1.0
c[43] = 0.0
c[44] = 0.0
c[45] = 1.0
c[46] = 0.0
c[47] = 0.5000000000000003
c[48] = 0.5000000000000001
c[49] = 0.9999999999999991
c[50] = 0.0
c[51

In [27]:
c29_opt = value.(c);
c29_opt = clean_vector(c29_opt);
M_29 = reshape(c29_opt_round, 8, 8)'

8×8 adjoint(::Matrix{Float64}) with eltype Float64:
 0.0  1.0  1.0  0.5  1.0  0.5  0.5  0.0
 0.5  0.0  0.5  0.5  0.0  1.0  0.0  0.0
 0.0  0.0  0.0  1.0  0.5  0.0  0.5  0.0
 1.0  1.0  1.0  0.0  0.0  0.5  0.0  0.5
 0.0  0.5  1.0  0.0  0.0  0.5  1.0  0.0
 0.5  1.0  0.0  0.0  1.0  0.0  0.5  0.5
 1.0  0.0  0.0  0.5  1.0  0.0  0.0  0.5
 1.0  0.5  0.5  0.0  0.5  0.0  0.0  1.0

In [28]:
c29_opt_round= round.(c29_opt, digits=4);

### In this part we confirm that the inequality found in fact has a minimum value at the classical extremal points equal to 1.

In [12]:
using LinearAlgebra  # Import required for the dot function

# Scenario parameters
n_p = 3
n_o_list = 2   # Number of outcomes for each party
n_m = 2   # Number of inputs for each party

# Conditional dependencies for each party (clock-wise)
subsets_list = [[3, 1], [1, 2], [2, 3]]

# Generate all multipartite deterministic behaviours
det_behaviours = generate_multipartite_deterministic_behaviours(n_o_list, n_p, n_m, subsets_list)

# Compute the inner product of each behaviour with Ineq_clock
inner_products = [dot(c29_opt_round, behaviour) for behaviour in det_behaviours]

# Find the minimum value of the functional over all deterministic behaviours
min_value = minimum(inner_products)

println("Minimum value of the inequality over clock-wise deterministic behaviours: ", min_value)


Minimum value of the inequality over clock-wise deterministic behaviours: 1.0


### Here we check whether the inequality found is faceted or not

In [13]:
using LinearAlgebra

# Identify behaviours that saturate the inequality
min_behaviours = [b for (i, b) in enumerate(det_behaviours) if abs(inner_products[i] - min_value) < 10^(-10)]

# Stack the behaviours into a matrix for affine dimension computation
M = hcat(min_behaviours...)'

# Initialize the difference matrix
diff_matrix = zeros(size(M))

# Subtract the first behaviour from each row explicitly
for i in 1:size(M, 1)
    diff_matrix[i, :] = M[i, :] - M[1, :]
end

# The affine dimension is the rank of this matrix
affine_dim = rank(diff_matrix)

println("Affine dimension of the saturating set: ", affine_dim)

Affine dimension of the saturating set: 41


### It is not a facet, facet must have affine dimension equal to 43

## Deriving new inequalities by mixing with classical behaviours
Now we apply the ideas from *“Bell Inequalities from No-Signalling Distributions”*, where instead of using only the extremal nonsignalling point, we mix it with local deterministic points lying on the same face of the polytope. This allows us to explore whether such combinations can generate facet inequalities with a bigger gap.

In [14]:
using JuMP, GLPK, LinearAlgebra

"""
    generate_inequality_from_perturbed_point(p29, behaviours_matrix, det_behaviours, j, k, epsilon)

Given:
- `p29`: probability vector (target distribution)
- `behaviours_matrix`: matrix where each row is a p_sl^i vector (for the constraints)
- `det_behaviours`: list (vectors) of deterministic behaviors
- `j`, `k`: indices of deterministic behaviors that saturate the inequality
- `epsilon`: small positive parameter

The function:
1. Creates a new perturbed point: `(1 - 3ε/2)p29 + ε*p_j + (ε/2)*p_k`
2. Solves the linear program analogous to the original to generate the new inequality.
3. Returns the coefficient vector `c_opt`, the minimum value `obj_value`, and the solved model.

"""
function generate_inequality_from_perturbed_point(p29, behaviours_matrix, det_behaviours, j, k, epsilon)
    # Dimensions
    d = length(p29)
    n = size(behaviours_matrix, 1)

    # Take the deterministic points
    p_j = det_behaviours[j]
    p_k = det_behaviours[k]

    # Creates the disturbed point
    p_perturbed = (1 - 3 * epsilon / 2) * p29 .+ epsilon * p_j .+ (epsilon / 2) * p_k

    # Create the optimization model
    model = Model(GLPK.Optimizer)
    @variable(model, c[1:d] >= 0)
    @objective(model, Min, sum(c[j] * p_perturbed[j] for j in 1:d))

    # Restrictions: dot(c, p_sl^i) >= 1
    for i in 1:n
        @constraint(model, sum(c[j] * behaviours_matrix[i, j] for j in 1:d) >= 1)
    end

    # Solve the problem
    optimize!(model)

    # Extracts solution and optimal value.
    c_opt = value.(c)
    obj_value = objective_value(model)

    return c_opt, obj_value, model, p_perturbed
end

generate_inequality_from_perturbed_point

In [15]:
epsilon = 1e-3
j = 1
k = 134
c_new, val_new, model_new, p_new = generate_inequality_from_perturbed_point(p29, behaviours_matrix, det_behaviours, j, k, epsilon);

# Print optimal objective value
println("Optimal objective value: ", val_new)

Optimal objective value: 0.50525


In [16]:
c29_disturbed1 = value.(c_new);
c29_disturbed_round1 = round.(c29_disturbed1, digits=4);
p29_disturbed1 = p_new;
M29_disturbed1 = reshape(c29_disturbed_round1, 8, 8)'

8×8 adjoint(::Matrix{Float64}) with eltype Float64:
 0.0  1.0  1.0  0.5  1.0  0.5  0.5  0.0
 0.0  0.0  0.5  0.5  1.0  1.0  0.0  0.0
 0.0  1.0  0.0  1.0  0.5  0.0  0.5  0.0
 1.0  0.0  1.0  0.0  0.0  0.5  0.0  0.5
 0.0  0.5  1.0  0.0  0.0  0.5  1.0  0.0
 1.0  1.0  0.0  0.0  0.0  0.0  0.5  0.5
 1.0  0.0  0.0  0.5  1.0  0.0  0.0  0.5
 1.0  0.5  0.5  0.0  0.5  0.0  0.0  1.0

In [17]:
dot(c29_disturbed1, p29_disturbed1)

0.50525

In [18]:
M_29 - M29_disturbed1

8×8 Matrix{Float64}:
  0.0   0.0   0.0  0.0   0.0  0.0   0.0  0.0
  0.5   0.0   0.0  0.0  -1.0  0.0  -0.0  0.0
  0.0  -1.0   0.0  0.0   0.0  0.0   0.0  0.0
  0.0   1.0   0.0  0.0   0.0  0.0   0.0  0.0
  0.0   0.0   0.0  0.0   0.0  0.0   0.0  0.0
 -0.5   0.0   0.0  0.0   1.0  0.0   0.0  0.0
  0.0   0.0  -0.0  0.0   0.0  0.0   0.0  0.0
  0.0   0.0   0.0  0.0   0.0  0.0   0.0  0.0

### In this part we confirm that the new inequality found in fact has a minimum value at the classical extremal points equal to 1.

### Here we check whether the new inequality found is faceted or not

In [19]:
using LinearAlgebra


# Compute the inner product of each behaviour with Ineq_clock
inner_products = [dot(c29_disturbed_round1, behaviour) for behaviour in det_behaviours]

# Find the minimum value of the functional over all deterministic behaviours
min_value = minimum(inner_products)

println("Minimum value of the inequality over clock-wise deterministic behaviours: ", min_value)

# Identify behaviours that saturate the inequality
min_behaviours_new = [b for (i, b) in enumerate(det_behaviours) if abs(inner_products[i] - min_value) < 10^(-10)]

# Stack the behaviours into a matrix for affine dimension computation
M = hcat(min_behaviours_new...)'

# Initialize the difference matrix
diff_matrix = zeros(size(M))

# Subtract the first behaviour from each row explicitly
for i in 1:size(M, 1)
    diff_matrix[i, :] = M[i, :] - M[1, :]
end

# The affine dimension is the rank of this matrix
affine_dim = rank(diff_matrix)

println("Affine dimension of the saturating set: ", affine_dim)

Minimum value of the inequality over clock-wise deterministic behaviours: 1.0
Affine dimension of the saturating set: 43


In [20]:
using ProgressMeter

function find_facets(p29, behaviours_matrix, det_behaviours;
                     epsilon=1e-3, target_dim=43)

    n_det = length(det_behaviours)

    facet_results = []
    seen_c = Set{Vector{Float64}}()
    
    total = n_det * (n_det - 1)
    p = Progress(total; dt=0.2)

    for j in 1:n_det
        for k in 1:n_det

            # Atualiza barra mostrando (j,k)
            update!(p; desc="j=$j, k=$k")

            if j == k
                continue
            end

            try
                c_new, val_new, p_new =
                    generate_inequality_from_perturbed_point(
                        p29, behaviours_matrix, det_behaviours, j, k, epsilon
                    )

                c_round = round.(c_new, digits=6)

                # Verificar repetido
                if any(==(c_round), seen_c)
                    continue
                else
                    push!(seen_c, c_round)
                end

                inner_products = [dot(c_round, det_behaviours[i]) for i in 1:n_det]
                min_value = 1

                min_behaviers = [
                    b for (i, b) in enumerate(det_behaviours)
                    if abs(inner_products[i] - min_value) < 1e-6
                ]

                if length(min_behaviers) <= 1
                    continue
                end

                M = hcat(min_behaviers...)'
                diff_matrix = zeros(size(M))

                for i in 1:size(M, 1)
                    diff_matrix[i, :] = M[i, :] - M[1, :]
                end

                affine_dim = rank(diff_matrix)

                if affine_dim == target_dim
                    push!(facet_results, (j, k, val_new))
                    println("✅ Facet found with j=$j, k=$k (dim = $affine_dim), opt = $val_new")
                end

            catch e
                @warn "Error processing (j=$j, k=$k): $e"
            end
        end
    end

    println("\nTotal number of facets found: ", length(facet_results))
    return facet_results, seen_c
end


find_facets (generic function with 1 method)

In [21]:
epsilon = 1e-3
# facet_pairs = find_facets(p29, behaviours_matrix, det_behaviours; epsilon=epsilon, target_dim=43)

0.001

In [22]:
epsilon = 1e-3
j = 5
k = 2731
c_new, val_new, model_new, p_new = generate_inequality_from_perturbed_point(p29, behaviours_matrix, det_behaviours, j, k, epsilon);

# Print optimal objective value
println("Optimal objective value: ", val_new)

Optimal objective value: 0.5037499999999999


In [23]:
c29_disturbed2 = value.(c_new);
c29_disturbed_round2 = round.(c29_disturbed2, digits=4);
M29_disturbed2 = reshape(c29_disturbed_round2, 8, 8)'

8×8 adjoint(::Matrix{Float64}) with eltype Float64:
 0.0  1.0  1.0  0.5  1.0  0.5  0.5  0.0
 0.5  0.0  1.0  0.5  0.0  1.0  0.5  0.0
 0.5  0.0  0.0  1.0  1.0  0.5  0.5  0.0
 0.5  1.0  1.0  0.0  1.0  0.0  0.0  0.5
 0.5  1.0  0.0  0.5  0.0  0.5  1.0  0.0
 0.5  1.0  1.0  0.0  1.0  0.0  0.0  0.5
 0.5  1.0  1.0  0.0  1.0  0.0  0.0  0.5
 1.0  0.5  0.5  0.0  0.5  0.0  0.0  1.0

In [24]:
M29_disturbed1 - M29_disturbed2

8×8 Matrix{Float64}:
  0.0   0.0   0.0   0.0   0.0   0.0   0.0  0.0
 -0.5   0.0  -0.5   0.0   1.0   0.0  -0.5  0.0
 -0.5   1.0   0.0   0.0  -0.5  -0.5   0.0  0.0
  0.5  -1.0   0.0   0.0  -1.0   0.5   0.0  0.0
 -0.5  -0.5   1.0  -0.5   0.0   0.0   0.0  0.0
  0.5   0.0  -1.0   0.0  -1.0   0.0   0.5  0.0
  0.5  -1.0  -1.0   0.5   0.0   0.0   0.0  0.0
  0.0   0.0   0.0   0.0   0.0   0.0   0.0  0.0

In [25]:
epsilon = 1e-2
j = 1
k = 137
c_new, val_new, model_new, p_new = generate_inequality_from_perturbed_point(p29, behaviours_matrix, det_behaviours, j, k, epsilon);

# Print optimal objective value
println("Optimal objective value: ", val_new)

Optimal objective value: 0.5374999999999999


In [26]:
c29_disturbed3 = value.(c_new);
c29_disturbed_round3 = round.(c29_disturbed3, digits=4);
M29_disturbed3 = reshape(c29_disturbed_round3, 8, 8)'

8×8 adjoint(::Matrix{Float64}) with eltype Float64:
 0.0  1.0   1.0  0.5   1.0   0.5  0.5  0.0
 0.0  0.0   0.5  0.5   1.0   1.0  0.0  0.0
 0.0  1.0   0.0  1.0   0.5  -0.0  0.5  0.0
 1.0  0.0   1.0  0.0   0.0   0.5  0.0  0.5
 0.0  0.5   1.0  0.0   0.0   0.5  1.0  0.0
 1.0  1.0   0.0  0.0  -0.0   0.0  0.5  0.5
 1.0  0.0  -0.0  0.5   1.0   0.0  0.0  0.5
 1.0  0.5   0.5  0.0   0.5   0.0  0.0  1.0

In [27]:
M29_disturbed3 - M29_disturbed1

8×8 Matrix{Float64}:
 0.0  0.0   0.0  0.0   0.0   0.0  0.0  0.0
 0.0  0.0   0.0  0.0   0.0   0.0  0.0  0.0
 0.0  0.0   0.0  0.0   0.0  -0.0  0.0  0.0
 0.0  0.0   0.0  0.0   0.0   0.0  0.0  0.0
 0.0  0.0   0.0  0.0   0.0   0.0  0.0  0.0
 0.0  0.0   0.0  0.0  -0.0   0.0  0.0  0.0
 0.0  0.0  -0.0  0.0   0.0   0.0  0.0  0.0
 0.0  0.0   0.0  0.0   0.0   0.0  0.0  0.0

### É igual ao que encontrou antes.

In [28]:
epsilon = 1e-2
j = 1
k = 2731
c_new, val_new, model_new, p_new = generate_inequality_from_perturbed_point(p29, behaviours_matrix, det_behaviours, j, k, epsilon);

# Print optimal objective value
println("Optimal objective value: ", val_new) 

Optimal objective value: 0.5375


In [29]:
c29_disturbed4 = value.(c_new);
c29_disturbed_round4 = round.(c29_disturbed4, digits=4);
M29_disturbed4 = reshape(c29_disturbed_round4, 8, 8)'

8×8 adjoint(::Matrix{Float64}) with eltype Float64:
 0.0  1.0  1.0  0.5  1.0  0.5  0.5  0.0
 0.5  0.0  1.0  0.5  0.0  1.0  0.5  0.0
 0.5  0.0  0.0  1.0  1.0  0.5  0.5  0.0
 0.5  1.0  1.0  0.0  1.0  0.0  0.0  0.5
 0.5  1.0  0.0  0.5  0.0  0.5  1.0  0.0
 0.5  1.0  1.0  0.0  1.0  0.0  0.0  0.5
 0.5  1.0  1.0  0.0  1.0  0.0  0.0  0.5
 1.0  0.5  0.5  0.0  0.5  0.0  0.0  1.0

In [30]:
M29_disturbed4 - M29_disturbed1

8×8 Matrix{Float64}:
  0.0   0.0   0.0   0.0   0.0   0.0   0.0  0.0
  0.5   0.0   0.5   0.0  -1.0   0.0   0.5  0.0
  0.5  -1.0   0.0   0.0   0.5   0.5   0.0  0.0
 -0.5   1.0   0.0   0.0   1.0  -0.5   0.0  0.0
  0.5   0.5  -1.0   0.5   0.0   0.0   0.0  0.0
 -0.5   0.0   1.0   0.0   1.0   0.0  -0.5  0.0
 -0.5   1.0   1.0  -0.5   0.0   0.0   0.0  0.0
  0.0   0.0   0.0   0.0   0.0   0.0   0.0  0.0

In [31]:
M29_disturbed4 - M29_disturbed2

8×8 Matrix{Float64}:
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0

### É igual ao que encontrou antes.

## Upper Bounds on the Quantum Value using NPA Hierarchy

In [32]:
function format_sums(M)
    comb = [
        (0,0,0),
        (0,0,1),
        (0,1,0),
        (0,1,1),
        (1,0,0),
        (1,0,1),
        (1,1,0),
        (1,1,1)
    ]

    for i in 1:size(M,1)
        x4, x5, x6 = comb[i]
        termos = []

        for j in 1:size(M,2)
            coef = M[i, j]
            if coef != 0.0
                x1, x2, x3 = comb[j]
                if coef == 1.0
                    push!(termos, "p($x1,$x2,$x3,$x4,$x5,$x6)")
                else
                    push!(termos, "$(coef)*p($x1,$x2,$x3,$x4,$x5,$x6)")
                end
            end
        end

        if isempty(termos)
            expr = "0"
        else
            expr = join(termos, "+")
        end

        println("S_$(x4)$(x5)$(x6) = $expr\n")
    end
end

format_sums (generic function with 1 method)

In [33]:
format_sums(M_29)

S_000 = p(0,0,1,0,0,0)+p(0,1,0,0,0,0)+0.5*p(0,1,1,0,0,0)+p(1,0,0,0,0,0)+0.5*p(1,0,1,0,0,0)+0.5*p(1,1,0,0,0,0)

S_001 = 0.5*p(0,0,0,0,0,1)+0.5*p(0,1,0,0,0,1)+0.5*p(0,1,1,0,0,1)+p(1,0,1,0,0,1)

S_010 = p(0,1,1,0,1,0)+0.5*p(1,0,0,0,1,0)+0.5*p(1,1,0,0,1,0)

S_011 = p(0,0,0,0,1,1)+p(0,0,1,0,1,1)+p(0,1,0,0,1,1)+0.5*p(1,0,1,0,1,1)+0.5*p(1,1,1,0,1,1)

S_100 = 0.5*p(0,0,1,1,0,0)+p(0,1,0,1,0,0)+0.5*p(1,0,1,1,0,0)+p(1,1,0,1,0,0)

S_101 = 0.5*p(0,0,0,1,0,1)+p(0,0,1,1,0,1)+p(1,0,0,1,0,1)+0.5*p(1,1,0,1,0,1)+0.5*p(1,1,1,1,0,1)

S_110 = p(0,0,0,1,1,0)+0.5*p(0,1,1,1,1,0)+p(1,0,0,1,1,0)+0.5*p(1,1,1,1,1,0)

S_111 = p(0,0,0,1,1,1)+0.5*p(0,0,1,1,1,1)+0.5*p(0,1,0,1,1,1)+0.5*p(1,0,0,1,1,1)+p(1,1,1,1,1,1)



In [34]:
M_29_double = 2*M_29

8×8 Matrix{Float64}:
 0.0  2.0   2.0  1.0  2.0  1.0   1.0  0.0
 1.0  0.0   1.0  1.0  0.0  2.0  -0.0  0.0
 0.0  0.0   0.0  2.0  1.0  0.0   1.0  0.0
 2.0  2.0   2.0  0.0  0.0  1.0   0.0  1.0
 0.0  1.0   2.0  0.0  0.0  1.0   2.0  0.0
 1.0  2.0   0.0  0.0  2.0  0.0   1.0  1.0
 2.0  0.0  -0.0  1.0  2.0  0.0   0.0  1.0
 2.0  1.0   1.0  0.0  1.0  0.0   0.0  2.0

In [35]:
format_sums(M_29_double)

S_000 = 2.0*p(0,0,1,0,0,0)+2.0*p(0,1,0,0,0,0)+p(0,1,1,0,0,0)+2.0*p(1,0,0,0,0,0)+p(1,0,1,0,0,0)+p(1,1,0,0,0,0)

S_001 = p(0,0,0,0,0,1)+p(0,1,0,0,0,1)+p(0,1,1,0,0,1)+2.0*p(1,0,1,0,0,1)

S_010 = 2.0*p(0,1,1,0,1,0)+p(1,0,0,0,1,0)+p(1,1,0,0,1,0)

S_011 = 2.0*p(0,0,0,0,1,1)+2.0*p(0,0,1,0,1,1)+2.0*p(0,1,0,0,1,1)+p(1,0,1,0,1,1)+p(1,1,1,0,1,1)

S_100 = p(0,0,1,1,0,0)+2.0*p(0,1,0,1,0,0)+p(1,0,1,1,0,0)+2.0*p(1,1,0,1,0,0)

S_101 = p(0,0,0,1,0,1)+2.0*p(0,0,1,1,0,1)+2.0*p(1,0,0,1,0,1)+p(1,1,0,1,0,1)+p(1,1,1,1,0,1)

S_110 = 2.0*p(0,0,0,1,1,0)+p(0,1,1,1,1,0)+2.0*p(1,0,0,1,1,0)+p(1,1,1,1,1,0)

S_111 = 2.0*p(0,0,0,1,1,1)+p(0,0,1,1,1,1)+p(0,1,0,1,1,1)+p(1,0,0,1,1,1)+2.0*p(1,1,1,1,1,1)



In [36]:
using QuantumNPA
PA = projector(1, 1:2, 1:2, full=true);
PB = projector(2, 1:2, 1:2, full=true);
PC = projector(3, 1:2, 1:2, full=true);

p(a,b,c,x,y,z) = PA[a+1,x+1]*PB[b+1,y+1]*PC[c+1,z+1]

S_000 = p(0,0,1,0,0,0)+p(0,1,0,0,0,0)+0.5*p(0,1,1,0,0,0)+p(1,0,0,0,0,0)+0.5*p(1,0,1,0,0,0)+0.5*p(1,1,0,0,0,0)

S_001 = 0.5*p(0,0,0,0,0,1)+0.5*p(0,1,0,0,0,1)+0.5*p(0,1,1,0,0,1)+p(1,0,1,0,0,1)

S_010 = p(0,1,1,0,1,0) + 0.5*p(1,0,0,0,1,0) + 0.5*p(1,1,0,0,1,0)

S_011 = p(0,0,0,0,1,1) + p(0,0,1,0,1,1) + p(0,1,0,0,1,1) + 0.5*p(1,0,1,0,1,1) + 0.5*p(1,1,1,0,1,1)

S_100 = 0.5*p(0,0,1,1,0,0) + p(0,1,0,1,0,0) + 0.5*p(1,0,1,1,0,0) + p(1,1,0,1,0,0)

S_101 = 0.5*p(0,0,0,1,0,1) + p(0,0,1,1,0,1) + p(1,0,0,1,0,1) + 0.5*p(1,1,0,1,0,1) + 0.5*p(1,1,1,1,0,1)

S_110 = p(0,0,0,1,1,0) + 0.5*p(0,1,1,1,1,0) + p(1,0,0,1,1,0) + 0.5*p(1,1,1,1,1,0)

S_111 = p(0,0,0,1,1,1) + 0.5*p(0,0,1,1,1,1) + 0.5*p(0,1,0,1,1,1) + 0.5*p(1,0,0,1,1,1) + p(1,1,1,1,1,1)

S = S_000 + S_001 + S_010 + S_011 + S_100 + S_101 + S_110 + S_111 


2.5 Id + 1.5 PA1|1 - 1.5 PA1|2 + 1.5 PB1|1 - 1.5 PB1|2 + 1.5 PC1|1 - 1.5 PC1|2 - 1.5 PA1|1 PB1|1 - 1.5 PA1|1 PC1|1 + 1.5 PA1|1 PC1|2 + 1.5 PA1|2 PB1|1 + 1.5 PA1|2 PB1|2 + 1.5 PA1|2 PC1|2 - 1.5 PB1|1 PC1|1 + 1.5 PB1|2 PC1|1 + 1.5 PB1|2 PC1|2 - 1.5 PA1|1 PB1|1 PC1|1 + 1.5 PA1|1 PB1|1 PC1|2 + PA1|1 PB1|2 PC1|1 - PA1|1 PB1|2 PC1|2 - 1.5 PA1|2 PB1|1 PC1|2 - 1.5 PA1|2 PB1|2 PC1|2

In [37]:

using QuantumNPA
using SparseArrays, LinearAlgebra, JuMP

# Sobrescreve o método dentro do namespace do QuantumNPA
@eval QuantumNPA begin
    import SparseArrays: SparseMatrixCSC
    import JuMP: VariableRef

    function dot(A::SparseMatrixCSC{Float64, Int64}, B::Symmetric{VariableRef, Matrix{VariableRef}})
        # Converte a matriz esparsa em densa só durante o cálculo do produto
        return sum(A[i, j] * B[j, i] for i in 1:size(A, 1), j in 1:size(A, 2))
    end
end

dot (generic function with 1 method)

In [38]:
npa_min(S,4), npa_min(S,3), npa_min(S,2)

(0.9753462816549989, 0.9755570797800306, 0.9361981682677669)

### tem uma violação, não é tão grande mais tem!)

In [39]:
format_sums(M29_disturbed1)

S_000 = p(0,0,1,0,0,0)+p(0,1,0,0,0,0)+0.5*p(0,1,1,0,0,0)+p(1,0,0,0,0,0)+0.5*p(1,0,1,0,0,0)+0.5*p(1,1,0,0,0,0)

S_001 = 0.5*p(0,1,0,0,0,1)+0.5*p(0,1,1,0,0,1)+p(1,0,0,0,0,1)+p(1,0,1,0,0,1)

S_010 = p(0,0,1,0,1,0)+p(0,1,1,0,1,0)+0.5*p(1,0,0,0,1,0)+0.5*p(1,1,0,0,1,0)

S_011 = p(0,0,0,0,1,1)+p(0,1,0,0,1,1)+0.5*p(1,0,1,0,1,1)+0.5*p(1,1,1,0,1,1)

S_100 = 0.5*p(0,0,1,1,0,0)+p(0,1,0,1,0,0)+0.5*p(1,0,1,1,0,0)+p(1,1,0,1,0,0)

S_101 = p(0,0,0,1,0,1)+p(0,0,1,1,0,1)+0.5*p(1,1,0,1,0,1)+0.5*p(1,1,1,1,0,1)

S_110 = p(0,0,0,1,1,0)+0.5*p(0,1,1,1,1,0)+p(1,0,0,1,1,0)+0.5*p(1,1,1,1,1,0)

S_111 = p(0,0,0,1,1,1)+0.5*p(0,0,1,1,1,1)+0.5*p(0,1,0,1,1,1)+0.5*p(1,0,0,1,1,1)+p(1,1,1,1,1,1)



In [40]:
using QuantumNPA
PA = projector(1, 1:2, 1:2, full=true);
PB = projector(2, 1:2, 1:2, full=true);
PC = projector(3, 1:2, 1:2, full=true);

p(a,b,c,x,y,z) = PA[a+1,x+1]*PB[b+1,y+1]*PC[c+1,z+1]

S_000 = p(0,0,1,0,0,0)+p(0,1,0,0,0,0)+0.5*p(0,1,1,0,0,0)+p(1,0,0,0,0,0)+0.5*p(1,0,1,0,0,0)+0.5*p(1,1,0,0,0,0)

S_001 = 0.5*p(0,1,0,0,0,1)+0.5*p(0,1,1,0,0,1)+p(1,0,0,0,0,1)+p(1,0,1,0,0,1)

S_010 = p(0,0,1,0,1,0)+p(0,1,1,0,1,0)+0.5*p(1,0,0,0,1,0)+0.5*p(1,1,0,0,1,0)

S_011 = p(0,0,0,0,1,1)+p(0,1,0,0,1,1)+0.5*p(1,0,1,0,1,1)+0.5*p(1,1,1,0,1,1)

S_100 = 0.5*p(0,0,1,1,0,0)+p(0,1,0,1,0,0)+0.5*p(1,0,1,1,0,0)+p(1,1,0,1,0,0)

S_101 = p(0,0,0,1,0,1)+p(0,0,1,1,0,1)+0.5*p(1,1,0,1,0,1)+0.5*p(1,1,1,1,0,1)

S_110 = p(0,0,0,1,1,0)+0.5*p(0,1,1,1,1,0)+p(1,0,0,1,1,0)+0.5*p(1,1,1,1,1,0)

S_111 = p(0,0,0,1,1,1)+0.5*p(0,0,1,1,1,1)+0.5*p(0,1,0,1,1,1)+0.5*p(1,0,0,1,1,1)+p(1,1,1,1,1,1)

S = S_000 + S_001 + S_010 + S_011 + S_100 + S_101 + S_110 + S_111 

2.5 Id + 1.5 PA1|1 - 1.5 PA1|2 + 1.5 PB1|1 - 1.5 PB1|2 + 1.5 PC1|1 - 1.5 PC1|2 - 1.5 PA1|1 PB1|1 - 1.5 PA1|1 PC1|1 + 1.5 PA1|1 PC1|2 + 1.5 PA1|2 PB1|1 + 1.5 PA1|2 PB1|2 + 1.5 PA1|2 PC1|2 - 1.5 PB1|1 PC1|1 + 1.5 PB1|2 PC1|1 + 1.5 PB1|2 PC1|2 - 1.5 PA1|1 PB1|1 PC1|1 - 1.5 PA1|2 PB1|2 PC1|2

In [41]:
npa_min(S,4), npa_min(S,3), npa_min(S,2)

(0.9406541466968572, 0.9407531100155304, 0.8345364834651317)

### Resultado see-saw 0.94245, acho que tá uma boa aproximação

## Tem violação e é um pouco melhor

In [42]:
format_sums(M29_disturbed2)

S_000 = p(0,0,1,0,0,0)+p(0,1,0,0,0,0)+0.5*p(0,1,1,0,0,0)+p(1,0,0,0,0,0)+0.5*p(1,0,1,0,0,0)+0.5*p(1,1,0,0,0,0)

S_001 = 0.5*p(0,0,0,0,0,1)+p(0,1,0,0,0,1)+0.5*p(0,1,1,0,0,1)+p(1,0,1,0,0,1)+0.5*p(1,1,0,0,0,1)

S_010 = 0.5*p(0,0,0,0,1,0)+p(0,1,1,0,1,0)+p(1,0,0,0,1,0)+0.5*p(1,0,1,0,1,0)+0.5*p(1,1,0,0,1,0)

S_011 = 0.5*p(0,0,0,0,1,1)+p(0,0,1,0,1,1)+p(0,1,0,0,1,1)+p(1,0,0,0,1,1)+0.5*p(1,1,1,0,1,1)

S_100 = 0.5*p(0,0,0,1,0,0)+p(0,0,1,1,0,0)+0.5*p(0,1,1,1,0,0)+0.5*p(1,0,1,1,0,0)+p(1,1,0,1,0,0)

S_101 = 0.5*p(0,0,0,1,0,1)+p(0,0,1,1,0,1)+p(0,1,0,1,0,1)+p(1,0,0,1,0,1)+0.5*p(1,1,1,1,0,1)

S_110 = 0.5*p(0,0,0,1,1,0)+p(0,0,1,1,1,0)+p(0,1,0,1,1,0)+p(1,0,0,1,1,0)+0.5*p(1,1,1,1,1,0)

S_111 = p(0,0,0,1,1,1)+0.5*p(0,0,1,1,1,1)+0.5*p(0,1,0,1,1,1)+0.5*p(1,0,0,1,1,1)+p(1,1,1,1,1,1)



In [43]:
using QuantumNPA
PA = projector(1, 1:2, 1:2, full=true);
PB = projector(2, 1:2, 1:2, full=true);
PC = projector(3, 1:2, 1:2, full=true);

p(a,b,c,x,y,z) = PA[a+1,x+1]*PB[b+1,y+1]*PC[c+1,z+1]

S_000 = p(0,0,1,0,0,0)+p(0,1,0,0,0,0)+0.5*p(0,1,1,0,0,0)+p(1,0,0,0,0,0)+0.5*p(1,0,1,0,0,0)+0.5*p(1,1,0,0,0,0)

S_001 = 0.5*p(0,0,0,0,0,1)+p(0,1,0,0,0,1)+0.5*p(0,1,1,0,0,1)+p(1,0,1,0,0,1)+0.5*p(1,1,0,0,0,1)

S_010 = 0.5*p(0,0,0,0,1,0)+p(0,1,1,0,1,0)+p(1,0,0,0,1,0)+0.5*p(1,0,1,0,1,0)+0.5*p(1,1,0,0,1,0)

S_011 = 0.5*p(0,0,0,0,1,1)+p(0,0,1,0,1,1)+p(0,1,0,0,1,1)+p(1,0,0,0,1,1)+0.5*p(1,1,1,0,1,1)

S_100 = 0.5*p(0,0,0,1,0,0)+p(0,0,1,1,0,0)+0.5*p(0,1,1,1,0,0)+0.5*p(1,0,1,1,0,0)+p(1,1,0,1,0,0)

S_101 = 0.5*p(0,0,0,1,0,1)+p(0,0,1,1,0,1)+p(0,1,0,1,0,1)+p(1,0,0,1,0,1)+0.5*p(1,1,1,1,0,1)

S_110 = 0.5*p(0,0,0,1,1,0)+p(0,0,1,1,1,0)+p(0,1,0,1,1,0)+p(1,0,0,1,1,0)+0.5*p(1,1,1,1,1,0)

S_111 = p(0,0,0,1,1,1)+0.5*p(0,0,1,1,1,1)+0.5*p(0,1,0,1,1,1)+0.5*p(1,0,0,1,1,1)+p(1,1,1,1,1,1)

S = S_000 + S_001 + S_010 + S_011 + S_100 + S_101 + S_110 + S_111 

2.5 Id + 1.5 PA1|1 - 1.5 PA1|2 + 1.5 PB1|1 - 1.5 PB1|2 + 1.5 PC1|1 - 1.5 PC1|2 - 1.5 PA1|1 PB1|1 - 1.5 PA1|1 PC1|1 + 1.5 PA1|1 PC1|2 + 1.5 PA1|2 PB1|1 + 3.0 PA1|2 PB1|2 + 3.0 PA1|2 PC1|2 - 1.5 PB1|1 PC1|1 + 1.5 PB1|2 PC1|1 + 3.0 PB1|2 PC1|2 - 1.5 PA1|1 PB1|1 PC1|1 + 1.5 PA1|1 PB1|1 PC1|2 + 1.5 PA1|1 PB1|2 PC1|1 - 3.0 PA1|1 PB1|2 PC1|2 + 1.5 PA1|2 PB1|1 PC1|1 - 3.0 PA1|2 PB1|1 PC1|2 - 3.0 PA1|2 PB1|2 PC1|1 - 1.5 PA1|2 PB1|2 PC1|2

In [44]:
npa_min(S,4), npa_min(S,3), npa_min(S,2)

(1.0001003158733208, 1.0000857070223608, 1.0000014806183393)

### Não tem violação

## Logo, só encontramos duas facetas com o metodo de misturar e apenas uma delas tem violação quantica.

---

# Testing a Linear Functional over Deterministic Behaviours of the clockwise cycle determinist behaviours

In this section, we will evaluate the value of a **linear functional** over each of the deterministic behaviours of the tripartite clockwise cycle scenario. This procedure is equivalent to testing a **Bell inequality**: the linear functional represents the inequality, and the evaluation over deterministic points allows us to identify its maximal value within the *classical* set **CLASSICO NÃO É BOM, USAR O NOME QUE O PLANET USA**.

Given that we have constructed all possible deterministic behaviours consistent with the specified input-output dependencies, the next step is to define the linear functional and compute its value over each deterministic behaviour.

The process is as follows:

1. **Define the linear functional**: represented as a vector with the same dimension as each deterministic behaviour. This vector encodes the coefficients of the inequality.

2. **Evaluate the functional**: for each deterministic behaviour, compute the scalar product between the functional vector and the deterministic vector. This yields the value of the inequality for that deterministic point.

3. **Identify the maximal value**: the supremum of all these evaluations corresponds to the maximal classical  bound of the inequality in the clockwise cycle scenario.

The functional that we will text is given by:

\begin{equation}
        \begin{aligned}
        %
       S_{clock} =  &p(001|000) + p(010|000) +2 p(011|000) + p(100|000) + 2 p(101|000) +2 p(110|000) \\
        &+2 p(010|001) +2 p(011|001) + p(100|001)  + p(101|001) \\
        &+ p(001|010) + p(011|010) + 2 p(100|010) +2 p(110|010)  \\
        &+ p(000|011) + p(010|011) +2 p(101|011) +2 p(111|011)\\
        &+ 2p(001|100) + p(010|100) +2 p(101|100) + p(110|100)   \\
        &+ p(000|101) + p(001|101) +2 p(110|101)  +2 p(111|101) \\
        &+ p(000|110) +  2 p(011|110) + p(100|110) +2 p(111|110) \\
        &+2 p(000|111) + p(011|111) + p(101|111) + p(110|111)  +2 p(111|111) \geq 2.
        %
        \label{eq:inequality_probs}
        %
    \end{aligned}
\end{equation}

In the next cell, we will define the linear functional and implement the code to perform these evaluations.


### Coeficientes of the S$_{clock}$ Bell inequality:

In [45]:
Ineq_clock = [0 1 1 2 1 2 2 0;
         0 0 2 2 1 1 0 0;
         0 1 0 1 2 0 2 0;
         1 0 1 0 0 2 0 2;
         0 2 1 0 0 2 1 0;
         1 1 0 0 0 0 2 2;
         1 0 0 2 1 0 0 2;
         2 0 0 1 0 1 1 2]

Ineq_clock = float.(Ineq_clock)

Ineq_clock_vec = vec(Ineq_clock');

In [46]:
Ineq_clock

8×8 Matrix{Float64}:
 0.0  1.0  1.0  2.0  1.0  2.0  2.0  0.0
 0.0  0.0  2.0  2.0  1.0  1.0  0.0  0.0
 0.0  1.0  0.0  1.0  2.0  0.0  2.0  0.0
 1.0  0.0  1.0  0.0  0.0  2.0  0.0  2.0
 0.0  2.0  1.0  0.0  0.0  2.0  1.0  0.0
 1.0  1.0  0.0  0.0  0.0  0.0  2.0  2.0
 1.0  0.0  0.0  2.0  1.0  0.0  0.0  2.0
 2.0  0.0  0.0  1.0  0.0  1.0  1.0  2.0

In [47]:
Ineq_clock - 2*M29_disturbed1

8×8 Matrix{Float64}:
  0.0  -1.0  -1.0   1.0  -1.0   1.0   1.0  0.0
  0.0   0.0   1.0   1.0  -1.0  -1.0   0.0  0.0
  0.0  -1.0   0.0  -1.0   1.0   0.0   1.0  0.0
 -1.0   0.0  -1.0   0.0   0.0   1.0   0.0  1.0
  0.0   1.0  -1.0   0.0   0.0   1.0  -1.0  0.0
 -1.0  -1.0   0.0   0.0   0.0   0.0   1.0  1.0
 -1.0   0.0   0.0   1.0  -1.0   0.0   0.0  1.0
  0.0  -1.0  -1.0   1.0  -1.0   1.0   1.0  0.0

In [48]:
Ineq_clock - 2*M29_disturbed2

8×8 Matrix{Float64}:
  0.0  -1.0  -1.0   1.0  -1.0   1.0   1.0  0.0
 -1.0   0.0   0.0   1.0   1.0  -1.0  -1.0  0.0
 -1.0   1.0   0.0  -1.0   0.0  -1.0   1.0  0.0
  0.0  -2.0  -1.0   0.0  -2.0   2.0   0.0  1.0
 -1.0   0.0   1.0  -1.0   0.0   1.0  -1.0  0.0
  0.0  -1.0  -2.0   0.0  -2.0   0.0   2.0  1.0
  0.0  -2.0  -2.0   2.0  -1.0   0.0   0.0  1.0
  0.0  -1.0  -1.0   1.0  -1.0   1.0   1.0  0.0

# Eu pensei que essa Ineq_clock seria igual a M29_disturbed, tem alguma treta kkk

# Ah tá de boa, ela tem que ser diferente mesmo já que quando avalia ela nos pontos que incluem os anti-horario essa desigualdade Ineq_clock tem valor zero, enquanto que a M29_disturbed tem valor 1. Mas ai a dúvida que fica é que essa Ineq_clock parece ser uma faceta que também poderia ser resultado da minimização. Entretanto, a gente não chega nela fazendo o esquema de misturar com ruído os outros pontos. 

## Evaluating the Inequality over the Clock-Wise Deterministic Behaviours

Now that we have defined the inequality vector `Ineq_clock`, we can proceed to evaluate its value over all deterministic behaviours corresponding to the tripartite clock-wise cycle scenario.

### Scenario specifications:

- **Number of parties**: $n_p = 3$
- **Outcomes per measurement**: $n_o = 2$
- **Inputs per measurement**: $n_m = 2$
- **Conditional dependencies**:
  - Party 1: depends on inputs $\{1, 3\}$
  - Party 2: depends on inputs $\{1, 2\}$
  - Party 3: depends on inputs $\{2, 3\}$

For each deterministic behaviour generated under these conditions, we compute the **inner product** between the behaviour vector and the inequality vector `Ineq_clock`. This yields the value of the Bell expression for each deterministic point.

The result is a list of scalar values representing the inequality evaluated over all possible deterministic behaviours in this scenario.

Finally, we will extract the **minimum** value of this functional over all deterministic behaviours.


The following cell implements this computation.

In [49]:
using LinearAlgebra  # Import required for the dot function

# Scenario parameters
n_p = 3
n_o_list = 2   # Number of outcomes for each party
n_m = 2   # Number of inputs for each party

# Conditional dependencies for each party (clock-wise)
subsets_list = [[3, 1], [1, 2], [2, 3]]

# Generate all multipartite deterministic behaviours
det_behaviours = generate_multipartite_deterministic_behaviours(n_o_list, n_p, n_m, subsets_list)

# Compute the inner product of each behaviour with Ineq_clock
inner_products = [dot(Ineq_clock_vec, behaviour) for behaviour in det_behaviours]

# Find the minimum value of the functional over all deterministic behaviours
min_value = minimum(inner_products)

println("Minimum value of the inequality over clock-wise deterministic behaviours: ", min_value)


Minimum value of the inequality over clock-wise deterministic behaviours: 2.0


## Checking if the Inequality Defines a Facet

To verify whether the inequality vector `Ineq_clock` represents a **facet** of the polytope defined by the deterministic behaviours, we need to check the **affine dimension** of the set of behaviours that **saturate** the inequality, i.e., those for which the inner product with `Ineq_clock` achieves the **minimal value**.

### Procedure:

1. Identify all deterministic behaviours that minimize the value of the inequality.
2. Compute the **affine dimension** of this set.
3. Compare the affine dimension with the expected dimension for a facet.

If the affine dimension of the saturating set is equal to $\text{dim}(\mathcal{P}) - 1$, where $\mathcal{P}$ is the polytope of all admissible behaviours, then the inequality defines a **facet** of this polytope.

The following cell implements this verification.


In [50]:
using LinearAlgebra

# Identify behaviours that saturate the inequality
min_behaviours = [b for (i, b) in enumerate(det_behaviours) if inner_products[i] == min_value]

# Stack the behaviours into a matrix for affine dimension computation
M = hcat(min_behaviours...)'

# Initialize the difference matrix
diff_matrix = zeros(size(M))

# Subtract the first behaviour from each row explicitly
for i in 1:size(M, 1)
    diff_matrix[i, :] = M[i, :] - M[1, :]
end

# The affine dimension is the rank of this matrix
affine_dim = rank(diff_matrix)

println("Affine dimension of the saturating set: ", affine_dim)

Affine dimension of the saturating set: 43


### OBS: Pelas minhas contas a dimensão do politopo é 44 (calculei com base na representação por correlatores calculando o numéro de coordenadas), então a dimensão de suas facetas é 43, como tá acima.

## Evaluating the Inequality over Additional Deterministic Sets

For completeness, we will also evaluate the inequality vector `Ineq_clock` over:

- The **counter-clockwise deterministic behaviours**, corresponding to the tripartite cycle scenario with reversed communication:  
  - Party 1 depends on $\{1, 2\}$  
  - Party 2 depends on $\{2, 3\}$  
  - Party 3 depends on $\{3, 1\}$  

- The **minimal Boring scenario deterministic behaviours**, corresponding to the tripartite cycle scenario with reversed communication:  
  - Party 1 depends on $\{1\}$  
  - Party 2 depends on $\{1, 2\}$  
  - Party 3 depends on $\{1, 2, 3\}$  
  It is known that this scenario has as a subset all **non-signaling** behaviours and hence all **quantum** behaviours.

For each set, we compute the inner product with `Ineq_clock` and extract the **minimum value** of the functional.  

As we will see below, the minimum of this functional for both sets is **0**, which is **lower** than the **quantum value** that we will compute next.


In [51]:
using LinearAlgebra  # Import required for the dot function

# Scenario parameters
n_p = 3
n_o_list = 2   # Number of outcomes for each party
n_m = 2   # Number of inputs for each party

# Conditional dependencies for each party (Counter clock-wise scenario)
subsets_list = [[2,1], [3, 2], [1,3]]

# Generate all multipartite deterministic behaviours
det_behaviours = generate_multipartite_deterministic_behaviours(n_o_list, n_p, n_m, subsets_list)

# Compute the inner product of each behaviour with Ineq_clock
inner_products = [dot(Ineq_clock_vec, behaviour) for behaviour in det_behaviours]

# Find the minimum value of the functional over all deterministic behaviours
min_value = minimum(inner_products)

println("Minimum value of the inequality over counter clock-wise deterministic behaviours: ", min_value)


Minimum value of the inequality over counter clock-wise deterministic behaviours: 0.0


In [52]:
using LinearAlgebra  # Import required for the dot function

# Scenario parameters
n_p = 3
n_o_list = 2   # Number of outcomes for each party
n_m = 2   # Number of inputs for each party

# Conditional dependencies for each party (Boring scenario)
subsets_list = [[1], [1, 2], [1, 2, 3]]

# Generate all multipartite deterministic behaviours
det_behaviours = generate_multipartite_deterministic_behaviours(n_o_list, n_p, n_m, subsets_list)

# Compute the inner product of each behaviour with Ineq_clock
inner_products = [dot(Ineq_clock, behaviour) for behaviour in det_behaviours]

# Find the minimum value of the functional over all deterministic behaviours
min_value = minimum(inner_products)

println("Minimum value of the inequality over det behaviours of a boring scenario: ", min_value)


Minimum value of the inequality over det behaviours of a boring scenario: 0.0


---

# Vou ter que fazer um see-saw para refazer essa parte de um exemplo quantico, já que parece que a desigualdade encontrada é melhor do que a que estava no artigo, considerando o ciclo (soma convexa de horario e anti-horario

### See-saw SDP for the Optimal Quantum Value (3 qubits, projective measurements)

We optimize the minimum value of a given linear Bell expression
$$
I(\rho, \{M_{a|x}\},\{N_{b|y}\},\{O_{c|z}\}) \;=\; \sum_{xyz}\sum_{abc} C_{abc|xyz}\, p(abc|xyz) =  \sum_{xyz}\sum_{abc} C_{abc|xyz} \mbox{Tr}\left\{\rho \, \cdot \,(M_{a|x} \otimes N_{b|y} \otimes O_{c|z}) \right\},
$$
over quantum realizations with **three qubits** (Alice, Bob, Charlie) and **binary projective measurements**.  
We use a see-saw algorithm:

1. initialize with random local projective measurements (Bloch vectors → rank-1 projectors);
2. fix measurements, solve an SDP for the global state $\rho$ (minimize $\mathrm{Tr}(\rho\,W)$);
3. fix $\rho$ and two parties' measurements; optimize the third party's local POVMs by SDP (impose $0 \preceq M_{0|x} \preceq I$, $M_{1|x}=I-M_{0|x}$);
4. optionally project the optimal local POVM elements to nearest rank-1 projectors (to enforce projectivity);
5. iterate steps 2–4 until convergence or maximum iterations.

The implementation below uses `JuMP` + `SCS` for the SDPs, and standard linear algebra utilities for partial traces and Kronecker products. The function `see_saw_optimize` receives the coefficient matrix `C` (8×8, rows indexed by (x,y,z), cols by (a,b,c)) and returns the final value, the optimized state ρ, and the local measurement projectors.


## Essa função eu fiz no Gemini, preciso entender!

In [53]:
using LinearAlgebra
using Convex
using SCS
using Random
using Printf # To format the output nicely

# ==============================================================================
# 1. Helper Functions (Math & Tracing)
# ==============================================================================

"""
    tensor_prod(A, B, C)
Returns the Kronecker product A ⊗ B ⊗ C.
"""
function tensor_prod(A, B, C)
    return kron(kron(A, B), C)
end

"""
    partial_trace_BC(rho, da, db, dc)
Traces out Bob and Charlie, returning the reduced state for Alice.
"""
function partial_trace_BC(rho, da, db, dc)
    rho_tensor = reshape(rho, (da, db, dc, da, db, dc))
    rho_A = zeros(ComplexF64, da, da)
    for i in 1:da, j in 1:da
        sum_val = 0.0 + 0.0im
        for k in 1:db, l in 1:dc
            sum_val += rho_tensor[i, k, l, j, k, l]
        end
        rho_A[i, j] = sum_val
    end
    return rho_A
end

# Helper operator
⊗(A, B) = kron(A, B)

# ==============================================================================
# 2. Optimization Core (SDP & Eigenvalue)
# ==============================================================================

"""
    optimize_measurements(party_index, psi, M_A, M_B, M_C, coeffs)
Optimizes measurements for one party using SDP via Convex.jl.
"""
function optimize_measurements(party_idx, psi, current_meas, coeffs_matrix)
    d = 2 
    num_inputs = 2 
    num_outputs = 2 
    
    rho = psi * psi'
    
    M_A, M_B, M_C = current_meas
    
    # Define Complex Variables using Convex explicitly
    E = [[Convex.ComplexVariable(d, d) for _ in 1:num_outputs] for _ in 1:num_inputs]
    
    objective_expr = 0
    
    # Build Objective Function
    for row in 0:7 
        x = ((row >> 2) & 1) + 1; y = ((row >> 1) & 1) + 1; z = (row & 1) + 1
        
        for col in 0:7 
            a = ((col >> 2) & 1) + 1; b = ((col >> 1) & 1) + 1; c = (col & 1) + 1
            
            coeff = coeffs_matrix[row+1, col+1]
            
            if coeff != 0
                if party_idx == 1 # Optimize Alice
                    op_BC = kron(M_B[y][b], M_C[z][c]) # Operator on Bob and Charlie
                    Id = Matrix{ComplexF64}(I, 2, 2) # Identity
                    rho_A_eff = partial_trace_BC(rho * (Id ⊗ op_BC), 2, 2, 2) # Effective state for Alice
                    objective_expr += coeff * real(tr(E[x][a] * rho_A_eff))
                    
                elseif party_idx == 2 # Optimize Bob
                    op_AC = kron(M_A[x][a], M_C[z][c]) # Operator on Alice and Charlie
                    rho_perm = permutedims(reshape(rho, (2,2,2,2,2,2)), (2,1,3, 5,4,6))
                    rho_perm = reshape(rho_perm, (8,8))
                    Id = Matrix{ComplexF64}(I, 2, 2)
                    rho_B_eff = partial_trace_BC(rho_perm * (Id ⊗ op_AC), 2, 2, 2)
                    objective_expr += coeff * real(tr(E[y][b] * rho_B_eff))
                    
                elseif party_idx == 3 # Optimize Charlie
                    op_AB = kron(M_A[x][a], M_B[y][b])
                    rho_perm = permutedims(reshape(rho, (2,2,2,2,2,2)), (3,1,2, 6,4,5))
                    rho_perm = reshape(rho_perm, (8,8))
                    Id = Matrix{ComplexF64}(I, 2, 2)
                    rho_C_eff = partial_trace_BC(rho_perm * (Id ⊗ op_AB), 2, 2, 2)
                    objective_expr += coeff * real(tr(E[z][c] * rho_C_eff))
                end
            end
        end
    end
    
    # Constraints (POVM)
    constraints = Convex.Constraint[]
    Id_d = Matrix{ComplexF64}(I, d, d)
    for i in 1:num_inputs
        push!(constraints, E[i][1] + E[i][2] == Id_d)
        push!(constraints, E[i][1] ⪰ 0)
        push!(constraints, E[i][2] ⪰ 0)
    end
    
    # Solve silently
    problem = minimize(objective_expr, constraints)
    Convex.solve!(problem, SCS.Optimizer; silent_solver = true)
    
    return [[evaluate(E[i][j]) for j in 1:num_outputs] for i in 1:num_inputs]
end

"""
    optimize_state(M_A, M_B, M_C, coeffs_matrix)
Finds the state |psi> that minimizes the Bell inequality (Eigenvalue problem).
"""
function optimize_state(M_A, M_B, M_C, coeffs_matrix)
    Bell_Operator = zeros(ComplexF64, 8, 8)
    
    for row in 0:7 
        x = ((row >> 2) & 1) + 1; y = ((row >> 1) & 1) + 1; z = (row & 1) + 1
        for col in 0:7 
            a = ((col >> 2) & 1) + 1; b = ((col >> 1) & 1) + 1; c = (col & 1) + 1
            
            coeff = coeffs_matrix[row+1, col+1]
            if coeff != 0
                term = tensor_prod(M_A[x][a], M_B[y][b], M_C[z][c])
                Bell_Operator += coeff * term
            end
        end
    end
    
    evals, evecs = eigen(Hermitian(Bell_Operator))
    min_val = evals[1] # Smallest eigenvalue
    best_psi = evecs[:, 1]
    
    return best_psi, min_val
end

# ==============================================================================
# 3. Single Run Logic
# ==============================================================================

"""
    seesaw_single_run(coeffs_matrix; max_iter=50, tol=1e-6, verbose=false)
Performs ONE complete See-Saw optimization starting from random conditions.
"""
function seesaw_single_run(coeffs_matrix; max_iter=50, tol=1e-6, verbose=false)
    d = 2
    
    # Random Initialization
    function rand_pvm()
        U = qr(randn(ComplexF64, d, d)).Q
        E0 = U[:,1] * U[:,1]'
        E1 = I(d) - E0
        return [[E0, E1], [E0, E1]] 
    end
    
    M_A = rand_pvm(); M_B = rand_pvm(); M_C = rand_pvm()
    psi = normalize(randn(ComplexF64, 8))
    
    current_val = 0.0
    
    for k in 1:max_iter
        prev_val = current_val
        
        M_A = optimize_measurements(1, psi, (M_A, M_B, M_C), coeffs_matrix)
        M_B = optimize_measurements(2, psi, (M_A, M_B, M_C), coeffs_matrix)
        M_C = optimize_measurements(3, psi, (M_A, M_B, M_C), coeffs_matrix)
        psi, current_val = optimize_state(M_A, M_B, M_C, coeffs_matrix)
        
        if verbose
            println("  -> Iter $k: $current_val")
        end
        
        if abs(current_val - prev_val) < tol && k > 1
            break
        end
    end
    
    return current_val, psi, (M_A, M_B, M_C)
end

# ==============================================================================
# 4. Global Optimization (Multiple Restarts)
# ==============================================================================

"""
    seesaw_tripartite(coeffs_matrix; num_runs=50)
Runs the See-Saw optimization `num_runs` times with random initializations
and returns the absolute best result found.
"""
function seesaw_tripartite(coeffs_matrix; num_runs=50)
    println("Starting Global Optimization with $num_runs restarts...")
    println("---------------------------------------------------")
    
    global_best_val = Inf # Start with a very high number
    global_best_psi = nothing
    global_best_meas = nothing
    
    for i in 1:num_runs
        # Run single optimization (silently)
        val, psi, meas = seesaw_single_run(coeffs_matrix, verbose=false)
        
        # Check if this run is better than the best so far
        if  val < global_best_val 
            global_best_val = val
            global_best_psi = psi
            global_best_meas = meas
            @printf "Run %02d: New Best Found! -> %.6f\n" i global_best_val
        else
            # Optional: just to show life signs
            # @printf "Run %02d: %.6f (not best)\n" i val 
        end
    end
    
    println("---------------------------------------------------")
    println("Optimization Finished.")
    println("Best Global Value: $global_best_val")
    
    return global_best_val, global_best_psi, global_best_meas
end

# ==============================================================================
# Example Usage
# ==============================================================================

# Dummy Coeffs (Mermin-like term just for testing)
coeffs = zeros(8, 8) 
coeffs[1, 1] = -1.0 # p(000|000)
coeffs[8, 8] = -1.0 # p(111|111)

# Run the global optimization
best_val, best_psi, best_meas = seesaw_tripartite(coeffs, num_runs=10) # Using 10 just to be fast in demo

println("\nFinal Result Check:")
println("Value: ", best_val)

Starting Global Optimization with 10 restarts...
---------------------------------------------------


┌ Warning: The keyword argument `silent_solver` in `Convex.solve!` has been deprecated in favor of `silent`.
└ @ Convex ~/.julia/packages/Convex/IPPoR/src/solution.jl:93


Run 01: New Best Found! -> -2.000000
Run 02: New Best Found! -> -2.000000
Run 10: New Best Found! -> -2.000000
---------------------------------------------------
Optimization Finished.
Best Global Value: -1.9999999970613263

Final Result Check:
Value: -1.9999999970613263


In [54]:
# ==============================================================================
# 5. Funções de Formatação e Visualização (Pretty Printing)
# ==============================================================================

"""
    clean_num(z, tol=1e-6)
Auxiliar: Trata partes reais e imaginárias separadamente, zera ruídos
e retorna explicitamente um número Complexo arredondado.
"""
function clean_num(z; tol=1e-6)
    # Extrai partes
    re = real(z)
    im = imag(z)

    # Zera se for ruído numérico (menor que a tolerância)
    re = abs(re) < tol ? 0.0 : re
    im = abs(im) < tol ? 0.0 : im

    # Arredonda cada componente individualmente
    re_rounded = round(re, digits=5)
    im_rounded = round(im, digits=5)

    # Reconstrói forçando o tipo Complexo
    return Complex(re_rounded, im_rounded)
end

"""
    print_results_formatted(val, rho, measurements)
Imprime o valor, o estado e as medições de forma limpa e organizada.
"""
function print_results_formatted(val, rho, measurements)
    println("="^60)
    println("             RESULTADOS DA OTIMIZAÇÃO")
    println("="^60)
    
    # 1. Valor da Violação
    @printf "\n🔹 VALOR MÍNIMO ENCONTRADO: %.6f\n" val
    
    # 2. Estado (Rho ou Psi)
    println("\n🔹 ESTADO OTIMIZADO (PSI):")
    # Aplica a limpeza em cada elemento
    # Se rho for um vetor (ket), map funciona igual.
    rho_clean = map(x -> clean_num(x), rho)
    
    # Mostra formatado pelo Julia
    show(stdout, "text/plain", rho_clean)
    println("\n")
    
    # 3. Medições
    party_names = ["Alice", "Bob", "Charlie"]
    
    println("🔹 MEDIÇÕES OTIMIZADAS (POVMs):")
    
    # measurements é uma Tupla (M_A, M_B, M_C)
    for (p_idx, M_party) in enumerate(measurements)
        p_name = party_names[p_idx]
        println("\n   📌 PARTY $p_idx: $p_name")
        println("   " * "-"^30)
        
        # M_party é um Vetor de Inputs
        for (in_idx, input_set) in enumerate(M_party)
            in_label = in_idx - 1 
            println("      📍 Input $in_label:")
            
            # input_set é um Vetor de Outputs (Matrizes)
            for (out_idx, mat) in enumerate(input_set)
                out_label = out_idx - 1
                println("         ↳ Output $out_label (Matriz):")
                
                # Limpa a matriz antes de printar
                mat_clean = map(x -> clean_num(x), mat)
                
                # Truque para indentar a matriz no print
                io = IOBuffer()
                show(IOContext(io, :limit=>true), "text/plain", mat_clean)
                str_mat = String(take!(io))
                
                # Adiciona identação visual
                for line in split(str_mat, "\n")
                    println("            " * line)
                end
                println("") 
            end
        end
    end
    println("="^60)
end

print_results_formatted

In [55]:
val, opt_psi, measurements = seesaw_tripartite(Ineq_clock);
print_results_formatted(val, opt_psi, measurements)

Starting Global Optimization with 50 restarts...
---------------------------------------------------
Run 01: New Best Found! -> 1.885022
Run 02: New Best Found! -> 1.885021
Run 07: New Best Found! -> 1.885019
Run 09: New Best Found! -> 1.885013
Run 11: New Best Found! -> 1.885009
Run 12: New Best Found! -> 1.884987
---------------------------------------------------
Optimization Finished.
Best Global Value: 1.8849873701518538
             RESULTADOS DA OTIMIZAÇÃO

🔹 VALOR MÍNIMO ENCONTRADO: 1.884987

🔹 ESTADO OTIMIZADO (PSI):
8-element Vector{ComplexF64}:
 -0.53463 + 0.10595im
 -0.25455 - 0.34783im
 -0.24561 - 0.29229im
  0.18422 + 0.08831im
 -0.27905 - 0.20892im
  0.16684 + 0.11008im
  0.08281 + 0.06442im
  0.39647 + 0.0im

🔹 MEDIÇÕES OTIMIZADAS (POVMs):

   📌 PARTY 1: Alice
   ------------------------------
      📍 Input 0:
         ↳ Output 0 (Matriz):
            2×2 Matrix{ComplexF64}:
             0.81311+0.0im      0.11822-0.37146im
             0.11822+0.37146im  0.18692+0.0im


In [56]:
val1, opt_psi1, measurements1 = seesaw_tripartite(M29_disturbed1);
print_results_formatted(val1, opt_psi1, measurements1)

Starting Global Optimization with 50 restarts...
---------------------------------------------------
Run 01: New Best Found! -> 0.942517
Run 02: New Best Found! -> 0.942513
Run 09: New Best Found! -> 0.942512
---------------------------------------------------
Optimization Finished.
Best Global Value: 0.9425117433208992
             RESULTADOS DA OTIMIZAÇÃO

🔹 VALOR MÍNIMO ENCONTRADO: 0.942512

🔹 ESTADO OTIMIZADO (PSI):
8-element Vector{ComplexF64}:
 -0.22842 + 0.35716im
 -0.17607 - 0.15039im
 -0.22303 - 0.24547im
 -0.29634 - 0.19145im
 -0.26315 - 0.26983im
 -0.25738 - 0.26308im
   -0.282 - 0.28486im
  0.30659 + 0.0im

🔹 MEDIÇÕES OTIMIZADAS (POVMs):

   📌 PARTY 1: Alice
   ------------------------------
      📍 Input 0:
         ↳ Output 0 (Matriz):
            2×2 Matrix{ComplexF64}:
              0.48858+0.0im      -0.24168+0.43757im
             -0.24168-0.43757im   0.51144+0.0im

         ↳ Output 1 (Matriz):
            2×2 Matrix{ComplexF64}:
             0.51142+0.0im      0.241

In [57]:
val2, opt_psi2, measurements2 = seesaw_tripartite(M29_disturbed2);
print_results_formatted(val2, opt_psi2, measurements2)

Starting Global Optimization with 50 restarts...
---------------------------------------------------
Run 01: New Best Found! -> 0.999878
Run 02: New Best Found! -> 0.999866
Run 15: New Best Found! -> 0.999856
Run 19: New Best Found! -> 0.999855
Run 26: New Best Found! -> 0.999841
---------------------------------------------------
Optimization Finished.
Best Global Value: 0.9998411045019457
             RESULTADOS DA OTIMIZAÇÃO

🔹 VALOR MÍNIMO ENCONTRADO: 0.999841

🔹 ESTADO OTIMIZADO (PSI):
8-element Vector{ComplexF64}:
  -0.3585 + 0.17865im
  -0.3486 + 0.11175im
 -0.34999 + 0.11135im
  -0.3316 + 0.05213im
 -0.35104 + 0.11105im
  -0.3325 + 0.05172im
 -0.33371 + 0.05115im
 -0.30856 + 0.0im

🔹 MEDIÇÕES OTIMIZADAS (POVMs):

   📌 PARTY 1: Alice
   ------------------------------
      📍 Input 0:
         ↳ Output 0 (Matriz):
            2×2 Matrix{ComplexF64}:
              0.22884+0.0im     -0.24611+0.0386im
             -0.24611-0.0386im   0.27113+0.0im

         ↳ Output 1 (Matriz):
    

## Vou fazer versões alternativas do programa onde eu fixo o estado e/ou as medições para ver se encontra realizações quanticas mais bonitinhas

In [58]:
# ==============================================================================
# Helper: Create W State
# ==============================================================================

"""
    get_w_state()
Returns the tripartite W state: 1/√3 (|001> + |010> + |100>)
"""
function get_w_state()
    # Indices in Julia (1-based):
    # |000> -> 1
    # |001> -> 2
    # |010> -> 3
    # |011> -> 4
    # |100> -> 5
    # ...
    psi = zeros(ComplexF64, 8)
    psi[2] = 1.0 # |001>
    psi[3] = 1.0 # |010>
    psi[5] = 1.0 # |100>
    return normalize(psi)
end

# ==============================================================================
# Helper: Calculate Value (since we removed optimize_state)
# ==============================================================================

"""
    calc_bell_value(psi, M_A, M_B, M_C, coeffs_matrix)
Calculates <psi| Bell_Operator |psi> explicitly.
"""
function calc_bell_value(psi, M_A, M_B, M_C, coeffs_matrix)
    val = 0.0 + 0.0im
    
    for row in 0:7 
        x = ((row >> 2) & 1) + 1; y = ((row >> 1) & 1) + 1; z = (row & 1) + 1
        for col in 0:7 
            a = ((col >> 2) & 1) + 1; b = ((col >> 1) & 1) + 1; c = (col & 1) + 1
            
            coeff = coeffs_matrix[row+1, col+1]
            if coeff != 0
                # Calculate expectation value <psi| A x B x C |psi>
                # Using the previously defined tensor_prod helper
                op = tensor_prod(M_A[x][a], M_B[y][b], M_C[z][c])
                val += coeff * (psi' * op * psi)
            end
        end
    end
    return real(val)
end

# ==============================================================================
# Single Run Logic (Fixed State)
# ==============================================================================

"""
    seesaw_w_state_run(coeffs_matrix; max_iter=50, tol=1e-6, verbose=false)
Optimizes ONLY measurements for the W state.
"""
function seesaw_w_state_run(coeffs_matrix; max_iter=50, tol=1e-6, verbose=false)
    d = 2
    
    # 1. Initialize State as W state (Fixed)
    psi = get_w_state()
    
    # 2. Random Initial Measurements
    function rand_pvm()
        U = qr(randn(ComplexF64, d, d)).Q
        E0 = U[:,1] * U[:,1]'
        E1 = I(d) - E0
        return [[E0, E1], [E0, E1]] 
    end
    M_A = rand_pvm(); M_B = rand_pvm(); M_C = rand_pvm()
    
    current_val = 0.0
    
    # 3. Optimization Loop (Only Measurements)
    for k in 1:max_iter
        prev_val = current_val
        
        # Optimize Measurements sequentially
        M_A = optimize_measurements(1, psi, (M_A, M_B, M_C), coeffs_matrix)
        M_B = optimize_measurements(2, psi, (M_A, M_B, M_C), coeffs_matrix)
        M_C = optimize_measurements(3, psi, (M_A, M_B, M_C), coeffs_matrix)
        
        # Calculate current value explicitly (No state optimization step)
        current_val = calc_bell_value(psi, M_A, M_B, M_C, coeffs_matrix)
        
        if verbose
            println("  -> Iter $k: $current_val")
        end
        
        if abs(current_val - prev_val) < tol && k > 1
            break
        end
    end
    
    return current_val, psi, (M_A, M_B, M_C)
end

# ==============================================================================
# Global Optimization (Fixed State)
# ==============================================================================

"""
    find_best_violation_w_state(coeffs_matrix; num_runs=100)
Runs the See-Saw multiple times for W state (varying initial measurements).
"""
function find_best_violation_w_state(coeffs_matrix; num_runs=100)
    println("Starting W-State Optimization with $num_runs restarts...")
    println("---------------------------------------------------")
    
    global_best_val = Inf
    # We don't need to store best_psi because it's always W state
    global_best_meas = nothing
    
    for i in 1:num_runs
        val, _, meas = seesaw_w_state_run(coeffs_matrix, verbose=false)
        
        if val < global_best_val 
            global_best_val = val
            global_best_meas = meas
            @printf "Run %02d: New Best Found! -> %.6f\n" i global_best_val
        end
    end
    
    println("---------------------------------------------------")
    println("Optimization Finished for W State.")
    println("Best Global Value: $global_best_val")
    
    return global_best_val, get_w_state(), global_best_meas
end




find_best_violation_w_state

In [59]:
val_2, psi_w, measurements_w = find_best_violation_w_state(Ineq_clock);
print_results_formatted(val_w, psi_2, measurements)

Starting W-State Optimization with 100 restarts...
---------------------------------------------------
Run 01: New Best Found! -> 3.181276
Run 06: New Best Found! -> 2.097891
Run 16: New Best Found! -> 2.097890
Run 55: New Best Found! -> 2.097890
---------------------------------------------------
Optimization Finished for W State.
Best Global Value: 2.097890175294073


LoadError: UndefVarError: `val_w` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [60]:
val_w1, psi_w1, measurements_w1 = find_best_violation_w_state(M29_disturbed1);
print_results_formatted(val_w1, psi_w1, measurements_w1)

Starting W-State Optimization with 100 restarts...
---------------------------------------------------
Run 01: New Best Found! -> 1.590651
Run 02: New Best Found! -> 1.048936
Run 10: New Best Found! -> 1.048936
Run 26: New Best Found! -> 1.048924
---------------------------------------------------
Optimization Finished for W State.
Best Global Value: 1.048923977365786
             RESULTADOS DA OTIMIZAÇÃO

🔹 VALOR MÍNIMO ENCONTRADO: 1.048924

🔹 ESTADO OTIMIZADO (PSI):
8-element Vector{ComplexF64}:
     0.0 + 0.0im
 0.57735 + 0.0im
 0.57735 + 0.0im
     0.0 + 0.0im
 0.57735 + 0.0im
     0.0 + 0.0im
     0.0 + 0.0im
     0.0 + 0.0im

🔹 MEDIÇÕES OTIMIZADAS (POVMs):

   📌 PARTY 1: Alice
   ------------------------------
      📍 Input 0:
         ↳ Output 0 (Matriz):
            2×2 Matrix{ComplexF64}:
              0.0+0.0im     -0.0-0.0im
             -0.0+0.0im  -1.0e-5+0.0im

         ↳ Output 1 (Matriz):
            2×2 Matrix{ComplexF64}:
             1.0+0.0im      0.0+0.0im
        

### Tentando com estado GHZ

In [61]:
# ==============================================================================
# Helper: Create GHZ State
# ==============================================================================

"""
    get_ghz_state()
Returns the tripartite GHZ state: 1/√2 (|000> + |111>)
"""
function get_ghz_state()
    # Indices in Julia (1-based):
    # |000> -> 1 (Binário 0)
    # ...
    # |111> -> 8 (Binário 7)
    
    psi = zeros(ComplexF64, 8)
    psi[1] = 1.0 # |000>
    psi[8] = 1.0 # |111>
    
    # A função normalize cuida do fator 1/sqrt(2) automaticamente
    return normalize(psi)
end

# ==============================================================================
# Single Run Logic (Fixed GHZ State)
# ==============================================================================

"""
    seesaw_ghz_state_run(coeffs_matrix; max_iter=50, tol=1e-6, verbose=false)
Optimizes ONLY measurements for the GHZ state.
"""
function seesaw_ghz_state_run(coeffs_matrix; max_iter=50, tol=1e-6, verbose=false)
    d = 2
    
    # 1. Initialize State as GHZ state (Fixed)
    psi = get_ghz_state()
    
    # 2. Random Initial Measurements
    function rand_pvm()
        U = qr(randn(ComplexF64, d, d)).Q
        E0 = U[:,1] * U[:,1]'
        E1 = I(d) - E0
        return [[E0, E1], [E0, E1]] 
    end
    M_A = rand_pvm(); M_B = rand_pvm(); M_C = rand_pvm()
    
    current_val = 0.0
    
    # 3. Optimization Loop (Only Measurements)
    for k in 1:max_iter
        prev_val = current_val
        
        # Optimize Measurements sequentially
        M_A = optimize_measurements(1, psi, (M_A, M_B, M_C), coeffs_matrix)
        M_B = optimize_measurements(2, psi, (M_A, M_B, M_C), coeffs_matrix)
        M_C = optimize_measurements(3, psi, (M_A, M_B, M_C), coeffs_matrix)
        
        # Calculate current value explicitly (using the helper defined previously)
        current_val = calc_bell_value(psi, M_A, M_B, M_C, coeffs_matrix)
        
        if verbose
            println("  -> Iter $k: $current_val")
        end
        
        if abs(current_val - prev_val) < tol && k > 1
            break
        end
    end
    
    return current_val, psi, (M_A, M_B, M_C)
end

# ==============================================================================
# Global Optimization (Fixed GHZ State)
# ==============================================================================

"""
    find_best_violation_ghz_state(coeffs_matrix; num_runs=100)
Runs the See-Saw multiple times for GHZ state.
"""
function find_best_violation_ghz_state(coeffs_matrix; num_runs=100)
    println("Starting GHZ-State Optimization with $num_runs restarts...")
    println("---------------------------------------------------")
    
    global_best_val = Inf
    global_best_meas = nothing
    
    for i in 1:num_runs
        val, _, meas = seesaw_ghz_state_run(coeffs_matrix, verbose=false)
        
        if val < global_best_val 
            global_best_val = val
            global_best_meas = meas
            @printf "Run %02d: New Best Found! -> %.6f\n" i global_best_val
        end
    end
    
    println("---------------------------------------------------")
    println("Optimization Finished for GHZ State.")
    println("Best Global Value: $global_best_val")
    
    return global_best_val, get_ghz_state(), global_best_meas
end

find_best_violation_ghz_state

In [62]:
val_ghz, psi_ghz, measurements_ghz = find_best_violation_ghz_state(Ineq_clock);
print_results_formatted(val_ghz, psi_ghz, measurements_ghz)

Starting GHZ-State Optimization with 100 restarts...
---------------------------------------------------
Run 01: New Best Found! -> 2.243309
Run 05: New Best Found! -> 2.230262
Run 17: New Best Found! -> 2.188669
Run 20: New Best Found! -> 2.013807
Run 64: New Best Found! -> 2.002094
---------------------------------------------------
Optimization Finished for GHZ State.
Best Global Value: 2.002094441185772
             RESULTADOS DA OTIMIZAÇÃO

🔹 VALOR MÍNIMO ENCONTRADO: 2.002094

🔹 ESTADO OTIMIZADO (PSI):
8-element Vector{ComplexF64}:
 0.70711 + 0.0im
     0.0 + 0.0im
     0.0 + 0.0im
     0.0 + 0.0im
     0.0 + 0.0im
     0.0 + 0.0im
     0.0 + 0.0im
 0.70711 + 0.0im

🔹 MEDIÇÕES OTIMIZADAS (POVMs):

   📌 PARTY 1: Alice
   ------------------------------
      📍 Input 0:
         ↳ Output 0 (Matriz):
            2×2 Matrix{ComplexF64}:
             1.00001+0.0im     1.0e-5-1.0e-5im
              1.0e-5+1.0e-5im     1.0+0.0im

         ↳ Output 1 (Matriz):
            2×2 Matrix{Comple

In [63]:
val_ghz1, psi_ghz1, measurements_ghz1 = find_best_violation_ghz_state(M29_disturbed1);
print_results_formatted(val_ghz1, psi_ghz1, measurements_ghz1)

Starting GHZ-State Optimization with 100 restarts...
---------------------------------------------------
Run 01: New Best Found! -> 1.109229
Run 08: New Best Found! -> 1.010745
Run 39: New Best Found! -> 1.002148
Run 90: New Best Found! -> 1.000935
---------------------------------------------------
Optimization Finished for GHZ State.
Best Global Value: 1.000935197070352
             RESULTADOS DA OTIMIZAÇÃO

🔹 VALOR MÍNIMO ENCONTRADO: 1.000935

🔹 ESTADO OTIMIZADO (PSI):
8-element Vector{ComplexF64}:
 0.70711 + 0.0im
     0.0 + 0.0im
     0.0 + 0.0im
     0.0 + 0.0im
     0.0 + 0.0im
     0.0 + 0.0im
     0.0 + 0.0im
 0.70711 + 0.0im

🔹 MEDIÇÕES OTIMIZADAS (POVMs):

   📌 PARTY 1: Alice
   ------------------------------
      📍 Input 0:
         ↳ Output 0 (Matriz):
            2×2 Matrix{ComplexF64}:
              0.0+0.0im     -0.0-1.0e-5im
             -0.0+1.0e-5im   0.0+0.0im

         ↳ Output 1 (Matriz):
            2×2 Matrix{ComplexF64}:
             1.0+0.0im     0.0+1.0e-5im

## Agora fixando que as medições de Alice, Bob e Charlie sejam as mesmas:

In [64]:
# ==============================================================================
# 2. Symmetric Optimization Functions
# ==============================================================================

"""
    optimize_symmetric_measurements(psi, current_M, coeffs_matrix)

Optimizes a SINGLE set of measurements M such that M_A = M_B = M_C = M.
It constructs an objective function that linearizes the problem by summing 
the effective operators seen by Alice, Bob, and Charlie.
"""
function optimize_symmetric_measurements(psi, current_M, coeffs_matrix)
    d = 2 
    num_inputs = 2 
    num_outputs = 2 
    
    # Convert state vector to density matrix for tracing
    rho = psi * psi'
    
    # We use the SAME measurement 'current_M' for the fixed parts of the terms
    M_fixed = current_M 
    
    # Define SDP Variables (One set of measurements M)
    E = [[Convex.ComplexVariable(d, d) for _ in 1:num_outputs] for _ in 1:num_inputs]
    
    objective_expr = 0
    
    # Identity matrix for tensor products
    Id = Matrix{ComplexF64}(I, d, d)
    
    for row in 0:7 
        # Decode inputs (xyz)
        x = ((row >> 2) & 1) + 1
        y = ((row >> 1) & 1) + 1
        z = (row & 1) + 1
        
        for col in 0:7 
            # Decode outputs (abc)
            a = ((col >> 2) & 1) + 1
            b = ((col >> 1) & 1) + 1
            c = (col & 1) + 1
            
            coeff = coeffs_matrix[row+1, col+1]
            
            if coeff != 0
                # --- Term 1: Treat Alice as Variable E, fix Bob/Charlie as M_fixed ---
                # Operator on BC
                op_BC = kron(M_fixed[y][b], M_fixed[z][c])
                # Effective state for Alice
                rho_A_eff = partial_trace_BC(rho * (Id ⊗ op_BC), 2, 2, 2)
                # Add to objective (Variable is E[x][a])
                objective_expr += coeff * real(tr(E[x][a] * rho_A_eff))
                
                # --- Term 2: Treat Bob as Variable E, fix Alice/Charlie as M_fixed ---
                # Operator on AC
                op_AC = kron(M_fixed[x][a], M_fixed[z][c])
                # Permute rho to put Bob first (B, A, C)
                rho_perm_B = permutedims(reshape(rho, (2,2,2,2,2,2)), (2,1,3, 5,4,6))
                rho_perm_B = reshape(rho_perm_B, (8,8))
                # Effective state for Bob
                rho_B_eff = partial_trace_BC(rho_perm_B * (Id ⊗ op_AC), 2, 2, 2)
                # Add to objective (Variable is E[y][b])
                objective_expr += coeff * real(tr(E[y][b] * rho_B_eff))
                
                # --- Term 3: Treat Charlie as Variable E, fix Alice/Bob as M_fixed ---
                # Operator on AB
                op_AB = kron(M_fixed[x][a], M_fixed[y][b])
                # Permute rho to put Charlie first (C, A, B)
                rho_perm_C = permutedims(reshape(rho, (2,2,2,2,2,2)), (3,1,2, 6,4,5))
                rho_perm_C = reshape(rho_perm_C, (8,8))
                # Effective state for Charlie
                rho_C_eff = partial_trace_BC(rho_perm_C * (Id ⊗ op_AB), 2, 2, 2)
                # Add to objective (Variable is E[z][c])
                objective_expr += coeff * real(tr(E[z][c] * rho_C_eff))
            end
        end
    end
    
    # Constraints: POVM constraints for the single set E
    constraints = Convex.Constraint[]
    Id_d = Matrix{ComplexF64}(I, d, d)
    
    for i in 1:num_inputs
        push!(constraints, E[i][1] + E[i][2] == Id_d)
        push!(constraints, E[i][1] ⪰ 0)
        push!(constraints, E[i][2] ⪰ 0)
    end
    
    # Solve
    problem = minimize(objective_expr, constraints)
    Convex.solve!(problem, SCS.Optimizer; silent_solver = true)
    
    # Return the optimized symmetric measurement set
    return [[evaluate(E[i][j]) for j in 1:num_outputs] for i in 1:num_inputs]
end

"""
    optimize_state(M, coeffs_matrix)
Overloaded function. Finds the best state |psi> assuming symmetric measurements M.
"""
function optimize_state(M, coeffs_matrix)
    # We pass the same M for A, B, and C
    return optimize_state_general(M, M, M, coeffs_matrix)
end

"""
    optimize_state_general(M_A, M_B, M_C, coeffs_matrix)
Original state optimization function.
"""
function optimize_state_general(M_A, M_B, M_C, coeffs_matrix)
    Bell_Operator = zeros(ComplexF64, 8, 8)
    
    for row in 0:7 
        x = ((row >> 2) & 1) + 1; y = ((row >> 1) & 1) + 1; z = (row & 1) + 1
        for col in 0:7 
            a = ((col >> 2) & 1) + 1; b = ((col >> 1) & 1) + 1; c = (col & 1) + 1
            
            coeff = coeffs_matrix[row+1, col+1]
            if coeff != 0
                term = tensor_prod(M_A[x][a], M_B[y][b], M_C[z][c])
                Bell_Operator += coeff * term
            end
        end
    end
    
    evals, evecs = eigen(Hermitian(Bell_Operator))
    min_val = evals[1]
    best_psi = evecs[:, 1]
    
    return best_psi, min_val
end

# ==============================================================================
# 3. Main Loop (Symmetric)
# ==============================================================================

"""
    seesaw_symmetric_single_run(coeffs_matrix; max_iter=50, tol=1e-6, verbose=false)
Performs See-Saw optimization enforcing M_A = M_B = M_C.
"""
function seesaw_symmetric_single_run(coeffs_matrix; max_iter=50, tol=1e-6, verbose=false)
    d = 2
    
    # Generate one random PVM
    function rand_pvm()
        U = qr(randn(ComplexF64, d, d)).Q
        E0 = U[:,1] * U[:,1]'
        E1 = I(d) - E0
        return [[E0, E1], [E0, E1]] 
    end
    
    # Initialize SINGLE measurement set M
    M = rand_pvm()
    
    # Initialize random state
    psi = normalize(randn(ComplexF64, 8))
    
    current_val = 0.0
    
    for k in 1:max_iter
        prev_val = current_val
        
        # 1. Optimize M (Symmetric)
        M = optimize_symmetric_measurements(psi, M, coeffs_matrix)
        
        # 2. Optimize State (using M for all parties)
        psi, current_val = optimize_state(M, coeffs_matrix)
        
        if verbose
            println("  -> Iter $k: $current_val")
        end
        
        if abs(current_val - prev_val) < tol && k > 1
            break
        end
    end
    
    return current_val, psi, M
end

"""
    find_best_symmetric_violation(coeffs_matrix; num_runs=50)
Global optimizer for the symmetric case.
"""
function find_best_symmetric_violation(coeffs_matrix; num_runs=50)
    println("Starting SYMMETRIC Global Optimization (Ma=Mb=Mc)...")
    println("---------------------------------------------------")
    
    global_best_val = Inf
    global_best_psi = nothing
    global_best_M = nothing
    
    for i in 1:num_runs
        val, psi, M = seesaw_symmetric_single_run(coeffs_matrix, verbose=false)
        
        if val < global_best_val
            global_best_val = val
            global_best_psi = psi
            global_best_M = M
            @printf "Run %02d: New Best Found! -> %.6f\n" i global_best_val
        end
    end
    
    println("---------------------------------------------------")
    println("Symmetric Optimization Finished.")
    println("Best Global Value: $global_best_val")
    
    return global_best_val, global_best_psi, global_best_M
end

# ==============================================================================
# 5. Funções de Formatação (Versão Simétrica)
# ==============================================================================

using Printf # Garantindo que o pacote esteja carregado para o @printf

"""
    clean_num(z, tol=1e-6)
Auxiliar: Trata partes reais e imaginárias separadamente, zera ruídos
e retorna explicitamente um número Complexo arredondado.
"""
function clean_num(z; tol=1e-6)
    # Extrai partes
    re = real(z)
    im = imag(z)

    # Zera se for ruído numérico (menor que a tolerância)
    re = abs(re) < tol ? 0.0 : re
    im = abs(im) < tol ? 0.0 : im

    # Arredonda cada componente individualmente
    re_rounded = round(re, digits=5)
    im_rounded = round(im, digits=5)

    # Reconstrói forçando o tipo Complexo
    return Complex(re_rounded, im_rounded)
end

"""
    print_symmetric_results(val, rho, M)
Imprime os resultados para o caso onde Ma = Mb = Mc.
Recebe um único conjunto M, em vez de uma tupla.
"""
function print_symmetric_results(val, rho, M)
    println("="^60)
    println("         RESULTADOS DA OTIMIZAÇÃO SIMÉTRICA")
    println("="^60)
    
    # 1. Valor da Violação
    @printf "\n🔹 VALOR MÍNIMO ENCONTRADO: %.6f\n" val
    
    # 2. Estado (Psi)
    println("\n🔹 ESTADO OTIMIZADO (PSI):")
    # Aplica a limpeza
    rho_clean = map(x -> clean_num(x), rho)
    
    # Mostra formatado pelo Julia
    show(stdout, "text/plain", rho_clean)
    println("\n")
    
    # 3. Medições (Único conjunto compartilhado)
    println("🔹 MEDIÇÕES COMPARTILHADAS (Ma = Mb = Mc):")
    println("   " * "-"^40)
    
    # M é diretamente um Vetor de Inputs (não iteramos sobre parties aqui)
    for (in_idx, input_set) in enumerate(M)
        in_label = in_idx - 1 
        println("   📍 Input $in_label:")
        
        # input_set é um Vetor de Outputs (Matrizes)
        for (out_idx, mat) in enumerate(input_set)
            out_label = out_idx - 1
            println("      ↳ Output $out_label (Matriz):")
            
            # Limpa a matriz antes de printar
            mat_clean = map(x -> clean_num(x), mat)
            
            # Truque para indentar a matriz no print
            io = IOBuffer()
            show(IOContext(io, :limit=>true), "text/plain", mat_clean)
            str_mat = String(take!(io))
            
            # Adiciona identação visual
            for line in split(str_mat, "\n")
                println("         " * line)
            end
            println("") 
        end
    end
    println("="^60)
end

print_symmetric_results

In [65]:
val_symm1, psi_symm1, measurements_symm1 = find_best_symmetric_violation(M29_disturbed1)
print_symmetric_results(val_symm1, psi_symm1, measurements_symm1);

Starting SYMMETRIC Global Optimization (Ma=Mb=Mc)...
---------------------------------------------------
Run 01: New Best Found! -> 0.999899
Run 02: New Best Found! -> 0.944736
Run 13: New Best Found! -> 0.944732
Run 23: New Best Found! -> 0.944713
Run 30: New Best Found! -> 0.944679
Run 47: New Best Found! -> 0.944669
---------------------------------------------------
Symmetric Optimization Finished.
Best Global Value: 0.9446687948764221
         RESULTADOS DA OTIMIZAÇÃO SIMÉTRICA

🔹 VALOR MÍNIMO ENCONTRADO: 0.944669

🔹 ESTADO OTIMIZADO (PSI):
8-element Vector{ComplexF64}:
  0.44016 - 0.19093im
 -0.06378 + 0.1167im
 -0.06378 + 0.1167im
 -0.14323 + 0.28374im
 -0.06378 + 0.1167im
 -0.14323 + 0.28374im
 -0.14323 + 0.28374im
  0.64318 + 0.0im

🔹 MEDIÇÕES COMPARTILHADAS (Ma = Mb = Mc):
   ----------------------------------------
   📍 Input 0:
      ↳ Output 0 (Matriz):
         2×2 Matrix{ComplexF64}:
          0.84722+0.0im      0.04384-0.35709im
          0.04384+0.35709im  0.15278+0.0i

## Vou fazer o mesmo, só que agora assumindo que é tudo real.

In [66]:
using LinearAlgebra
using Convex
using SCS
using Random
using Printf

# ==============================================================================
# 1. Helper Functions (Real Domain)
# ==============================================================================

"""
    tensor_prod(A, B, C)
Returns the Kronecker product A ⊗ B ⊗ C.
"""
function tensor_prod(A, B, C)
    return kron(kron(A, B), C)
end

"""
    partial_trace_BC(rho, da, db, dc)
Traces out subsystems B and C. 
Input and Output are now Float64 (Real Quantum Mechanics).
"""
function partial_trace_BC(rho, da, db, dc)
    rho_tensor = reshape(rho, (da, db, dc, da, db, dc))
    rho_A = zeros(Float64, da, da) # Float64
    for i in 1:da, j in 1:da
        sum_val = 0.0
        for k in 1:db, l in 1:dc
            sum_val += rho_tensor[i, k, l, j, k, l]
        end
        rho_A[i, j] = sum_val
    end
    return rho_A
end

# Helper operator
⊗(A, B) = kron(A, B)

# ==============================================================================
# 2. Symmetric Optimization Core (Real)
# ==============================================================================

"""
    optimize_symmetric_measurements_real(psi, current_M, coeffs_matrix)
Optimizes symmetric measurements in the REAL domain.
"""
function optimize_symmetric_measurements_real(psi, current_M, coeffs_matrix)
    d = 2 
    num_inputs = 2 
    num_outputs = 2 
    
    # State is real, so transpose is equivalent to adjoint
    rho = psi * psi'
    
    M_fixed = current_M 
    
    # CHANGE: Use Convex.Variable (Real) instead of ComplexVariable
    E = [[Convex.Variable(d, d) for _ in 1:num_outputs] for _ in 1:num_inputs]
    
    objective_expr = 0
    
    # Identity matrix (Float64)
    Id = Matrix{Float64}(I, d, d)
    
    for row in 0:7 
        x = ((row >> 2) & 1) + 1
        y = ((row >> 1) & 1) + 1
        z = (row & 1) + 1
        
        for col in 0:7 
            a = ((col >> 2) & 1) + 1
            b = ((col >> 1) & 1) + 1
            c = (col & 1) + 1
            
            coeff = coeffs_matrix[row+1, col+1]
            
            if coeff != 0
                # Term 1: Alice Variable
                op_BC = kron(M_fixed[y][b], M_fixed[z][c])
                rho_A_eff = partial_trace_BC(rho * (Id ⊗ op_BC), 2, 2, 2)
                # No need for real() wrapper, everything is real, but keeping for safety
                objective_expr += coeff * tr(E[x][a] * rho_A_eff)
                
                # Term 2: Bob Variable
                op_AC = kron(M_fixed[x][a], M_fixed[z][c])
                rho_perm_B = permutedims(reshape(rho, (2,2,2,2,2,2)), (2,1,3, 5,4,6))
                rho_perm_B = reshape(rho_perm_B, (8,8))
                rho_B_eff = partial_trace_BC(rho_perm_B * (Id ⊗ op_AC), 2, 2, 2)
                objective_expr += coeff * tr(E[y][b] * rho_B_eff)
                
                # Term 3: Charlie Variable
                op_AB = kron(M_fixed[x][a], M_fixed[y][b])
                rho_perm_C = permutedims(reshape(rho, (2,2,2,2,2,2)), (3,1,2, 6,4,5))
                rho_perm_C = reshape(rho_perm_C, (8,8))
                rho_C_eff = partial_trace_BC(rho_perm_C * (Id ⊗ op_AB), 2, 2, 2)
                objective_expr += coeff * tr(E[z][c] * rho_C_eff)
            end
        end
    end
    
    # Constraints
    constraints = Convex.Constraint[]
    Id_d = Matrix{Float64}(I, d, d)
    
    for i in 1:num_inputs
        push!(constraints, E[i][1] + E[i][2] == Id_d)
        push!(constraints, E[i][1] ⪰ 0) # Positive Semidefinite (implying Symmetric for real vars)
        push!(constraints, E[i][2] ⪰ 0)
    end
    
    # Solve
    problem = minimize(objective_expr, constraints)
    Convex.solve!(problem, SCS.Optimizer; silent_solver = true)
    
    return [[evaluate(E[i][j]) for j in 1:num_outputs] for i in 1:num_inputs]
end

"""
    optimize_state_real(M, coeffs_matrix)
Finds the best REAL state |psi> assuming symmetric measurements M.
"""
function optimize_state_real(M, coeffs_matrix)
    d = 2
    # Matrix is Real (Float64)
    Bell_Operator = zeros(Float64, 8, 8)
    
    M_A = M; M_B = M; M_C = M
    
    for row in 0:7 
        x = ((row >> 2) & 1) + 1; y = ((row >> 1) & 1) + 1; z = (row & 1) + 1
        for col in 0:7 
            a = ((col >> 2) & 1) + 1; b = ((col >> 1) & 1) + 1; c = (col & 1) + 1
            
            coeff = coeffs_matrix[row+1, col+1]
            if coeff != 0
                term = tensor_prod(M_A[x][a], M_B[y][b], M_C[z][c])
                Bell_Operator += coeff * term
            end
        end
    end
    
    # Use Symmetric() to ensure Julia uses a real-symmetric eigensolver
    evals, evecs = eigen(Symmetric(Bell_Operator))
    min_val = evals[1]
    best_psi = evecs[:, 1]
    
    return best_psi, min_val
end

# ==============================================================================
# 3. Main Loop (Real Symmetric)
# ==============================================================================

"""
    seesaw_symmetric_real_run(coeffs_matrix; max_iter=50, tol=1e-6, verbose=false)
"""
function seesaw_symmetric_real_run(coeffs_matrix; max_iter=50, tol=1e-6, verbose=false)
    d = 2
    
    # Generate random Real Orthogonal Matrix for initialization
    function rand_real_pvm()
        # QR decomposition of real Gaussian matrix gives orthogonal Q
        Q = qr(randn(Float64, d, d)).Q
        E0 = Q[:,1] * Q[:,1]'
        E1 = I(d) - E0
        return [[E0, E1], [E0, E1]] 
    end
    
    M = rand_real_pvm()
    
    # Initialize random Real state
    psi = normalize(randn(Float64, 8))
    
    current_val = 0.0
    
    for k in 1:max_iter
        prev_val = current_val
        
        # Optimize M (Real)
        M = optimize_symmetric_measurements_real(psi, M, coeffs_matrix)
        
        # Optimize State (Real)
        psi, current_val = optimize_state_real(M, coeffs_matrix)
        
        if verbose
            println("  -> Iter $k: $current_val")
        end
        
        if abs(current_val - prev_val) < tol && k > 1
            break
        end
    end
    
    return current_val, psi, M
end

"""
    find_best_symmetric_real_violation(coeffs_matrix; num_runs=50)
Global optimizer for the symmetric REAL case.
"""
function find_best_symmetric_real_violation(coeffs_matrix; num_runs=50)
    println("Starting SYMMETRIC REAL Optimization...")
    println("---------------------------------------------------")
    
    global_best_val = Inf
    global_best_psi = nothing
    global_best_M = nothing
    
    for i in 1:num_runs
        val, psi, M = seesaw_symmetric_real_run(coeffs_matrix, verbose=false)
        
        if val < global_best_val
            global_best_val = val
            global_best_psi = psi
            global_best_M = M
            @printf "Run %02d: New Best Found! -> %.6f\n" i global_best_val
        end
    end
    
    println("---------------------------------------------------")
    println("Optimization Finished.")
    println("Best Global Value: $global_best_val")
    
    return global_best_val, global_best_psi, global_best_M
end

# ==============================================================================
# 4. Formatting (Adapted for Real Numbers)
# ==============================================================================

"""
    clean_real_num(x)
Rounds a real number.
"""
function clean_real_num(x)
    return round(x, digits=5)
end

"""
    print_symmetric_real_results(val, psi, M)
"""
function print_symmetric_real_results(val, psi, M)
    println("="^60)
    println("      RESULTADOS DA OTIMIZAÇÃO (REAL & SIMÉTRICA)")
    println("="^60)
    
    @printf "\n🔹 VALOR MÍNIMO ENCONTRADO: %.6f\n" val
    
    println("\n🔹 ESTADO OTIMIZADO (PSI - REAL):")
    psi_clean = map(x -> clean_real_num(x), psi)
    show(stdout, "text/plain", psi_clean)
    println("\n")
    
    println("🔹 MEDIÇÕES COMPARTILHADAS (REAL):")
    println("   " * "-"^40)
    
    for (in_idx, input_set) in enumerate(M)
        in_label = in_idx - 1 
        println("   📍 Input $in_label:")
        
        for (out_idx, mat) in enumerate(input_set)
            out_label = out_idx - 1
            println("      ↳ Output $out_label (Matriz):")
            
            mat_clean = map(x -> clean_real_num(x), mat)
            
            io = IOBuffer()
            show(IOContext(io, :limit=>true), "text/plain", mat_clean)
            str_mat = String(take!(io))
            
            for line in split(str_mat, "\n")
                println("         " * line)
            end
            println("") 
        end
    end
    println("="^60)
end

# ==============================================================================
# Example Usage
# ==============================================================================

coeffs = zeros(8, 8) 
coeffs[1, 1] = -1.0 
coeffs[8, 8] = -1.0 

val_real, psi_real, M_real = find_best_symmetric_real_violation(coeffs, num_runs=10)

print_symmetric_real_results(val_real, psi_real, M_real)


Starting SYMMETRIC REAL Optimization...
---------------------------------------------------
Run 01: New Best Found! -> -2.000000
Run 03: New Best Found! -> -2.000000
Run 05: New Best Found! -> -2.000001
---------------------------------------------------
Optimization Finished.
Best Global Value: -2.0000006138016104
      RESULTADOS DA OTIMIZAÇÃO (REAL & SIMÉTRICA)

🔹 VALOR MÍNIMO ENCONTRADO: -2.000001

🔹 ESTADO OTIMIZADO (PSI - REAL):
8-element Vector{Float64}:
 -0.00012
  0.00246
  0.00246
 -0.04948
  0.00246
 -0.04948
 -0.04948
  0.99631

🔹 MEDIÇÕES COMPARTILHADAS (REAL):
   ----------------------------------------
   📍 Input 0:
      ↳ Output 0 (Matriz):
         2×2 Matrix{Float64}:
           0.50123  -0.02477
          -0.02477   0.99877

      ↳ Output 1 (Matriz):
         2×2 Matrix{Float64}:
          0.49877  0.02477
          0.02477  0.00123

   📍 Input 1:
      ↳ Output 0 (Matriz):
         2×2 Matrix{Float64}:
          0.49877  0.02477
          0.02477  0.00123

      ↳

In [67]:
val_real1, psi_real1, M_real1 = find_best_symmetric_real_violation(M29_disturbed1);

print_symmetric_real_results(val_real1, psi_real1, M_real1)

Starting SYMMETRIC REAL Optimization...
---------------------------------------------------
Run 01: New Best Found! -> 0.944809
Run 02: New Best Found! -> 0.944804
Run 07: New Best Found! -> 0.944803
Run 12: New Best Found! -> 0.944802
Run 40: New Best Found! -> 0.944801
Run 45: New Best Found! -> 0.944801
---------------------------------------------------
Optimization Finished.
Best Global Value: 0.9448009580162355
      RESULTADOS DA OTIMIZAÇÃO (REAL & SIMÉTRICA)

🔹 VALOR MÍNIMO ENCONTRADO: 0.944801

🔹 ESTADO OTIMIZADO (PSI - REAL):
8-element Vector{Float64}:
 -0.44576
 -0.24431
 -0.24431
  0.23101
 -0.24431
  0.23101
  0.23101
 -0.6798

🔹 MEDIÇÕES COMPARTILHADAS (REAL):
   ----------------------------------------
   📍 Input 0:
      ↳ Output 0 (Matriz):
         2×2 Matrix{Float64}:
          0.88313  0.32124
          0.32124  0.11685

      ↳ Output 1 (Matriz):
         2×2 Matrix{Float64}:
           0.11687  -0.32124
          -0.32124   0.88315

   📍 Input 1:
      ↳ Output 0 

## Essa última ai se pá tá boa

# Quantum Behaviour and Bell Inequality Violation

Now, we will define a **quantum behaviour** for the tripartite Bell scenario, aiming to find a quantum probability distribution that achieves a **lower value** of the Bell functional than any deterministic behaviour.

## Quantum Measurement Scenario

The quantum behaviour is defined as:

$$
p(a,b,c|x,y,z) = \langle \psi | E_{a|x} \otimes F_{b|y} \otimes G_{c|z} |\psi \rangle
$$

where:

- $E_{0|0} = F_{0|0} = G_{0|0} = |0\rangle \langle 0|$
- $E_{1|0} = F_{1|0} = G_{1|0} = |1\rangle \langle 1|$
  
and:

- $|\phi\rangle = \cos(\pi/3) |0\rangle + \sin(\pi/3) |1\rangle$
- $E_{0|1} = F_{0|1} = G_{0|1} = |\phi\rangle \langle \phi|$
- $E_{1|1} = F_{1|1} = G_{1|1} = \mathbb{I} - |\phi\rangle \langle \phi|$

The optimal **quantum state** is:

$$
|\psi\rangle = \cos\theta(\cos\varphi |000\rangle + \sin\varphi |111\rangle) + \sin\theta(\cos \delta |W_0\rangle + \sin \delta |W_1\rangle),
$$

where:

- $|W_0\rangle = \frac{1}{\sqrt{3}}(|100\rangle + |010\rangle + |001\rangle)$
- $|W_1\rangle = \frac{1}{\sqrt{3}}(|011\rangle + |101\rangle + |110\rangle)$

and parameters:

- $\theta = 1/100$
- $\varphi = 5\pi/6$
- $\delta = 5\pi/8$

We will now implement this in Julia.


In [68]:
using LinearAlgebra
using KroneckerProducts

# Basic states
ket0 = [1.0; 0.0]
ket1 = [0.0; 1.0]

# Projectors
proj0 = ket0 * ket0'
proj1 = ket1 * ket1'

# Measurement in phi basis
phi = cos(pi/3) * ket0 + sin(pi/3) * ket1
proj_phi = phi * phi'
id2 = I(2)

# Measurements for all parties
E = Dict((0,0)=>proj0, (1,0)=>proj1, (0,1)=>proj_phi, (1,1)=>(id2 - proj_phi))
F = E  # identical
G = E  # identical

# Define W states
W0 = (kron(ket1, ket0, ket0) + kron(ket0, ket1, ket0) + kron(ket0, ket0, ket1)) / sqrt(3)
W1 = (kron(ket0, ket1, ket1) + kron(ket1, ket0, ket1) + kron(ket1, ket1, ket0)) / sqrt(3)

# Parameters
θ = 1/100
φ = 5pi/6
δ = 5pi/8

# State construction
psi = cos(θ) * (cos(φ) * kron(ket0, ket0, ket0) + sin(φ) * kron(ket1, ket1, ket1)) + 
      sin(θ) * (cos(δ) * W0 + sin(δ) * W1)

# Initialize quantum behaviour vector
behaviour = zeros(2^3 * 2^3)  # 8 outcomes * 8 input combinations

# Mapping: (a,b,c,x,y,z) -> index
index_map = Dict()
counter = 1
for x in 0:1, y in 0:1, z in 0:1, a in 0:1, b in 0:1, c in 0:1
    index_map[(a,b,c,x,y,z)] = counter
    counter += 1
end

# Compute quantum behaviour
for x in 0:1, y in 0:1, z in 0:1, a in 0:1, b in 0:1, c in 0:1
    M = kron(E[(a,x)], F[(b,y)], G[(c,z)])
    p = real(psi' * M * psi)
    idx = index_map[(a,b,c,x,y,z)]
    behaviour[idx] = p
end

# Flatten Bell inequality vector
Ineq_clock = [0 1 1 2 1 2 2 0;
         0 0 2 2 1 1 0 0;
         0 1 0 1 2 0 2 0;
         1 0 1 0 0 2 0 2;
         0 2 1 0 0 2 1 0;
         1 1 0 0 0 0 2 2;
         1 0 0 2 1 0 0 2;
         2 0 0 1 0 1 1 2]
Ineq_clock = vec(Ineq_clock')

# Compute inner product
ineq_value = dot(Ineq_clock, behaviour)

println("Value of the Bell functional for the quantum behaviour: ", ineq_value)

Value of the Bell functional for the quantum behaviour: 1.939017805193937


## Upper Bounds on the Quantum Value using NPA Hierarchy

We will now compute **upper bounds** on the **quantum value** (without communication) of the Bell inequality under investigation.

To achieve this, we will apply the **Navascués-Pironio-Acín (NPA) hierarchy**, a method that provides **semidefinite programming (SDP) relaxations** to characterize quantum correlations.

The NPA hierarchy systematically approximates the set of quantum correlations from the outside, allowing us to obtain **certified upper bounds** on the maximal quantum violation of a given Bell inequality.

For this task, we will employ the **QuantumNPA.jl** package in Julia.

We will perform the analysis for both the **full probability representation** and the **correlator representation** of the inequality. Let's proceed with the implementation.


### Full probability representation

Remembering that, in full probability notation, the inequality is given by:
\begin{equation}
        \begin{aligned}
        %
       S_{clock} &=p(001|000) + p(010|000) +2 p(011|000) + p(100|000) + 2 p(101|000) +2 p(110|000) \\
        &+2 p(010|001) +2 p(011|001) + p(100|001)  + p(101|001) \\
        &+ p(001|010) + p(011|010) + 2 p(100|010) +2 p(110|010)  \\
        &+ p(000|011) + p(010|011) +2 p(101|011) +2 p(111|011)\\
        &+ 2p(001|100) + p(010|100) +2 p(101|100) + p(110|100)   \\
        &+ p(000|101) + p(001|101) +2 p(110|101)  +2 p(111|101) \\
        &+ p(000|110) +  2 p(011|110) + p(100|110) +2 p(111|110) \\
        &+ 2p(000|111) + p(011|111) + p(101|111) + p(110|111)  +2 p(111|111) \geq 2.
    \end{aligned}
\end{equation}

In [69]:
using QuantumNPA
PA = projector(1, 1:2, 1:2, full=true);
PB = projector(2, 1:2, 1:2, full=true);
PC = projector(3, 1:2, 1:2, full=true);

p(a,b,c,x,y,z) = PA[a+1,x+1]*PB[b+1,y+1]*PC[c+1,z+1]

S_000 = p(0,0,1,0,0,0)+p(0,1,0,0,0,0)+2*p(0,1,1,0,0,0)+p(1,0,0,0,0,0)+2*p(1,0,1,0,0,0)+2*p(1,1,0,0,0,0)

S_001 = 2*p(0,1,0,0,0,1)+2*p(0,1,1,0,0,1)+p(1,0,0,0,0,1)+p(1,0,1,0,0,1)

S_010 = p(0,0,1,0,1,0)+p(0,1,1,0,1,0)+2*p(1,0,0,0,1,0)+2*p(1,1,0,0,1,0)

S_011 = p(0,0,0,0,1,1)+p(0,1,0,0,1,1)+2*p(1,0,1,0,1,1)+2*p(1,1,1,0,1,1)

S_100 = 2*p(0,0,1,1,0,0)+p(0,1,0,1,0,0)+2*p(1,0,1,1,0,0)+p(1,1,0,1,0,0)

S_101 = p(0,0,0,1,0,1)+p(0,0,1,1,0,1)+2*p(1,1,0,1,0,1)+2*p(1,1,1,1,0,1)

S_110 = p(0,0,0,1,1,0)+2*p(0,1,1,1,1,0)+p(1,0,0,1,1,0)+2*p(1,1,1,1,1,0)

S_111 = 2*p(0,0,0,1,1,1)+p(0,1,1,1,1,1)+p(1,0,1,1,1,1)+p(1,1,0,1,1,1)+2*p(1,1,1,1,1,1)

S = S_000 + S_001 + S_010 + S_011 + S_100 + S_101 + S_110 + S_111 

R = -S

-8 Id - 3 PA1|1 + 3 PA1|2 - 3 PB1|1 + 3 PB1|2 - 3 PC1|1 + 3 PC1|2 + 6 PA1|1 PB1|1 + 6 PA1|1 PC1|1 - 3 PA1|1 PC1|2 - 3 PA1|2 PB1|1 + 6 PB1|1 PC1|1 - 3 PB1|2 PC1|1 - 3 PA1|1 PB1|1 PC1|1 - 3 PA1|2 PB1|2 PC1|2

In [70]:
npa_min(S,4)

1.879328710041972

In [71]:
2/1.879328710041972

1.064209783692036

In [72]:
1/0.9406541466968572

1.0630899821273716

### Correlators representation

Remembering that, in correlator notation and assuming no signaling, the inequality is given by:
\begin{equation}
\begin{aligned}
    % 
    &9\left<A_{0}\right> + 4\left<A_{1}\right> + 9\left<B_{0}\right> 
    + 4\left<B_{1}\right> + 9\left<C_{0}\right> + 4\left<C_{1}\right>
    + 72\left<A_{0}B_{0}\right> - 9\left<A_{1}B_{0}\right>
    - 9\left<A_{0}B_{1}\right> - 9\left<A_{1}B_{1}\right> \\
    &+ 72\left<A_{0}C_{0}\right> - 9\left<A_{1}C_{0}\right>
    - 9\left<A_{0}C_{1}\right> - 9\left<A_{1}C_{1}\right>
    + 72\left<B_{0}C_{0}\right> - 9\left<B_{1}C_{0}\right> 
    - 9\left<B_{0}C_{1}\right> - 9\left<B_{1}C_{1}\right> \\
    &- 9\left<A_{0}B_{0}C_{0}\right> + 5\left<A_{1}B_{0}C_{0}\right> + 
    5\left<A_{0}B_{1}C_{0}\right>
    + 5\left<A_{0}B_{0}C_{1}\right> - 9\left<A_{1}B_{1}C_{1}\right> \leq 243.
    %
    \label{eq:inequality_probs_3}
    %
\end{aligned}
\end{equation}

# A S abaixo não bate com a de cima, mas não lembro pq eu fiz isso. De qualquer forma, se pá nao vou usar a desigualdade do artigo, então de boa

In [73]:
using QuantumNPA

# Define dichotomic observables for each party with 2 measurement settings
@dichotomic A[1:2] B[1:2] C[1:2];

# Define the Bell functional S as a symbolic expression of the observables
S = A[1] + A[2] + B[1] + B[2] + C[1] + C[2] +     # linear terms
    3*A[1]*B[1] - 2*A[2]*B[1] - A[2]*B[2] +      # A-B correlators
    3*A[1]*C[1] - 2*A[1]*C[2] - A[2]*C[2] +      # A-C correlators
    3*B[1]*C[1] - 2*B[2]*C[1] - B[2]*C[2] -      # B-C correlators
    A[1]*B[1]*C[1] - A[2]*B[2]*C[2]              # three-party correlators

#npa_max(S, "4")  # Compute the maximal quantum value of S using NPA hierarchy at level 4

A1 + A2 + B1 + B2 + C1 + C2 + 3 A1 B1 + 3 A1 C1 - 2 A1 C2 - 2 A2 B1 - A2 B2 - A2 C2 + 3 B1 C1 - 2 B2 C1 - B2 C2 - A1 B1 C1 - A2 B2 C2

### NPA Bounds at Lower Levels

Just for verification, we will now compute the **NPA bounds** of the inequality at lower levels of the hierarchy.


In [74]:
npa_max(S, "1 + A B + A C + B C"), npa_max(S, "2"), npa_max(S, "3")

(12.882398829806572, 12.882554012744988, 12.321485339479793)

## Converting a Probability Inequality into Correlator Form

In the tripartite scenario with binary inputs and outputs, a probability distribution $p(a,b,c|x,y,z)$ can be equivalently expressed in terms of correlators.

The correlators considered here are organized as:

- **Single-party correlators**:
  - $\langle A_{xz} \rangle$, where $x, z \in \{0,1\}$ → 4 components
  - $\langle B_{xy} \rangle$, where $x, y \in \{0,1\}$ → 4 components
  - $\langle C_{yz} \rangle$, where $y, z \in \{0,1\}$ → 4 components

- **Two-party correlators**:
  - $\langle AB_{xyz} \rangle$, $\langle AC_{xyz} \rangle$, $\langle BC_{xyz} \rangle$ for all combinations of inputs → $3 \times 8 = 24$ components

- **Three-party correlators**:
  - $\langle ABC_{xyz} \rangle$ for all combinations of inputs → 8 components

### Conversion formula
Given the probability vector $p(a,b,c|x,y,z)$, we can construct the correlators from the formulas:
  - $\langle A_{xz} \rangle = \sum_{a,b,c} a \cdot p(a,b,c|x,y,z) $
  - $\langle B_{xy} \rangle = \sum_{a,b,c} b \cdot p(a,b,c|x,y,z)$
  - $\langle C_{yz} \rangle = \sum_{a,b,c} c \cdot p(a,b,c|x,y,z) $
  - $\langle AB_{xyz} \rangle = \sum_{a,b,c} ab \cdot p(a,b,c|x,y,z) $
  - $\langle AC_{xyz} \rangle = \sum_{a,b,c} ac \cdot p(a,b,c|x,y,z) $
  - $\langle BC_{xyz} \rangle = \sum_{a,b,c} bc \cdot p(a,b,c|x,y,z) $
  - $\langle ABC_{xyz} \rangle = \sum_{a,b,c} abc \cdot p(a,b,c|x,y,z) $  
  
Conversely, given the correlators, we can recover the probability vector by:

$$
p(a,b,c|x,y,z) = \frac{1}{8} \left[ 1 + a \langle A_{xz} \rangle + b \langle B_{xy} \rangle + c \langle C_{yz} \rangle + ab \langle AB_{xyz} \rangle + ac \langle AC_{xyz} \rangle + bc \langle BC_{xyz} \rangle + abc \langle ABC_{xyz} \rangle \right]
$$

Here, $a,b,c \in \{-1,1\}$ and $x,y,z \in \{0,1\}$.

### Purpose of this function

The function `convert_prob_ineq_to_correlators` implements this conversion by taking as input a Bell inequality represented in the **probability vector form** (with 64 components, corresponding to all possible combinations of $(a,b,c,x,y,z)$), and producing the corresponding **correlator representation** as a vector of 44 components.

This correlator form is particularly convenient for analyzing symmetries, understanding physical constraints, and comparing classical and quantum bounds.

The function also organizes the correlators into clearly identifiable blocks, as follows:

| Indices   | Correlator                |
|-----------:|--------------------------|
| 1...4      | $\langle A_{xz} \rangle$ |
| 5...8      | $\langle B_{xy} \rangle$ |
| 9...12     | $\langle C_{yz} \rangle$ |
| 13...20    | $\langle AB_{xyz} \rangle$ |
| 21...28    | $\langle AC_{xyz} \rangle$ |
| 29...36    | $\langle BC_{xyz} \rangle$ |
| 37...44    | $\langle ABC_{xyz} \rangle$ |



In the next cell, we define this conversion function.


In [75]:
"""
    convert_prob_ineq_to_correlators_clockcycle(ineq_prob)

Given an inequality vector expressed in terms of probabilities p(a,b,c|x,y,z),
returns its equivalent expression in terms of correlators.

The correlator coordinates are ordered as:
- <A_{xz}> for x, z ∈ {0,1}
- <B_{xy}> for x, y ∈ {0,1}
- <C_{yz}> for y, z ∈ {0,1}
- <AB_{xyz}>, <AC_{xyz}>, <BC_{xyz}>, <ABC_{xyz}> for x, y, z ∈ {0,1}

The conversion rule is:
p(a,b,c|x,y,z) = 1/8 * [1 + a*<A_{xz}> + b*<B_{xy}> + c*<C_{yz}> 
                         + a*b*<AB_{xyz}> + a*c*<AC_{xyz}> + b*c*<BC_{xyz}> + a*b*c*<ABC_{xyz}>]
# Arguments
- `prob_ineq::Vector{Float64}`: vector of coefficients of the inequality in probability representation, ordered as p(abc|xyz) with standard lexicographic ordering.

# Returns
- `corr_ineq::Vector{Float64}`: vector of coefficients in the correlator representation ordered as above.
- `constant_term::Float64`: number corresponding to the const add to the ineq by the contributions of 1/8*coff
"""
function convert_prob_ineq_to_correlators_clockcycle(ineq_prob)
    correlator_ineq = zeros(44)  # Total of 44 correlators: 4 A_{xz}, 4 B_{xy}, 4 C_{yz}, 8 AB, 8 AC, 8 BC, 8 ABC
    constant_term = 0
    # Loop over all combinations of a, b, c, x, y, z ∈ {0,1}
    for x in 0:1, y in 0:1, z in 0:1, a in (-1,1), b in (-1,1), c in (-1,1)
        # Index for probability vector (order: (a,b,c,x,y,z))
        a_idx = (a == 1) ? 0 : 1 # If a == 1 then a_idx = 0, otherwise a_idx = 1
        b_idx = (b == 1) ? 0 : 1 # If b == 1 then b_idx = 0, otherwise b_idx = 1
        c_idx = (c == 1) ? 0 : 1 # If c == 1 then c_idx = 0, otherwise c_idx = 1
        prob_index = a_idx + 2*b_idx + 4*c_idx + 8*x + 16*y + 32*z + 1 # varying over the 64 coordinates of the prob vector p(abc|xyz)
        
        coeff = ineq_prob[prob_index] * 1/8
        
        # Indices for correlators
        idx_A = x*2 + z + 1                    # 1..4
        idx_B = 4 + x*2 + y + 1                # 5..8
        idx_C = 8 + y*2 + z + 1                # 9..12
        idx_AB = 12 + x*4 + y*2 + z + 1        # 13..20
        idx_AC = 20 + x*4 + y*2 + z + 1        # 21..28
        idx_BC = 28 + x*4 + y*2 + z + 1        # 29..36
        idx_ABC = 36 + x*4 + y*2 + z + 1       # 37..44
        
        # Add contributions
        constant_term += coeff  # Constant term
        
        correlator_ineq[idx_A] += coeff * a
        correlator_ineq[idx_B] += coeff * b
        correlator_ineq[idx_C] += coeff * c
        correlator_ineq[idx_AB] += coeff * a * b
        correlator_ineq[idx_AC] += coeff * a * c
        correlator_ineq[idx_BC] += coeff * b * c
        correlator_ineq[idx_ABC] += coeff * a * b * c
    end
    return correlator_ineq, constant_term
end



convert_prob_ineq_to_correlators_clockcycle

In [76]:
for x in 0:1, y in 0:1, z in 0:1
    println(x,y,z)
end


000
001
010
011
100
101
110
111


### Applying the Conversion to Ineq_clock

Now that we have defined the function `convert_prob_ineq_to_correlators_clockcycle`, we can apply it to the inequality vector `Ineq_clock`.

This decomposition allows us to express the inequality directly in terms of correlators.

In [77]:
Ineq_clock_correlators = convert_prob_ineq_to_correlators_clockcycle(Ineq_clock)[1];
constant_term = convert_prob_ineq_to_correlators_clockcycle(Ineq_clock)[2];

### Visualizing the Correlator Coefficients

To better understand the structure of the inequality in the correlator representation, we will now display the coefficients grouped by their respective correlators.

We multiply each coefficient by 8 to compensate for the $1/8$ factor present in the probability-to-correlator conversion formula. This step makes the visualization of the exact integer coefficients more transparent and aligned with the standard form of Bell-like inequalities.

The following code performs this visualization:

In [78]:
# Visualize as partes:
println("A correlators: ", 8*Ineq_clock_correlators[1:4])
println("B correlators: ", 8*Ineq_clock_correlators[5:8])
println("C correlators: ", 8*Ineq_clock_correlators[9:12])
println("AB correlators: ", 8*Ineq_clock_correlators[13:20])
println("AC correlators: ", 8*Ineq_clock_correlators[21:28])
println("BC correlators: ", 8*Ineq_clock_correlators[29:36])
println("ABC correlators: ", 8*Ineq_clock_correlators[37:44])

println("Constant term: ", 8*constant_term)


A correlators: [1.0, -4.0, -2.0, -1.0]
B correlators: [1.0, -2.0, -4.0, -1.0]
C correlators: [1.0, -2.0, -4.0, -1.0]
AB correlators: [-3.0, -6.0, 0.0, 6.0, 0.0, 0.0, 0.0, 3.0]
AC correlators: [-3.0, 0.0, -6.0, 0.0, 0.0, 0.0, 6.0, 3.0]
BC correlators: [-3.0, 0.0, 0.0, 0.0, -6.0, 6.0, 0.0, 3.0]
ABC correlators: [3.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.0]
Constant term: 52.0


In [79]:
M_29_correlators = convert_prob_ineq_to_correlators_clockcycle(M_29)[1];
constant_term_M29 = convert_prob_ineq_to_correlators_clockcycle(M_29)[2];

In [80]:
println("A correlators: ", M_29_correlators[1:4])
println("B correlators: ", M_29_correlators[5:8])
println("C correlators: ", M_29_correlators[9:12])
println("AB correlators: ", M_29_correlators[13:20])
println("AC correlators: ", M_29_correlators[21:28])
println("BC correlators: ", M_29_correlators[29:36])
println("ABC correlators: ", M_29_correlators[37:44])

println("Constant term: ", constant_term_M29)

A correlators: [-0.25, 0.3125, 0.0625, -0.25]
B correlators: [-0.25, 0.3125, 0.3125, -0.25]
C correlators: [-0.125, 0.0625, 0.3125, -0.25]
AB correlators: [0.0, -0.125, 0.375, 0.0625, 0.25, 0.0625, -0.1875, 0.0625]
AC correlators: [-0.125, 0.25, -0.125, 0.0625, 0.125, -0.1875, 0.0625, 0.0625]
BC correlators: [0.125, 0.125, 0.0, -0.1875, -0.125, 0.0625, 0.0625, 0.0625]
ABC correlators: [0.125, 0.25, 0.0, -0.0625, 0.25, -0.0625, -0.0625, 0.0625]
Constant term: 3.25


In [81]:
println("A correlators: ", 16*M_29_correlators[1:4])
println("B correlators: ", 16*M_29_correlators[5:8])
println("C correlators: ", 16*M_29_correlators[9:12])
println("AB correlators: ", 16*M_29_correlators[13:20])
println("AC correlators: ", 16*M_29_correlators[21:28])
println("BC correlators: ", 16*M_29_correlators[29:36])
println("ABC correlators: ", 16*M_29_correlators[37:44])

println("Constant term: ", 16*constant_term_M29)

A correlators: [-4.0, 5.0, 1.0, -4.0]
B correlators: [-4.0, 5.0, 5.0, -4.0]
C correlators: [-2.0, 1.0, 5.0, -4.0]
AB correlators: [0.0, -2.0, 6.0, 1.0, 4.0, 1.0, -3.0, 1.0]
AC correlators: [-2.0, 4.0, -2.0, 1.0, 2.0, -3.0, 1.0, 1.0]
BC correlators: [2.0, 2.0, 0.0, -3.0, -2.0, 1.0, 1.0, 1.0]
ABC correlators: [2.0, 4.0, 0.0, -1.0, 4.0, -1.0, -1.0, 1.0]
Constant term: 52.0


In [82]:
M29_dist_correlators = convert_prob_ineq_to_correlators_clockcycle(M29_disturbed1)[1];
constant_term_M29_dist = convert_prob_ineq_to_correlators_clockcycle(M29_disturbed1)[2];

In [83]:
println("A correlators: ", M29_dist_correlators[1:4])
println("B correlators: ", M29_dist_correlators[5:8])
println("C correlators: ", M29_dist_correlators[9:12])
println("AB correlators: ", M29_dist_correlators[13:20])
println("AC correlators: ", M29_dist_correlators[21:28])
println("BC correlators: ", M29_dist_correlators[29:36])
println("ABC correlators: ", M29_dist_correlators[37:44])

println("Constant term: ", constant_term_M29_dist)

A correlators: [-0.25, 0.3125, 0.3125, -0.25]
B correlators: [-0.25, 0.3125, 0.3125, -0.25]
C correlators: [-0.25, 0.3125, 0.3125, -0.25]
AB correlators: [0.0, -0.125, 0.375, 0.0625, 0.0, 0.0625, -0.1875, 0.0625]
AC correlators: [0.0, 0.0, -0.125, 0.0625, 0.375, -0.1875, 0.0625, 0.0625]
BC correlators: [0.0, 0.375, 0.0, -0.1875, -0.125, 0.0625, 0.0625, 0.0625]
ABC correlators: [0.25, 0.0, 0.0, -0.0625, 0.0, -0.0625, -0.0625, 0.0625]
Constant term: 3.25


In [84]:
println("A correlators: ", 16*M29_dist_correlators[1:4])
println("B correlators: ", 16*M29_dist_correlators[5:8])
println("C correlators: ", 16*M29_dist_correlators[9:12])
println("AB correlators: ", 16*M29_dist_correlators[13:20])
println("AC correlators: ", 16*M29_dist_correlators[21:28])
println("BC correlators: ", 16*M29_dist_correlators[29:36])
println("ABC correlators: ", 16*M29_dist_correlators[37:44])

println("Constant term: ", 16*constant_term_M29_dist)

A correlators: [-4.0, 5.0, 5.0, -4.0]
B correlators: [-4.0, 5.0, 5.0, -4.0]
C correlators: [-4.0, 5.0, 5.0, -4.0]
AB correlators: [0.0, -2.0, 6.0, 1.0, 0.0, 1.0, -3.0, 1.0]
AC correlators: [0.0, 0.0, -2.0, 1.0, 6.0, -3.0, 1.0, 1.0]
BC correlators: [0.0, 6.0, 0.0, -3.0, -2.0, 1.0, 1.0, 1.0]
ABC correlators: [4.0, 0.0, 0.0, -1.0, 0.0, -1.0, -1.0, 1.0]
Constant term: 52.0


### Escrever uma função que projeta no NS, mas to com preguiça

In [85]:
using QuantumNPA  # Load QuantumNPA package for NPA hierarchy computations

@dichotomic A[1:2] B[1:2] C[1:2];  # Define dichotomic observables A, B, C with 2 inputs each

#8. E000 +E001 +E010 +E011 +E100 +E101 +E110 -E111 <= 6
S = A[1]*B[1]*C[1] +A[1]*B[1]*C[2] +A[1]*B[2]*C[1] +A[1]*B[2]*C[2] +
A[2]*B[1]*C[1] +A[2]*B[1]*C[2] +A[2]*B[2]*C[1] -A[2]*B[2]*C[2] 

NPA_lvl4_cor = npa_max(S, "4")  # Compute the maximal quantum value of S using NPA hierarchy at level 4

5.997755573582024

#### OBS: Os correlatores de dois e tres corpos estão parecidos com a eq 8 do artigo. Agora os de um corpo não estão cooncordando. Não sei se sou eu que estou errado ou se é a derivação do paper. 

# Essa desigualdade S_cycle abaixo não tá batendo também, eu encontrei 18 como minimo nos clássico (agora deu 20, estranho, mas de qualquer forma não é faceta). Se pá vou fazer só com a faceta anterior mesmo.

# Testing a Bell Inequality in the Tripartite Cycle Scenario

In this section, we will test a specific Bell inequality over the set of deterministic behaviours associated with the *tripartite cycle scenario*.  

As previously described, the set of deterministic behaviours for this scenario consists of the **union** of the deterministic behaviours from both the **Clockwise cycle** and the **Counter-clockwise cycle** communication structures:  

The goal here is to evaluate the value of a linear functional — representing the Bell inequality — over all these deterministic behaviours. This will provide us with the *classical bound* for the inequality under the cycle scenario.  

This inequality is given by:

\begin{equation}
\begin{aligned}
    %
    S_{cycle} = &12 p(001|000) + 12 p(010|000)+ 15 p(011|000) + 12 p(100|000)  + 15 p(101|000)
    + 15 p(110|000)\\
    & + 4 p(001|001) + 15 p(010|001) + 12 p(011|001) + 15 p(100|001) + 12 p(101|001) 
    + 5 p(110|001) \\
    &+ 15 p(001|010) + 4 p(010|010) + 12 p(011|010) + 15 p(100|010)  + 5 p(101|010) 
    + 12 p(110|010) \\
    &+ 8 p(000|011) + 4 p(001|011) + 4 p(010|011) + 5 p(101|011) + 5 p(110|011)  
    + 10 p(111|011) \\
    &+ 15 p(001|100) + 15 p(010|100) + 5 p(011|100) + 4 p(100|100)  + 12 p(101|100) 
    + 12 p(110|100)\\
    &+ 8 p(000|101) + 4 p(001|101) + 5 p(011|101) + 4 p(100|101) + 5 p(110|101) 
    + 10 p(111|101)\\
    &+ 8 p(000|110) + 4 p(010|110) + 5 p(011|110)  + 4 p(100|110)  + 5 p(101|110) 
    + 10 p(111|110) \\    
    &+ 5 p(000|111) + 4 p(011|111) + 4 p(101|111) + 4 p(110|111)  
    + 8 p(111|111) \geq 20
    %
    \label{eq:inequality_probs_2}
    %        
\end{aligned}
\end{equation}

In the next cell, we will define the linear functional and after that implement the code to perform these evaluations.


### Coeficientes of the S$_{cycle}$ inequality:

In [86]:
Ineq_cycle = [0 12 12 15 12 15 15 0;
         0 4 15 12 15 12 5 0;
         0 15 4 12 15 5 12 0;
         8 4 4 0 0 5 5 10;
         0 15 15 5 4 12 12 0;
         8 4 0 5 4 0 5 10;
         8 0 4 5 4 5 0 10;
         5 0 0 4 0 4 4 8]
Ineq_cycle = vec(Ineq_cycle');

In [87]:
"""
    convert_prob_ineq_to_correlators_full(ineq_prob)

Given an inequality vector expressed in terms of probabilities p(a,b,c|x,y,z),
returns its equivalent expression in terms of correlators.

The correlator coordinates are ordered as:
- <A_{xz}>, <B_{xy}>, <C_{yz}>, <AB_{xyz}>, <AC_{xyz}>, <BC_{xyz}>, <ABC_{xyz}> for x, y, z ∈ {0,1}

The conversion rule is:
p(a,b,c|x,y,z) = 1/8 * [1 + a*<A_{xyz}> + b*<B_{xyz}> + c*<C_{xyz}> 
                         + a*b*<AB_{xyz}> + a*c*<AC_{xyz}> + b*c*<BC_{xyz}> + a*b*c*<ABC_{xyz}>]
# Arguments
- `prob_ineq::Vector{Float64}`: vector of coefficients of the inequality in probability representation, ordered as p(abc|xyz) with standard lexicographic ordering.

# Returns
- `corr_ineq::Vector{Float64}`: vector of coefficients in the correlator representation ordered as above.
- `constant_term::Float64`: number corresponding to the const add to the ineq by the contributions of 1/8*coff
"""
function convert_prob_ineq_to_correlators_full(ineq_prob)
    correlator_ineq = zeros(56)  # Total of 44 correlators: 8 A_{xz}, 8 B_{xy}, 8 C_{yz}, 8 AB, 8 AC, 8 BC, 8 ABC
    constant_term = 0
    # Loop over all combinations of a, b, c, x, y, z ∈ {0,1}
    for x in 0:1, y in 0:1, z in 0:1, a in (-1,1), b in (-1,1), c in (-1,1)
        # Index for probability vector (order: (a,b,c,x,y,z))
        a_idx = (a == 1) ? 0 : 1 # If a == 1 then a_idx = 0, otherwise a_idx = 1
        b_idx = (b == 1) ? 0 : 1 # If b == 1 then b_idx = 0, otherwise b_idx = 1
        c_idx = (c == 1) ? 0 : 1 # If c == 1 then c_idx = 0, otherwise c_idx = 1
        prob_index = a_idx + 2*b_idx + 4*c_idx + 8*x + 16*y + 32*z + 1 # varying over the 64 coordinates of the prob vector p(abc|xyz)
        
        coeff = ineq_prob[prob_index] * 1/8
        
        # Indices for correlators
        idx_A = 1 + x + 2*y + 4*z               # 1..8
        idx_B = 8 + idx_A                      # 9..16
        idx_C = 16 + idx_A                     # 17..24
        idx_AB = 24 + idx_A                    # 25..32
        idx_AC = 32 + idx_A                    # 33..40
        idx_BC = 40 + idx_A                    # 41..48
        idx_ABC = 48 + idx_A                   # 49..56
        
        # Add contributions
        constant_term += coeff  # Constant term
        
        correlator_ineq[idx_A] += coeff * a
        correlator_ineq[idx_B] += coeff * b
        correlator_ineq[idx_C] += coeff * c
        correlator_ineq[idx_AB] += coeff * a * b
        correlator_ineq[idx_AC] += coeff * a * c
        correlator_ineq[idx_BC] += coeff * b * c
        correlator_ineq[idx_ABC] += coeff * a * b * c
    end
    return correlator_ineq, constant_term
end



convert_prob_ineq_to_correlators_full

### Applying the Conversion to Ineq_cycle

Now that we have defined the function `convert_prob_ineq_to_correlators_full`, we can apply it to the inequality vector `Ineq_cycle`.

This decomposition allows us to express the inequality directly in terms of correlators.

In [88]:
Ineq_cycle_correlators = convert_prob_ineq_to_correlators_full(Ineq_cycle)[1];
constant_term = convert_prob_ineq_to_correlators_full(Ineq_cycle)[2];

### Visualizing the Correlator Coefficients

To better understand the structure of the inequality in the correlator representation, we will now display the coefficients grouped by their respective correlators.

We multiply each coefficient by 8 to compensate for the $1/8$ factor present in the probability-to-correlator conversion formula.

In [89]:
# Visualize as partes:
println("A correlators: ", 8*Ineq_cycle_correlators[1:8])
println("B correlators: ", 8*Ineq_cycle_correlators[9:16])
println("C correlators: ", 8*Ineq_cycle_correlators[17:24])
println("AB correlators: ", 8*Ineq_cycle_correlators[25:32])
println("AC correlators: ", 8*Ineq_cycle_correlators[33:40])
println("BC correlators: ", 8*Ineq_cycle_correlators[41:48])
println("ABC correlators: ", 8*Ineq_cycle_correlators[48:56])

println("Constant term: ", 8*constant_term)

A correlators: [-3.0, 7.0, -1.0, -2.0, -1.0, -2.0, -4.0, -7.0]
B correlators: [-3.0, -1.0, 7.0, -2.0, -1.0, -4.0, -2.0, -7.0]
C correlators: [-3.0, -1.0, -1.0, -4.0, 7.0, -2.0, -2.0, -7.0]
AB correlators: [-27.0, -9.0, -9.0, 0.0, -45.0, 18.0, 18.0, 9.0]
AC correlators: [-27.0, -9.0, -45.0, 18.0, -9.0, 0.0, 18.0, 9.0]
BC correlators: [-27.0, -45.0, -9.0, 18.0, -9.0, 18.0, 0.0, 9.0]
ABC correlators: [9.0, 9.0, -5.0, -5.0, 0.0, -5.0, 0.0, 0.0, 9.0]
Constant term: 403.0


## Projecting the correlator inequality into the non-signaling polytope

Now, we define a function that **projects a correlator inequality** expressed in terms of the full set of correlators onto the **non-signaling polytope**.

The projection works by **summing over the redundant dependencies** according to the non-signaling constraints:

- $⟨A_{x}⟩ = ∑_{y,z} ⟨A_{xyz}⟩$
- $⟨B_{y}⟩ = ∑_{x,z} ⟨B_{xyz}⟩$
- $⟨C_{z}⟩ = ∑_{x,y} ⟨C_{xyz}⟩$
- $⟨A_{x}B_{y}⟩ = ∑_{z} ⟨AB_{xyz}⟩$
- $⟨A_{x}C_{z}⟩ = ∑_{y} ⟨AC_{xyz}⟩$
- $⟨B_{y}C_{z}⟩ = ∑_{x} ⟨BC_{xyz}⟩$
- $⟨A_{x}B_{y}C_{z}⟩ = ⟨ABC_{xyz}⟩$ (no sum)

The result is a vector of **26 coefficients** that describe the inequality in terms of **non-signaling correlators**.


In [90]:
function project_correlators_to_nosignal(correlator_ineq::Vector{Float64})
    # correlator_ineq tem 56 componentes: 7 tipos x 8 combinações de (x,y,z)

    projected = zeros(26)  # tamanho final: 26

    # A_x: soma sobre y,z
    for x in 0:1
        for y in 0:1, z in 0:1
            idx = 1 + x + 2*y + 4*z
            projected[1 + x] += correlator_ineq[idx]
        end
    end

    # B_y: soma sobre x,z
    for y in 0:1
        for x in 0:1, z in 0:1
            idx = 8 + 1 + x + 2*y + 4*z
            projected[3 + y] += correlator_ineq[idx]
        end
    end

    # C_z: soma sobre x,y
    for z in 0:1
        for x in 0:1, y in 0:1
            idx = 16 + 1 + x + 2*y + 4*z
            projected[5 + z] += correlator_ineq[idx]
        end
    end

    # A_x B_y: soma sobre z
    for x in 0:1, y in 0:1
        for z in 0:1
            idx = 24 + 1 + x + 2*y + 4*z
            projected[7 + 2*x + y] += correlator_ineq[idx]
        end
    end

    # A_x C_z: soma sobre y
    for x in 0:1, z in 0:1
        for y in 0:1
            idx = 32 + 1 + x + 2*y + 4*z
            projected[11 + 2*x + z] += correlator_ineq[idx]
        end
    end

    # B_y C_z: soma sobre x
    for y in 0:1, z in 0:1
        for x in 0:1
            idx = 40 + 1 + x + 2*y + 4*z
            projected[15 + 2*y + z] += correlator_ineq[idx]
        end
    end

    # A_x B_y C_z: sem soma, cada combinação vai direto
    for x in 0:1, y in 0:1, z in 0:1
        idx = 48 + 1 + x + 2*y + 4*z
        projected[19 + x*4 + y*2 + z] = correlator_ineq[idx]
    end

    return projected
end


project_correlators_to_nosignal (generic function with 1 method)

In [91]:
projectNS_Ineq_cycle_correlators = project_correlators_to_nosignal(Ineq_cycle_correlators);

### Visualizing the Correlator Coefficients

To better understand the structure of the inequality in the correlator representation, we will now display the coefficients grouped by their respective correlators.

We multiply each coefficient by 8 to compensate for the $1/8$ factor present in the probability-to-correlator conversion formula.

In [92]:
# Visualize as partes:
println("A correlators: ", 8*projectNS_Ineq_cycle_correlators[1:2])
println("B correlators: ", 8*projectNS_Ineq_cycle_correlators[3:4])
println("C correlators: ", 8*projectNS_Ineq_cycle_correlators[5:6])
println("AB correlators: ", 8*projectNS_Ineq_cycle_correlators[7:10])
println("AC correlators: ", 8*projectNS_Ineq_cycle_correlators[11:14])
println("BC correlators: ", 8*projectNS_Ineq_cycle_correlators[15:18])
println("ABC correlators: ", 8*projectNS_Ineq_cycle_correlators[19:26])

println("Constant term: ", 8*constant_term)

A correlators: [-9.0, -4.0]
B correlators: [-9.0, -4.0]
C correlators: [-9.0, -4.0]
AB correlators: [-72.0, 9.0, 9.0, 9.0]
AC correlators: [-72.0, 9.0, 9.0, 9.0]
BC correlators: [-72.0, 9.0, 9.0, 9.0]
ABC correlators: [9.0, -5.0, -5.0, 0.0, -5.0, 0.0, 0.0, 9.0]
Constant term: 403.0


In [93]:
M29_correlators_full = convert_prob_ineq_to_correlators_full(M_29)[1];
constant_term = convert_prob_ineq_to_correlators_full(M_29)[2];

In [94]:
projectNS_M29_correlators = project_correlators_to_nosignal(M29_correlators_full)

26-element Vector{Float64}:
  0.0625
 -0.1875
  0.0625
  0.0625
  0.1875
 -0.1875
 -0.125
  0.4375
  0.3125
 -0.125
 -0.25
  0.3125
  0.1875
 -0.125
  0.0
  0.1875
  0.0625
 -0.125
  0.125
  0.25
  0.0
 -0.0625
  0.25
 -0.0625
 -0.0625
  0.0625

In [95]:
# Visualize as partes:
println("A correlators: ", 16*projectNS_M29_correlators[1:2])
println("B correlators: ", 16*projectNS_M29_correlators[3:4])
println("C correlators: ", 16*projectNS_M29_correlators[5:6])
println("AB correlators: ", 16*projectNS_M29_correlators[7:10])
println("AC correlators: ", 16*projectNS_M29_correlators[11:14])
println("BC correlators: ", 16*projectNS_M29_correlators[15:18])
println("ABC correlators: ", 16*projectNS_M29_correlators[19:26])

println("Constant term: ", 16*constant_term)

A correlators: [1.0, -3.0]
B correlators: [1.0, 1.0]
C correlators: [3.0, -3.0]
AB correlators: [-2.0, 7.0, 5.0, -2.0]
AC correlators: [-4.0, 5.0, 3.0, -2.0]
BC correlators: [0.0, 3.0, 1.0, -2.0]
ABC correlators: [2.0, 4.0, 0.0, -1.0, 4.0, -1.0, -1.0, 1.0]
Constant term: 52.0


In [96]:
M29_dist_correlators_full = convert_prob_ineq_to_correlators_full(M29_disturbed1)[1];
constant_term = convert_prob_ineq_to_correlators_full(M29_disturbed1)[2];

In [97]:
projectNS_M29_dist_correlators = project_correlators_to_nosignal(M29_dist_correlators_full)

26-element Vector{Float64}:
  0.0625
  0.0625
  0.0625
  0.0625
  0.0625
  0.0625
 -0.125
  0.4375
  0.0625
 -0.125
 -0.125
  0.0625
  0.4375
 -0.125
 -0.125
  0.4375
  0.0625
 -0.125
  0.25
  0.0
  0.0
 -0.0625
  0.0
 -0.0625
 -0.0625
  0.0625

In [98]:
# Visualize as partes:
println("A correlators: ", 16*projectNS_M29_dist_correlators[1:2])
println("B correlators: ", 16*projectNS_M29_dist_correlators[3:4])
println("C correlators: ", 16*projectNS_M29_dist_correlators[5:6])
println("AB correlators: ", 16*projectNS_M29_dist_correlators[7:10])
println("AC correlators: ", 16*projectNS_M29_dist_correlators[11:14])
println("BC correlators: ", 16*projectNS_M29_dist_correlators[15:18])
println("ABC correlators: ", 16*projectNS_M29_dist_correlators[19:26])

println("Constant term: ", 16*constant_term)

A correlators: [1.0, 1.0]
B correlators: [1.0, 1.0]
C correlators: [1.0, 1.0]
AB correlators: [-2.0, 7.0, 1.0, -2.0]
AC correlators: [-2.0, 1.0, 7.0, -2.0]
BC correlators: [-2.0, 7.0, 1.0, -2.0]
ABC correlators: [4.0, 0.0, 0.0, -1.0, 0.0, -1.0, -1.0, 1.0]
Constant term: 52.0


### Esse da certinho com o encontrado na eq. (18). Basta só inverter a ordem da ineq multiplicando por -1. Repare que -20*8 + 403 = 243. 

## Evaluating the Inequality over Deterministic Behaviours

Now that we have defined the inequality vector `Ineq_cycle`, we can proceed to evaluate its value over all deterministic behaviours corresponding to the *tripartite cycle scenario*.  

### Scenario specifications:

- **Number of parties**: $n_p = 3$
- **Outcomes per measurement**: $n_o = 2$
- **Inputs per measurement**: $n_m = 2$

We will consider **both** communication patterns:  

1. **Clockwise cycle**:
   - Party 1 depends on inputs $\{3, 1\}$
   - Party 2 depends on inputs $\{1, 2\}$
   - Party 3 depends on inputs $\{2, 3\}$

2. **Counter-clockwise cycle**:
   - Party 1 depends on inputs $\{1, 2\}$
   - Party 2 depends on inputs $\{2, 3\}$
   - Party 3 depends on inputs $\{3, 1\}$

For each communication pattern, we generate the corresponding set of deterministic behaviours.  

We then **combine** these two sets by taking their **union**, obtaining the complete set of deterministic behaviours allowed by the *tripartite cycle scenario*.  

For each deterministic behaviour in this combined set, we compute the **inner product** between the behaviour vector and the inequality vector `Ineq_clock`. This gives the value of the Bell expression for each deterministic point.  

The result is a list of scalar values representing the inequality evaluated over all possible deterministic behaviours in this scenario.  

Finally, we will extract the **minimum** value of this functional over the entire set of deterministic behaviours from both the Clockwise and Counter-clockwise cycles.  

The following cell implements this computation.


In [99]:
using LinearAlgebra  # Import required for the dot function

# Scenario parameters
n_p = 3
n_o_list = 2   # Number of outcomes for each party
n_m = 2        # Number of inputs for each party

# Conditional dependencies for each party
subsets_clockwise = [[3, 1], [1, 2], [2, 3]]
subsets_counter_clockwise = [[1, 2], [2, 3], [3, 1]]

# Generate all multipartite deterministic behaviours
det_behaviours_clockwise = generate_multipartite_deterministic_behaviours(n_o_list, n_p, n_m, subsets_clockwise)
det_behaviours_counter_clockwise = generate_multipartite_deterministic_behaviours(n_o_list, n_p, n_m, subsets_counter_clockwise)

# Combine both sets of deterministic behaviours
all_det_behaviours = vcat(det_behaviours_clockwise, det_behaviours_counter_clockwise)

# Compute the inner product of each behaviour with Ineq_cycle
inner_products = [dot(Ineq_cycle, behaviour) for behaviour in all_det_behaviours]

# Find the minimum value of the functional over all deterministic behaviours
minimal_value = minimum(inner_products)

println("Minimum value of the inequality over all deterministic behaviours: ", minimal_value)

Minimum value of the inequality over all deterministic behaviours: 20.0


## Aqui to encontrando as desigualdades do cenário cycle.

## Para a classe 29

In [100]:
using JuMP, GLPK

# Replace with your actual p29 and psl
p29 = NS_class29_fullprob_vector  # example p29 vector
psl = behaviours_matrix  # each row is a p_sl^i vector

# Example data: dimension and number of p_sl vectors
d = length(p29)  # dimension of the vectors
n = size(behaviours_matrix)[1]   # number of p_sl^i vectors

# Create optimization model
model = Model(GLPK.Optimizer)

# Define variables: c >= 0
@variable(model, c[1:d] >= 0)

# Objective: minimize dot(c, p29)
@objective(model, Min, sum(c[j] * p29[j] for j in 1:d))

# Constraints: dot(c, psl_i) >= 1 for all i
for i in 1:n
    @constraint(model, sum(c[j] * psl[i, j] for j in 1:d) >= 1)
end

# Printing the prepared optimization model
#print(model)

# Solve the problem
optimize!(model)

# Check solver status
println("Solver status: ", termination_status(model))

# Print optimal solution
#println("Optimal solution for c:")
#for j in 1:d
#    println("c[$j] = ", value(c[j]))
#end

# Print optimal objective value
println("Optimal objective value: ", objective_value(model))

Solver status: OPTIMAL
Optimal objective value: 0.49999999999999994


In [101]:
c29_opt_cycle = value.(c);
c29_opt_cycle_round = round.(c29_opt_cycle, digits=4);
M_29_cycle = reshape(c29_opt_cycle_round, 8, 8)'

8×8 adjoint(::Matrix{Float64}) with eltype Float64:
 0.0  1.0   1.0  0.5  1.0  0.5   0.5  0.0
 0.5  0.0   0.5  0.5  0.0  1.0  -0.0  0.0
 0.0  0.0   0.0  1.0  0.5  0.0   0.5  0.0
 1.0  1.0   1.0  0.0  0.0  0.5   0.0  0.5
 0.0  0.5   1.0  0.0  0.0  0.5   1.0  0.0
 0.5  1.0   0.0  0.0  1.0  0.0   0.5  0.5
 1.0  0.0  -0.0  0.5  1.0  0.0   0.0  0.5
 1.0  0.5   0.5  0.0  0.5  0.0   0.0  1.0

In [102]:
M_29_cycle - M_29

8×8 Matrix{Float64}:
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0

### Segue que M_29_cycle é diferente da M_29.

In [103]:
Ineq_cycle = [0 12 12 15 12 15 15 0;
         0 4 15 12 15 12 5 0;
         0 15 4 12 15 5 12 0;
         8 4 4 0 0 5 5 10;
         0 15 15 5 4 12 12 0;
         8 4 0 5 4 0 5 10;
         8 0 4 5 4 5 0 10;
         5 0 0 4 0 4 4 8]
## Essa é a desigualdade que tá no artigo. Dá pra ver que não é a mesma que a encontrada aqui

8×8 Matrix{Int64}:
 0  12  12  15  12  15  15   0
 0   4  15  12  15  12   5   0
 0  15   4  12  15   5  12   0
 8   4   4   0   0   5   5  10
 0  15  15   5   4  12  12   0
 8   4   0   5   4   0   5  10
 8   0   4   5   4   5   0  10
 5   0   0   4   0   4   4   8

### Aqui só to confirmando que essa desigualdade nova associado ao vertice 29 tem maximo 1 nos clássicos

In [104]:
using LinearAlgebra  # Import required for the dot function

# Scenario parameters
n_p = 3
n_o_list = 2   # Number of outcomes for each party
n_m = 2   # Number of inputs for each party

# Conditional dependencies for each party
subsets_clockwise = [[3, 1], [1, 2], [2, 3]]
subsets_counter_clockwise = [[1, 2], [2, 3], [3, 1]]

# Generate all multipartite deterministic behaviours
det_behaviours_clockwise = generate_multipartite_deterministic_behaviours(n_o_list, n_p, n_m, subsets_clockwise)
det_behaviours_counter_clockwise = generate_multipartite_deterministic_behaviours(n_o_list, n_p, n_m, subsets_counter_clockwise)

# Combine both sets of deterministic behaviours
all_det_behaviours = vcat(det_behaviours_clockwise, det_behaviours_counter_clockwise)

# Compute the inner product of each behaviour with Ineq_cycle
inner_products = [dot(c29_opt_cycle_round, behaviour) for behaviour in all_det_behaviours]

# Find the minimum value of the functional over all deterministic behaviours
minimal_value = minimum(inner_products)

println("Minimum value of the inequality over all deterministic behaviours: ", minimal_value)

Minimum value of the inequality over all deterministic behaviours: 1.0


## Abaixo eu confiro se a desigualdade obtida anterior (no caso horario) é violada por esse conjunto maior de pontos deterministicos (ciclos). Como da para ver, não é violada não então continua sendo uma desigualdade valida aqui também.

In [105]:
# Compute the inner product of each behaviour with Ineq_cycle
inner_products = [dot(c29_opt_round, behaviour) for behaviour in all_det_behaviours]

# Find the minimum value of the functional over all deterministic behaviours
minimal_value = minimum(inner_products)

println("Minimum value of the inequality over all deterministic behaviours: ", minimal_value)

Minimum value of the inequality over all deterministic behaviours: 1.0


## Como não é violada (melhor dizendo, tem mesma cota no det horarios e nos dets ciclos), significa que a desigualdade `M_29`também é uma desigualdade válida para o cenário ciclo.

In [106]:
dot(c29_opt_cycle_round,p29)

0.5

## Calculando a dimensão do politopo da BDAG ciclica.

In [107]:
using LinearAlgebra

# Matrix contaning all the behaviour
behaviours_matrix = reduce(vcat, [Float64.(b)' for b in all_det_behaviours])

# Initialize the difference matrix
diff_matrix = zeros(size(behaviours_matrix))

# Subtract the first behaviour from each row explicitly
for i in 1:size(behaviours_matrix, 1)
    diff_matrix[i, :] = behaviours_matrix[i, :] - behaviours_matrix[1, :]
end

# The affine dimension is the rank of this matrix
affine_dim = rank(diff_matrix)

println("Affine dimension of the saturating set: ", affine_dim)

Affine dimension of the saturating set: 44


In [108]:
using LinearAlgebra
# Compute the inner product of each behaviour with Ineq_cycle
inner_products = [dot(c29_opt_cycle_round, behaviour) for behaviour in all_det_behaviours]

# Identify behaviours that saturate the inequality
min_behaviours = [b for (i, b) in enumerate(all_det_behaviours) if abs(inner_products[i] - minimal_value) < 10^(-10)]

# Stack the behaviours into a matrix for affine dimension computation
M = hcat(min_behaviours...)'

# Initialize the difference matrix
diff_matrix = zeros(size(M))

# Subtract the first behaviour from each row explicitly
for i in 1:size(M, 1)
    diff_matrix[i, :] = M[i, :] - M[1, :]
end

# The affine dimension is the rank of this matrix
affine_dim = rank(diff_matrix)

println("Affine dimension of the saturating set: ", affine_dim)

Affine dimension of the saturating set: 41


### Vamos ver agora se as desigualdades anteriores eram faceta

In [109]:
using LinearAlgebra
# Compute the inner product of each behaviour with Ineq_cycle
inner_products = [dot(c29_opt_round, behaviour) for behaviour in all_det_behaviours]

# Identify behaviours that saturate the inequality
min_behaviours = [b for (i, b) in enumerate(all_det_behaviours) if abs(inner_products[i] - minimal_value) < 10^(-10)]

# Stack the behaviours into a matrix for affine dimension computation
M = hcat(min_behaviours...)'

# Initialize the difference matrix
diff_matrix = zeros(size(M))

# Subtract the first behaviour from each row explicitly
for i in 1:size(M, 1)
    diff_matrix[i, :] = M[i, :] - M[1, :]
end

# The affine dimension is the rank of this matrix
affine_dim = rank(diff_matrix)

println("Affine dimension of the saturating set: ", affine_dim)

Affine dimension of the saturating set: 41


In [110]:
using LinearAlgebra
# Compute the inner product of each behaviour with Ineq_cycle
inner_products = [dot(c29_disturbed_round1, behaviour) for behaviour in all_det_behaviours]

# Find the minimum value of the functional over all deterministic behaviours
minimal_value = minimum(inner_products)
println("minimal valeu: ", minimal_value)

# Identify behaviours that saturate the inequality
min_behaviours = [b for (i, b) in enumerate(all_det_behaviours) if abs(inner_products[i] - minimal_value) < 10^(-6)]

# Stack the behaviours into a matrix for affine dimension computation
M = hcat(min_behaviours...)'

# Initialize the difference matrix
diff_matrix = zeros(size(M))

# Subtract the first behaviour from each row explicitly
for i in 1:size(M, 1)
    diff_matrix[i, :] = M[i, :] - M[1, :]
end

# The affine dimension is the rank of this matrix
affine_dim = rank(diff_matrix)

println("Affine dimension of the saturating set: ", affine_dim)

minimal valeu: 1.0
Affine dimension of the saturating set: 43


## É FACETA

In [111]:
using LinearAlgebra
# Compute the inner product of each behaviour with Ineq_cycle
inner_products = [dot(c29_disturbed_round2, behaviour) for behaviour in all_det_behaviours]

# Find the minimum value of the functional over all deterministic behaviours
minimal_value = minimum(inner_products)
println("minimal valeu: ", minimal_value)

# Identify behaviours that saturate the inequality
min_behaviours = [b for (i, b) in enumerate(all_det_behaviours) if abs(inner_products[i] - minimal_value) < 10^(-6)]

# Stack the behaviours into a matrix for affine dimension computation
M = hcat(min_behaviours...)'

# Initialize the difference matrix
diff_matrix = zeros(size(M))

# Subtract the first behaviour from each row explicitly
for i in 1:size(M, 1)
    diff_matrix[i, :] = M[i, :] - M[1, :]
end

# The affine dimension is the rank of this matrix
affine_dim = rank(diff_matrix)

println("Affine dimension of the saturating set: ", affine_dim)

minimal valeu: 1.0
Affine dimension of the saturating set: 43


## Também é faceta, mas lembrando que essa não tem violação quântica

### Vamos verificar se a desigualdade do paper é faceta

In [112]:
using LinearAlgebra
# Compute the inner product of each behaviour with Ineq_cycle
inner_products = [dot(Ineq_cycle, behaviour) for behaviour in all_det_behaviours]

# Find the minimum value of the functional over all deterministic behaviours
minimal_value = minimum(inner_products)
println("minimal valeu: ", minimal_value)

# Identify behaviours that saturate the inequality
min_behaviours = [b for (i, b) in enumerate(all_det_behaviours) if abs(inner_products[i] - minimal_value) < 10^(-6)]

# Stack the behaviours into a matrix for affine dimension computation
M = hcat(min_behaviours...)'

# Initialize the difference matrix
diff_matrix = zeros(size(M))

# Subtract the first behaviour from each row explicitly
for i in 1:size(M, 1)
    diff_matrix[i, :] = M[i, :] - M[1, :]
end

# The affine dimension is the rank of this matrix
affine_dim = rank(diff_matrix)

println("Affine dimension of the saturating set: ", affine_dim)

minimal valeu: 18.0
Affine dimension of the saturating set: 7


## Essa deu longe de uma faceta!

## Upper Bounds on the Quantum Value using NPA Hierarchy

In [113]:
function format_sums(M)
    comb = [
        (0,0,0),
        (0,0,1),
        (0,1,0),
        (0,1,1),
        (1,0,0),
        (1,0,1),
        (1,1,0),
        (1,1,1)
    ]

    for i in 1:size(M,1)
        x4, x5, x6 = comb[i]
        termos = []

        for j in 1:size(M,2)
            coef = M[i, j]
            if coef != 0.0
                x1, x2, x3 = comb[j]
                if coef == 1.0
                    push!(termos, "p($x1,$x2,$x3,$x4,$x5,$x6)")
                else
                    push!(termos, "$(coef)*p($x1,$x2,$x3,$x4,$x5,$x6)")
                end
            end
        end

        if isempty(termos)
            expr = "0"
        else
            expr = join(termos, "+")
        end

        println("S_$(x4)$(x5)$(x6) = $expr\n")
    end
end

format_sums (generic function with 1 method)

In [114]:
Ineq_cycle = [0 12 12 15 12 15 15 0;
         0 4 15 12 15 12 5 0;
         0 15 4 12 15 5 12 0;
         8 4 4 0 0 5 5 10;
         0 15 15 5 4 12 12 0;
         8 4 0 5 4 0 5 10;
         8 0 4 5 4 5 0 10;
         5 0 0 4 0 4 4 8]

8×8 Matrix{Int64}:
 0  12  12  15  12  15  15   0
 0   4  15  12  15  12   5   0
 0  15   4  12  15   5  12   0
 8   4   4   0   0   5   5  10
 0  15  15   5   4  12  12   0
 8   4   0   5   4   0   5  10
 8   0   4   5   4   5   0  10
 5   0   0   4   0   4   4   8

In [115]:
format_sums(Ineq_cycle)

S_000 = 12*p(0,0,1,0,0,0)+12*p(0,1,0,0,0,0)+15*p(0,1,1,0,0,0)+12*p(1,0,0,0,0,0)+15*p(1,0,1,0,0,0)+15*p(1,1,0,0,0,0)

S_001 = 4*p(0,0,1,0,0,1)+15*p(0,1,0,0,0,1)+12*p(0,1,1,0,0,1)+15*p(1,0,0,0,0,1)+12*p(1,0,1,0,0,1)+5*p(1,1,0,0,0,1)

S_010 = 15*p(0,0,1,0,1,0)+4*p(0,1,0,0,1,0)+12*p(0,1,1,0,1,0)+15*p(1,0,0,0,1,0)+5*p(1,0,1,0,1,0)+12*p(1,1,0,0,1,0)

S_011 = 8*p(0,0,0,0,1,1)+4*p(0,0,1,0,1,1)+4*p(0,1,0,0,1,1)+5*p(1,0,1,0,1,1)+5*p(1,1,0,0,1,1)+10*p(1,1,1,0,1,1)

S_100 = 15*p(0,0,1,1,0,0)+15*p(0,1,0,1,0,0)+5*p(0,1,1,1,0,0)+4*p(1,0,0,1,0,0)+12*p(1,0,1,1,0,0)+12*p(1,1,0,1,0,0)

S_101 = 8*p(0,0,0,1,0,1)+4*p(0,0,1,1,0,1)+5*p(0,1,1,1,0,1)+4*p(1,0,0,1,0,1)+5*p(1,1,0,1,0,1)+10*p(1,1,1,1,0,1)

S_110 = 8*p(0,0,0,1,1,0)+4*p(0,1,0,1,1,0)+5*p(0,1,1,1,1,0)+4*p(1,0,0,1,1,0)+5*p(1,0,1,1,1,0)+10*p(1,1,1,1,1,0)

S_111 = 5*p(0,0,0,1,1,1)+4*p(0,1,1,1,1,1)+4*p(1,0,1,1,1,1)+4*p(1,1,0,1,1,1)+8*p(1,1,1,1,1,1)



In [116]:
using QuantumNPA
PA = projector(1, 1:2, 1:2, full=true);
PB = projector(2, 1:2, 1:2, full=true);
PC = projector(3, 1:2, 1:2, full=true);

p(a,b,c,x,y,z) = PA[a+1,x+1]*PB[b+1,y+1]*PC[c+1,z+1]

S_000 = 12*p(0,0,1,0,0,0)+12*p(0,1,0,0,0,0)+15*p(0,1,1,0,0,0)+12*p(1,0,0,0,0,0)+15*p(1,0,1,0,0,0)+15*p(1,1,0,0,0,0)

S_001 = 4*p(0,0,1,0,0,1)+15*p(0,1,0,0,0,1)+12*p(0,1,1,0,0,1)+15*p(1,0,0,0,0,1)+12*p(1,0,1,0,0,1)+5*p(1,1,0,0,0,1)

S_010 = 15*p(0,0,1,0,1,0)+4*p(0,1,0,0,1,0)+12*p(0,1,1,0,1,0)+15*p(1,0,0,0,1,0)+5*p(1,0,1,0,1,0)+12*p(1,1,0,0,1,0)

S_011 = 8*p(0,0,0,0,1,1)+4*p(0,0,1,0,1,1)+4*p(0,1,0,0,1,1)+5*p(1,0,1,0,1,1)+5*p(1,1,0,0,1,1)+10*p(1,1,1,0,1,1)

S_100 = 15*p(0,0,1,1,0,0)+15*p(0,1,0,1,0,0)+5*p(0,1,1,1,0,0)+4*p(1,0,0,1,0,0)+12*p(1,0,1,1,0,0)+12*p(1,1,0,1,0,0)

S_101 = 8*p(0,0,0,1,0,1)+4*p(0,0,1,1,0,1)+5*p(0,1,1,1,0,1)+4*p(1,0,0,1,0,1)+5*p(1,1,0,1,0,1)+10*p(1,1,1,1,0,1)

S_110 = 8*p(0,0,0,1,1,0)+4*p(0,1,0,1,1,0)+5*p(0,1,1,1,1,0)+4*p(1,0,0,1,1,0)+5*p(1,0,1,1,1,0)+10*p(1,1,1,1,1,0)

S_111 = 5*p(0,0,0,1,1,1)+4*p(0,1,1,1,1,1)+4*p(1,0,1,1,1,1)+4*p(1,1,0,1,1,1)+8*p(1,1,1,1,1,1)

S = S_000 + S_001 + S_010 + S_011 + S_100 + S_101 + S_110 + S_111 

38 Id + 29 PA1|1 - 9 PA1|2 + 29 PB1|1 - 9 PB1|2 + 29 PC1|1 - 9 PC1|2 - 38 PA1|1 PB1|1 + 7 PA1|1 PB1|2 - 38 PA1|1 PC1|1 + 7 PA1|1 PC1|2 + 7 PA1|2 PB1|1 + 7 PA1|2 PC1|1 - 38 PB1|1 PC1|1 + 7 PB1|1 PC1|2 + 7 PB1|2 PC1|1 + 9 PA1|1 PB1|1 PC1|1 - 5 PA1|1 PB1|1 PC1|2 - 5 PA1|1 PB1|2 PC1|1 - 5 PA1|2 PB1|1 PC1|1 + 9 PA1|2 PB1|2 PC1|2

In [117]:
npa_min(S,4), npa_min(S,3), npa_min(S,2)

(19.73539027128424, 19.73538332479489, 19.00710541471535)

In [118]:
20/19.73539

1.0134078931300572

### Comparando com a desigualdade que é faceta encontrada:

In [119]:
1/0.9406541466968572

1.0630899821273716

### Tem uma violação

## M_29_disturbed1 tem um gap melhor

## Checking if the Inequality Defines a Facet

To verify whether the inequality vector `Ineq_clock` represents a **facet** of the polytope defined by the deterministic behaviours, we need to check the **affine dimension** of the set of behaviours that **saturate** the inequality, i.e., those for which the inner product with `Ineq_clock` achieves the **minimal value**.

### Procedure:

1. Identify all deterministic behaviours that minimize the value of the inequality.
2. Compute the **affine dimension** of this set.
3. Compare the affine dimension with the expected dimension for a facet.

If the affine dimension of the saturating set is equal to $\text{dim}(\mathcal{P}) - 1$, where $\mathcal{P}$ is the polytope of all admissible behaviours, then the inequality defines a **facet** of this polytope.

The following cell implements this verification.


In [120]:
using LinearAlgebra

# Identify behaviours that saturate the inequality
minimal_behaviours = [b for (i, b) in enumerate(all_det_behaviours) if inner_products[i] == minimal_value]

# Stack the behaviours into a matrix for affine dimension computation
N = hcat(minimal_behaviours...)'

# Initialize the difference matrix
diffe_matrix = zeros(size(N))

# Subtract the first behaviour from each row explicitly
for i in 1:size(N, 1)
    diffe_matrix[i, :] = N[i, :] - N[1, :]
end

# The affine dimension is the rank of this matrix
affine_dimension = rank(diffe_matrix)

println("Affine dimension of the saturating set: ", affine_dimension)

Affine dimension of the saturating set: 7


## Evaluating the Inequality over Additional Deterministic Sets

For completeness, we will also evaluate the inequality vector `Ineq_cycle` over:

- The **minimal Boring scenario deterministic behaviours**, corresponding to the tripartite cycle scenario with reversed communication:  
  - Party 1 depends on $\{1\}$  
  - Party 2 depends on $\{1, 2\}$  
  - Party 3 depends on $\{1, 2, 3\}$  
  It is known that this scenario has as a subset all **non-signaling** behaviours and hence all **quantum** behaviours.

We compute the inner product with `Ineq_cycle` and extract the **minimum value** of the functional.  

As we will see below, the minimum of this functional is **8**, which is **lower** than the **quantum value** that we will compute next.

In [121]:
using LinearAlgebra  # Import required for the dot function

# Scenario parameters
n_p = 3
n_o_list = 2   # Number of outcomes for each party
n_m = 2   # Number of inputs for each party

# Conditional dependencies for each party (Boring scenario)
subsets_list = [[1], [1, 2], [1, 2, 3]]

# Generate all multipartite deterministic behaviours
det_behaviours = generate_multipartite_deterministic_behaviours(n_o_list, n_p, n_m, subsets_list)

# Compute the inner product of each behaviour with Ineq_clock
inner_products = [dot(Ineq_cycle, behaviour) for behaviour in det_behaviours]

# Find the minimum value of the functional over all deterministic behaviours
min_value = minimum(inner_products)

println("Minimum value of the inequality over det behaviours of a boring scenario: ", min_value)

Minimum value of the inequality over det behaviours of a boring scenario: 17.0


# Quantum Behaviour and Bell Inequality Violation

Now, we will define a **quantum behaviour** for the tripartite Bell scenario, aiming to find a quantum probability distribution that achieves a **lower value** of the Bell functional than any deterministic behaviour.

## Quantum Measurement Scenario

The quantum behaviour is defined as:

$$
p(a,b,c|x,y,z) = \langle \psi | E_{a|x} \otimes F_{b|y} \otimes G_{c|z} |\psi \rangle
$$

where:

- $E_{0|0} = F_{0|0} = G_{0|0} = |0\rangle \langle 0|$
- $E_{1|0} = F_{1|0} = G_{1|0} = |1\rangle \langle 1|$
  
and:

- $|\phi\rangle = \cos(\alpha) |0\rangle + \sin(\alpha) |1\rangle$
- $E_{0|1} = F_{0|1} = G_{0|1} = |\phi\rangle \langle \phi|$
- $E_{1|1} = F_{1|1} = G_{1|1} = \mathbb{I} - |\phi\rangle \langle \phi|$

with $\alpha = \pi/2.8955$.

The optimal **quantum state** is:

$$
|\psi\rangle = \cos\theta(\cos\varphi |000\rangle + \sin\varphi |111\rangle) + \sin\theta(\cos \delta |W_0\rangle + \sin \delta |W_1\rangle),
$$

where:

- $|W_0\rangle = \frac{1}{\sqrt{3}}(|100\rangle + |010\rangle + |001\rangle)$
- $|W_1\rangle = \frac{1}{\sqrt{3}}(|011\rangle + |101\rangle + |110\rangle)$

and parameters:

- $\theta =  0.041638$
- $\varphi = 5.78939$
- $\delta = 5.182659$

We will now implement this in Julia.

In [122]:
using LinearAlgebra
using KroneckerProducts

# Basic states
ket0 = [1.0; 0.0]
ket1 = [0.0; 1.0]

# Projectors
proj0 = ket0 * ket0'
proj1 = ket1 * ket1'

# Measurement in phi basis
alpha = pi/2.8955
phi = cos(alpha) * ket0 + sin(alpha) * ket1
proj_phi = phi * phi'
id2 = I(2)

# Measurements for all parties
E = Dict((0,0)=>proj0, (1,0)=>proj1, (0,1)=>proj_phi, (1,1)=>(id2 - proj_phi))
F = E  # identical
G = E  # identical

# Define W states
W0 = (kron(ket1, ket0, ket0) + kron(ket0, ket1, ket0) + kron(ket0, ket0, ket1)) / sqrt(3)
W1 = (kron(ket0, ket1, ket1) + kron(ket1, ket0, ket1) + kron(ket1, ket1, ket0)) / sqrt(3)

# Parameters
θ = 0.041638
φ = 5.78939
δ = 5.182659

# State construction
psi = cos(θ) * (cos(φ) * kron(ket0, ket0, ket0) + sin(φ) * kron(ket1, ket1, ket1)) + 
      sin(θ) * (cos(δ) * W0 + sin(δ) * W1)

# Initialize quantum behaviour vector
behaviour = zeros(2^3 * 2^3)  # 8 outcomes * 8 input combinations

# Mapping: (a,b,c,x,y,z) -> index
index_map = Dict()
counter = 1
for x in 0:1, y in 0:1, z in 0:1, a in 0:1, b in 0:1, c in 0:1
    index_map[(a,b,c,x,y,z)] = counter
    counter += 1
end

# Compute quantum behaviour
for x in 0:1, y in 0:1, z in 0:1, a in 0:1, b in 0:1, c in 0:1
    M = kron(E[(a,x)], F[(b,y)], G[(c,z)])
    p = real(psi' * M * psi)
    idx = index_map[(a,b,c,x,y,z)]
    behaviour[idx] = p
end

# Flatten Bell inequality vector
Ineq_cycle = [0 12 12 15 12 15 15 0;
         0 4 15 12 15 12 5 0;
         0 15 4 12 15 5 12 0;
         8 4 4 0 0 5 5 10;
         0 15 15 5 4 12 12 0;
         8 4 0 5 4 0 5 10;
         8 0 4 5 4 5 0 10;
         5 0 0 4 0 4 4 8]
Ineq_cycle = vec(Ineq_cycle');

# Compute inner product
ineq_value = dot(Ineq_cycle, behaviour)

println("Value of the Bell functional for the quantum behaviour: ", ineq_value)


Value of the Bell functional for the quantum behaviour: 19.734369765878924


### Correlator representation

Remembering that, in correlator notation and assuming no signaling, the inequality is given by:
\begin{equation}
\begin{aligned}
    % 
    &9\left<A_{0}\right> + 4\left<A_{1}\right> + 9\left<B_{0}\right> 
    + 4\left<B_{1}\right> + 9\left<C_{0}\right> + 4\left<C_{1}\right>
    + 72\left<A_{0}B_{0}\right> - 9\left<A_{1}B_{0}\right>
    - 9\left<A_{0}B_{1}\right> - 9\left<A_{1}B_{1}\right> \\
    &+ 72\left<A_{0}C_{0}\right> - 9\left<A_{1}C_{0}\right>
    - 9\left<A_{0}C_{1}\right> - 9\left<A_{1}C_{1}\right>
    + 72\left<B_{0}C_{0}\right> - 9\left<B_{1}C_{0}\right> 
    - 9\left<B_{0}C_{1}\right> - 9\left<B_{1}C_{1}\right> \\
    &- 9\left<A_{0}B_{0}C_{0}\right> + 5\left<A_{1}B_{0}C_{0}\right> + 
    5\left<A_{0}B_{1}C_{0}\right>
    + 5\left<A_{0}B_{0}C_{1}\right> - 9\left<A_{1}B_{1}C_{1}\right> \leq 243.
\end{aligned}
\end{equation}

In [123]:
using QuantumNPA  # Load QuantumNPA package for NPA hierarchy computations

@dichotomic A[1:2] B[1:2] C[1:2];  # Define dichotomic observables A, B, C with 2 inputs each


S  = 9*A[1] + 4*A[2] + 9*B[1] + 4*B[2] + 9*C[1] + 4*C[2] +      # linear terms
    72*A[1]*B[1] - 9*A[2]*B[1] - 9*A[1]*B[2] - 9*A[2]*B[2] +   # A-B correlators
    72*A[1]*C[1] - 9*A[2]*C[1] - 9*A[1]*C[2] - 9*A[2]*C[2] +   # A-C correlators
    72*B[1]*C[1] - 9*B[2]*C[1] - 9*B[1]*C[2] - 9*B[2]*C[2] -   # B-C correlators
    9*A[1]*B[1]*C[1] + 5*A[2]*B[1]*C[1] + 5*A[1]*B[2]*C[1] +   
    5*A[1]*B[1]*C[2] - 9*A[2]*B[2]*C[2]                        # tripartite correlator 

#NPA_lvl4_cor = npa_max(S, "4")  # Compute the maximal quantum value of S using NPA hierarchy at level 4

9 A1 + 4 A2 + 9 B1 + 4 B2 + 9 C1 + 4 C2 + 72 A1 B1 - 9 A1 B2 + 72 A1 C1 - 9 A1 C2 - 9 A2 B1 - 9 A2 B2 - 9 A2 C1 - 9 A2 C2 + 72 B1 C1 - 9 B1 C2 - 9 B2 C1 - 9 B2 C2 - 9 A1 B1 C1 + 5 A1 B1 C2 + 5 A1 B2 C1 + 5 A2 B1 C1 - 9 A2 B2 C2

#### Converting the bounds in correlators to the bounds in full probabilities

In [124]:
#NPA_lvl4_prob = -NPA_lvl4_cor/8 + 403/8 

### NPA Bounds at Lower Levels

Just for verification, we will now compute the **NPA bounds** of the inequality at lower levels of the hierarchy.

In [125]:
#npa_max(S, "1 + A B + A C + B C")

In [126]:
#npa_max(S, "2")

In [127]:
#npa_max(S, "3")

# Gerando os pontos extremais em correlatores cheios para o cenário relogio e o cenario ciclo

### Full Correlators from Behaviour Probabilities

Let us define a function that maps a behaviour given in the full probability representation $p(abc|xyz)$ to the corresponding full correlators $\langle ABC_{xyz} \rangle$.

The correlators are computed using the formula:

$$
\langle ABC_{xyz} \rangle = \sum_{a,b,c \in \{-1,1\}} abc \cdot p(abc|xyz)
$$

where the inputs $x, y, z$ and outputs $a, b, c$ range over $\{-1,1\}$.


In [128]:
function correlators_from_probabilities(p)
    correlators = zeros(8)  # for each xyz ∈ {000, 001, ..., 111}

    for x in 0:1, y in 0:1, z in 0:1
        total = 0.0

        for a in 0:1, b in 0:1, c in 0:1
            # Mapear 0 -> -1, 1 -> +1
            A = 2a - 1
            B = 2b - 1
            C = 2c - 1

            # Índice da probabilidade p(abc|xyz)
            index = c + 2b + 4a + 8z + 16y + 32x + 1

            total += A * B * C * p[index]
        end

        # The coordinate in the correlator:
        pos = z + 2y + 4x + 1
        correlators[pos] = total
    end

    return correlators
end


correlators_from_probabilities (generic function with 1 method)

In [129]:
function correlators_from_probabilities(p)
    correlators = zeros(8)
    index = 1
    for x in 0:1, y in 0:1, z in 0:1
        s = 0.0
        for a in 0:1, b in 0:1, c in 0:1
            a_val = 2a - 1
            b_val = 2b - 1
            c_val = 2c - 1
            prob_index = a + 2b + 4c + 8x + 16y + 32z + 1
            s += a_val * b_val * c_val * p[prob_index]
        end
        correlators[index] = s
        index += 1
    end
    @show correlators
    return correlators
end

correlators_from_probabilities (generic function with 1 method)

### Applying the Correlator Function to the Extremal Points of the Classical BDAG

We now apply the `correlators_from_probabilities` function to the list of extremal points of the classical correlation set defined by the BDAG with a clockwise structure. Each of these extremal points is given as a behaviour in full probability notation, and we will project them onto the correlator space to facilitate comparison and further analysis.


In [130]:
# Scenario parameters
n_p = 3
n_o_list = 2   # Number of outcomes for each party
n_m = 2   # Number of inputs for each party

# Conditional dependencies for each party (clock-wise)
subsets_list = [[3, 1], [1, 2], [2, 3]]

# Generate all multipartite deterministic behaviours
det_behaviours = generate_multipartite_deterministic_behaviours(n_o_list, n_p, n_m, subsets_list)

det_behaviours_matrix = reduce(vcat, [Float64.(b)' for b in det_behaviours])

4096×64 Matrix{Float64}:
 1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     1.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0     1.0  0.0  0.0  0.0  0.0  0.0  0.0
 1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  1.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0     1.0  0.0  0.0  0.0  0.0  0.0  0.0
 1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0

In [131]:
a =det_behaviours_matrix[1,:]

64-element Vector{Float64}:
 1.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 1.0
 0.0
 0.0
 0.0
 0.0
 ⋮
 0.0
 0.0
 0.0
 0.0
 1.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0
 0.0

In [132]:
correlators_from_probabilities(a)

correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]


8-element Vector{Float64}:
 -1.0
 -1.0
 -1.0
 -1.0
 -1.0
 -1.0
 -1.0
 -1.0

In [133]:
# Aplicando a função em cada linha
correlator_matrix = [
    correlators_from_probabilities(vcat(det_behaviours_matrix[i, :]...))
    for i in 1:size(det_behaviours_matrix, 1)
];

correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [1.0, 1.0, -1.0, -1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [1.0, -1.0, -1.0, -1.0, 1.0, -1.0, -1.0, -

correlators = [-1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, -1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [1.0, 1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [1.0, 1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [1.0, 1.0, -1.0, 1.0, -1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, -1.0, -1.0, 1.0, -1.0]
correlators = [1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.0]

correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [1.0, 1.0, -1.0, -1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [1.0, 1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, -1.0, -1.0, 1.0, -1.0, -1.0, 1.

correlators = [-1.0, -1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0]
correlators = [1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0, -1.0]
correlators = [1.0, -1.0, 1.0, 1.0, -1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0, -1.0]
correlators = [1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 

correlators = [1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, -1.0, 1.0, -1.0, -1.0]
correlators = [1.0, 1.0, 1.0, -1.0, -1.0, 1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, -1.0, -1.0, 1.0, -1.0, -1.0]
correlators = [1.0, 1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0]
correlators = [1.0, 1.0, 1.0, -1.0, 1.0, -1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, -1.0, 1.0, -1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0]
correlators = [1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, 1.0, -1.0, -1.0, 1.0]
correlators = [1.0, 1.0, -1.0, -1.0, 1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, 1.0, -1.0, 

...

correlators = [1.0, 1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [1.0, 1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [1.0, 1.0, -1.0, -1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, 1.0, -1.0, -1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, 1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0]

correlators = [-1.0, -1.0, -1.0, 1.0, -1.0, 1.0, -1.0, -1.0]
correlators = [1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, 1.0]
correlators = [1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, 1.0, -1.0, 1.0, 1.0]
correlators = [1.0, 1.0, -1.0, 1.0, 1.0, -1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0]
correlators = [1.0, 1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, -1.0, 1.0, 1.0, 1.0]
correlators = [1.0, 1.0, -1.0, 1.0, -1.0, 1.0, 1.0, 1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [1.0, 1.0, -1.0, -1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [1.0, 1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0]
cor

correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [1.0, 1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [1.0, 1.0, -1.0, -1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, 1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0, 

correlators = [-1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0]
correlators = [1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0]
correlators = [-1.0, 1.0, 1.0, 1.0, 1.0, -1.0, 1.0, 1.0]
correlators = [1.0, -1.0, 1.0, 1.0, 1.0, -1.0, 1.0, 1.0]
correlators = [-1.0, 1.0, -1.0, -1.0, 1.0, -1.0, 1.0, 1.0]
correlators = [1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0, 1.0]
correlators = [-1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0, -1.0]
correlators = [1.0, -1.0, 1.0, 1.0, -1.0, 1.0, -1.0, -1.0]
correlators = [-1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.0, -1.0]
correlators = [1.0, -1.0, -1.0, -1.0, -1.0, 1.0, -1.0, -1.0]
correlators = [-1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0, -1.0]
correlators = [1.0, -1.0, 1.0, 1.0, 1.0, -1.0, -1.0, -1.0]
correlators = [-1.0, 1.0, -1.0, -1.0, 1.0, -1.0, -1.0, -1.0]
correlators = [1.0, -1.0, -1.0, -1.0, 1.0, -1.0, -1.0, -1.0]
correlators = [1.0, -1.0, 1.0, 1.0, 1.0, -1.0, 1.0, 1.0]
correlators = [-1.0, 1.0, 1.0, 1.0, 1.0, -1.0, 1.0, 1.0]
correlators = [1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0, 1.

correlators = [1.0, 1.0, -1.0, -1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [-1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.0, -

In [134]:
final_matrix = hcat(correlator_matrix...)'

4096×8 adjoint(::Matrix{Float64}) with eltype Float64:
 -1.0  -1.0  -1.0  -1.0  -1.0  -1.0  -1.0  -1.0
  1.0   1.0  -1.0  -1.0  -1.0  -1.0  -1.0  -1.0
 -1.0  -1.0   1.0   1.0  -1.0  -1.0  -1.0  -1.0
  1.0   1.0   1.0   1.0  -1.0  -1.0  -1.0  -1.0
 -1.0  -1.0  -1.0  -1.0   1.0   1.0  -1.0  -1.0
  1.0   1.0  -1.0  -1.0   1.0   1.0  -1.0  -1.0
 -1.0  -1.0   1.0   1.0   1.0   1.0  -1.0  -1.0
  1.0   1.0   1.0   1.0   1.0   1.0  -1.0  -1.0
 -1.0  -1.0  -1.0  -1.0  -1.0  -1.0   1.0   1.0
  1.0   1.0  -1.0  -1.0  -1.0  -1.0   1.0   1.0
 -1.0  -1.0   1.0   1.0  -1.0  -1.0   1.0   1.0
  1.0   1.0   1.0   1.0  -1.0  -1.0   1.0   1.0
 -1.0  -1.0  -1.0  -1.0   1.0   1.0   1.0   1.0
  ⋮                             ⋮          
 -1.0  -1.0  -1.0  -1.0   1.0   1.0  -1.0  -1.0
  1.0   1.0  -1.0  -1.0   1.0   1.0  -1.0  -1.0
 -1.0  -1.0   1.0   1.0   1.0   1.0  -1.0  -1.0
  1.0   1.0   1.0   1.0   1.0   1.0  -1.0  -1.0
 -1.0  -1.0  -1.0  -1.0  -1.0  -1.0   1.0   1.0
  1.0   1.0  -1.0  -1.0  -1.0  -1.0  

In [135]:
unique_rows = unique(eachrow(final_matrix))

128-element Vector{SubArray{Float64, 1, Matrix{Float64}, Tuple{Base.Slice{Base.OneTo{Int64}}, Int64}, true}}:
 [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
 [1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
 [-1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
 [1.0, 1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
 [-1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0, -1.0]
 [1.0, 1.0, -1.0, -1.0, 1.0, 1.0, -1.0, -1.0]
 [-1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0]
 [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0]
 [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0]
 [1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0]
 [-1.0, -1.0, 1.0, 1.0, -1.0, -1.0, 1.0, 1.0]
 [1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0, 1.0]
 [-1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
 ⋮
 [-1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0]
 [1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0]
 [-1.0, -1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0]
 [1.0, 1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0]
 [-1.0, -1.0, -1.0, -1.0, 1.0, -1.0, -1.0, 1.0]
 [1.0, 1.0, -1.0, -1.0, 1.0, -1.0, -1.0, 1.0]
 [-

In [136]:
# Scenario parameters
n_p = 3
n_o_list = 2   # Number of outcomes for each party
n_m = 2   # Number of inputs for each party

# Conditional dependencies for each party
subsets_clockwise = [[3, 1], [1, 2], [2, 3]]
subsets_counter_clockwise = [[1, 2], [2, 3], [3, 1]]

# Generate all multipartite deterministic behaviours
det_behaviours_clockwise = generate_multipartite_deterministic_behaviours(n_o_list, n_p, n_m, subsets_clockwise)
det_behaviours_counter_clockwise = generate_multipartite_deterministic_behaviours(n_o_list, n_p, n_m, subsets_counter_clockwise)

# Combine both sets of deterministic behaviours
all_det_behaviours = vcat(det_behaviours_clockwise, det_behaviours_counter_clockwise)

all_det_behaviours_matrix = reduce(vcat, [Float64.(b)' for b in all_det_behaviours])

8192×64 Matrix{Float64}:
 1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     1.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0     1.0  0.0  0.0  0.0  0.0  0.0  0.0
 1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  1.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0     1.0  0.0  0.0  0.0  0.0  0.0  0.0
 1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0

In [137]:
# Aplicando a função em cada linha
correlator_matrix_ciclo = [
    correlators_from_probabilities(vcat(all_det_behaviours_matrix[i, :]...))
    for i in 1:size(all_det_behaviours_matrix, 1)
];


correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [1.0, 1.0, -1.0, -1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [1.0, -1.0, -1.0, -1.0, 1.0, -1.0, -1.0, -

correlators = [-1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, -1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [1.0, 1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [1.0, 1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [1.0, 1.0, -1.0, 1.0, -1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, -1.0, -1.0, 1.0, -1.0]
correlators = [1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.0]

correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [1.0, 1.0, -1.0, -1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [1.0, 1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, -1.0, -1.0, 1.0, -1.0, -1.0, 1.

correlators = [-1.0, -1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0]
correlators = [1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0, -1.0]
correlators = [1.0, -1.0, 1.0, 1.0, -1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0, -1.0]
correlators = [1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 

correlators = [1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, -1.0, 1.0, -1.0, -1.0]
correlators = [1.0, 1.0, 1.0, -1.0, -1.0, 1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, -1.0, -1.0, 1.0, -1.0, -1.0]
correlators = [1.0, 1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0]
correlators = [1.0, 1.0, 1.0, -1.0, 1.0, -1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, -1.0, 1.0, -1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0]
correlators = [1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, 1.0, -1.0, -1.0, 1.0]
correlators = [1.0, 1.0, -1.0, -1.0, 1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, 1.0, -1.0, 

...

correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [1.0, 1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [1.0, 1.0, -1.0, -1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [1.0, 1.0, -1.0, 1.0, -1.0, 1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, -1.0, 1.0, 1.0, 1.0]
correlators = [1.0, 1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0]
correlators = [1.0, 1.0, -1.0, 1.0, 1.0, -1.0, 1.0, 1.0]
cor

correlators = [1.0, -1.0, 1.0, 1.0, -1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0, -1.0]
correlators = [1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, 1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, 1.0, -1.0, -1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [1.0, 1.0, -1.0, -1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0]

correlators = [1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0]
correlators = [1.0, 1.0, -1.0, -1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [1.0, 1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, -1.0, -1.0, 1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, 1.0, -1.0, -1.0, 1.0]
correlators = [1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, 1.0, -1.0, 1.0, -

correlators = [-1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [1.0, 1.0, -1.0, -1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, -1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [1.0, 1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [1.0, 1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [1.0, 1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [1.0, 1.0, -1.0, 1.0, -1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, -1.0, -1.0, 1.0, -1.0]
correlators = [1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.

correlators = [1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0]
correlators = [-1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0]
correlators = [1.0, -1.0, 1.0, 1.0, -1.0, 1.0, 1.0, 1.0]
correlators = [-1.0, 1.0, 1.0, 1.0, -1.0, 1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [1.0, 1.0, -1.0, -1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0

In [138]:
final_matrix_ciclo = hcat(correlator_matrix_ciclo...)'

8192×8 adjoint(::Matrix{Float64}) with eltype Float64:
 -1.0  -1.0  -1.0  -1.0  -1.0  -1.0  -1.0  -1.0
  1.0   1.0  -1.0  -1.0  -1.0  -1.0  -1.0  -1.0
 -1.0  -1.0   1.0   1.0  -1.0  -1.0  -1.0  -1.0
  1.0   1.0   1.0   1.0  -1.0  -1.0  -1.0  -1.0
 -1.0  -1.0  -1.0  -1.0   1.0   1.0  -1.0  -1.0
  1.0   1.0  -1.0  -1.0   1.0   1.0  -1.0  -1.0
 -1.0  -1.0   1.0   1.0   1.0   1.0  -1.0  -1.0
  1.0   1.0   1.0   1.0   1.0   1.0  -1.0  -1.0
 -1.0  -1.0  -1.0  -1.0  -1.0  -1.0   1.0   1.0
  1.0   1.0  -1.0  -1.0  -1.0  -1.0   1.0   1.0
 -1.0  -1.0   1.0   1.0  -1.0  -1.0   1.0   1.0
  1.0   1.0   1.0   1.0  -1.0  -1.0   1.0   1.0
 -1.0  -1.0  -1.0  -1.0   1.0   1.0   1.0   1.0
  ⋮                             ⋮          
 -1.0  -1.0  -1.0  -1.0   1.0   1.0  -1.0  -1.0
  1.0   1.0  -1.0  -1.0   1.0   1.0  -1.0  -1.0
 -1.0  -1.0   1.0   1.0   1.0   1.0  -1.0  -1.0
  1.0   1.0   1.0   1.0   1.0   1.0  -1.0  -1.0
 -1.0  -1.0  -1.0  -1.0  -1.0  -1.0   1.0   1.0
  1.0   1.0  -1.0  -1.0  -1.0  -1.0  

In [139]:
unique_rows_ciclo = unique(eachrow(final_matrix_ciclo))

128-element Vector{SubArray{Float64, 1, Matrix{Float64}, Tuple{Base.Slice{Base.OneTo{Int64}}, Int64}, true}}:
 [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
 [1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
 [-1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
 [1.0, 1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
 [-1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0, -1.0]
 [1.0, 1.0, -1.0, -1.0, 1.0, 1.0, -1.0, -1.0]
 [-1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0]
 [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0]
 [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0]
 [1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0]
 [-1.0, -1.0, 1.0, 1.0, -1.0, -1.0, 1.0, 1.0]
 [1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0, 1.0]
 [-1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
 ⋮
 [-1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0]
 [1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0]
 [-1.0, -1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0]
 [1.0, 1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0]
 [-1.0, -1.0, -1.0, -1.0, 1.0, -1.0, -1.0, 1.0]
 [1.0, 1.0, -1.0, -1.0, 1.0, -1.0, -1.0, 1.0]
 [-

In [140]:
unique_rows_ciclo == unique_rows

true

In [141]:
function save_matrix_to_txt(unique_rows, filename::String)
    mat = reduce(vcat, unique_rows)
    open(filename, "w") do io
        for row in eachrow(mat)
            println(io, join(row, " "))  # elementos separados por espaço
        end
    end
end

save_matrix_to_txt (generic function with 1 method)

In [142]:
save_matrix_to_txt(unique(eachrow(correlator_matrix)), "correlators_unique.txt")

In [143]:
using LinearAlgebra  # Import required for the dot function

# Scenario parameters
n_p = 3
n_o_list = 2   # Number of outcomes for each party
n_m = 2   # Number of inputs for each party

# Conditional dependencies for each party (Boring scenario)
subsets_list = [[1], [1, 2], [1, 2, 3]]

# Generate all multipartite deterministic behaviours
det_behaviours_boring = generate_multipartite_deterministic_behaviours(n_o_list, n_p, n_m, subsets_list)

det_behaviours_matrix_boring = reduce(vcat, [Float64.(b)' for b in det_behaviours_boring])

16384×64 Matrix{Float64}:
 1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     1.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0     1.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  1.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  1.0  0.0  0.0  0.0  0.0  …  0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  1.0  0.0  0.0  0.0  0.0  0.0     1.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  0.0  1.0  0.0  0.0  0.0  0.0     1.0  0.0  0.0  0.0  0.0  0.0  0.0
 1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0     0.0  0.0  0.0  0.0  0.0  0.0  0.0
 1.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  …  1.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  1.0  0.0  0.0  0.0  0.0  0.0  0.0     1.0  0.0  0.0  0.0  0.0  0.0  0.0
 0.0  0.0  1.0  0.0  0.0  0.0  0.0  0.

In [144]:
# Aplicando a função em cada linha
correlator_matrix_boring = [
    correlators_from_probabilities(vcat(det_behaviours_matrix_boring[i, :]...))
    for i in 1:size(det_behaviours_matrix_boring, 1)
]


correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [1.0, 1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0, -

correlators = [1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [-1.0, 1.0, -1.0, -1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [-1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [-1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [-1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [-1.0, 1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [-1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [-1.0, 1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [-1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]


correlators = [-1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [1.0, -1.0, 1.0, -1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [-1.0, 1.0, -1.0, 1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [-1.0, 1.0, -1.0, 1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [1.0, -1.0, 1.0, -1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [-1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [-1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [1.0, -1.0, -1.0, 1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [-1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [1.0, -1.0, -1.0, 1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [1.0, -1.0, -1.0, 1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [-1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [1.0, -1.0, -1.0, 1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [-1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [-1.0, 1.0, -1.0, 1.0, -1.0, -1.0, 1.0, 1.

correlators = [-1.0, -1.0, 1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [1.0, 1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [1.0, 1.0, 1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, -1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [1.0, 1.0, -1.0, 1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, -1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0, 1.0]


correlators = [-1.0, 1.0, 1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [-1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [1.0, -1.0, -1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [1.0, -1.0, 1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [-1.0, 1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [1.0, -1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [-1.0, 1.0, -1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [1.0, 1.0, -1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [1.0, 1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [1.0, 1.0, 1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
correlators = [1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1

...

correlators = [1.0, -1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, 1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, 1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [1.0, -1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, 1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [1.0, -1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [1.0, -1.0, 1.0, -1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, 1.0, -1.0, 1.0, -1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [1.0, -1.0, -1.0, 1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.0]
correlators = [1.0, -1.0, -1.0, 1.0, -1.0, -1.0, 1.0, -1.0]
correlators = [1.0, -1.0, -1.0, 1.0, 1.0, 1.0, -1.0, 1.0

correlators = [-1.0, -1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [1.0, 1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [1.0, 1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [1.0, 1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [1.0, 1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [1.0, 1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [1.0, 1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [-1.0, -1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [1.0, 1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -

Excessive output truncated after 524331 bytes.


correlators = [-1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.0]
correlators = [1.0, -1.0, -1.0, 1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, 1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.0]
correlators = [1.0, -1.0, -1.0, 1.0, -1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, 1.0, 1.0, -1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [1.0, -1.0, -1.0, 1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0, -1.0]
correlators = [-1.0, 1.0, -1.0, 1.0, -1.0, -1.0, 1.0, -1.0]
correlators = [1.0, -1.0, 1.0, -1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0, 1.0]
correlators = [-1.0, 1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [1.0, -1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, 1.0, -1.0, 1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [1.0, -1.0, 1.0, -1.0, -1.0, -1.0, -1.0, 1.0]
correlators = [1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0, -1.0]
correlators = [-1.0, 1.0, 1.0, -1.0, 1.0, 1.0, 1.0, -1.0]

16384-element Vector{Vector{Float64}}:
 [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
 [1.0, 1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
 [-1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
 [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
 [1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
 [-1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
 [1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
 [-1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
 [-1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
 [1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
 [-1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
 [1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
 [1.0, 1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
 ⋮
 [-1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
 [1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
 [-1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
 [1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
 [1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
 [-1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
 [1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
 [-1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.

In [145]:
final_matrix_boring = hcat(correlator_matrix_boring...)'

16384×8 adjoint(::Matrix{Float64}) with eltype Float64:
 -1.0  -1.0  -1.0  -1.0  -1.0  -1.0  -1.0  -1.0
  1.0   1.0   1.0   1.0  -1.0  -1.0  -1.0  -1.0
 -1.0  -1.0  -1.0  -1.0   1.0   1.0   1.0   1.0
  1.0   1.0   1.0   1.0   1.0   1.0   1.0   1.0
  1.0   1.0  -1.0  -1.0  -1.0  -1.0  -1.0  -1.0
 -1.0  -1.0   1.0   1.0  -1.0  -1.0  -1.0  -1.0
  1.0   1.0  -1.0  -1.0   1.0   1.0   1.0   1.0
 -1.0  -1.0   1.0   1.0   1.0   1.0   1.0   1.0
 -1.0  -1.0   1.0   1.0  -1.0  -1.0  -1.0  -1.0
  1.0   1.0  -1.0  -1.0  -1.0  -1.0  -1.0  -1.0
 -1.0  -1.0   1.0   1.0   1.0   1.0   1.0   1.0
  1.0   1.0  -1.0  -1.0   1.0   1.0   1.0   1.0
  1.0   1.0   1.0   1.0  -1.0  -1.0  -1.0  -1.0
  ⋮                             ⋮          
 -1.0  -1.0   1.0   1.0  -1.0  -1.0  -1.0  -1.0
  1.0   1.0  -1.0  -1.0  -1.0  -1.0  -1.0  -1.0
 -1.0  -1.0   1.0   1.0   1.0   1.0   1.0   1.0
  1.0   1.0  -1.0  -1.0   1.0   1.0   1.0   1.0
  1.0   1.0  -1.0  -1.0  -1.0  -1.0  -1.0  -1.0
 -1.0  -1.0   1.0   1.0  -1.0  -1.0 

In [146]:
unique_rows_boring = unique(eachrow(final_matrix_boring))

256-element Vector{SubArray{Float64, 1, Matrix{Float64}, Tuple{Base.Slice{Base.OneTo{Int64}}, Int64}, true}}:
 [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
 [1.0, 1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
 [-1.0, -1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
 [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
 [1.0, 1.0, -1.0, -1.0, -1.0, -1.0, -1.0, -1.0]
 [-1.0, -1.0, 1.0, 1.0, -1.0, -1.0, -1.0, -1.0]
 [1.0, 1.0, -1.0, -1.0, 1.0, 1.0, 1.0, 1.0]
 [-1.0, -1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0]
 [-1.0, -1.0, -1.0, -1.0, 1.0, 1.0, -1.0, -1.0]
 [1.0, 1.0, 1.0, 1.0, 1.0, 1.0, -1.0, -1.0]
 [-1.0, -1.0, -1.0, -1.0, -1.0, -1.0, 1.0, 1.0]
 [1.0, 1.0, 1.0, 1.0, -1.0, -1.0, 1.0, 1.0]
 [1.0, 1.0, -1.0, -1.0, 1.0, 1.0, -1.0, -1.0]
 ⋮
 [-1.0, 1.0, 1.0, -1.0, 1.0, -1.0, 1.0, -1.0]
 [1.0, -1.0, -1.0, 1.0, 1.0, -1.0, 1.0, -1.0]
 [-1.0, 1.0, 1.0, -1.0, -1.0, 1.0, -1.0, 1.0]
 [1.0, -1.0, -1.0, 1.0, -1.0, 1.0, -1.0, 1.0]
 [1.0, -1.0, 1.0, -1.0, -1.0, 1.0, 1.0, -1.0]
 [-1.0, 1.0, -1.0, 1.0, -1.0, 1.0, 1.0, -1.0]
 [1.0, -1.0, 

In [147]:
unique_rows_boring == unique_rows

false

### Como o conjunto boring é maior que o do ciclo, talvez vale que existam desigualdades full correlators que são violadas.

In [148]:
import Pkg; Pkg.add("Combinatorics")
using Combinatorics

function generate_index_symmetries()
    base_indices = [(x, y, z) for x in 0:1, y in 0:1, z in 0:1]

    index_map = Dict{Tuple{Int,Int,Int}, String}()
    for (x, y, z) in base_indices
        index_map[(x, y, z)] = "E$(x)$(y)$(z)"
    end

    symmetries = Set{Vector{String}}()

    for perm in permutations((1, 2, 3))  # all 6 permutations of x, y, z
        for flip_x in [0, 1], flip_y in [0, 1], flip_z in [0, 1]
            current = String[]
            for (x, y, z) in base_indices
                coords = (x, y, z)
                permuted = (coords[perm[1]], coords[perm[2]], coords[perm[3]])
                flipped = (
                    permuted[1] ⊻ flip_x,
                    permuted[2] ⊻ flip_y,
                    permuted[3] ⊻ flip_z,
                )
                push!(current, index_map[flipped])
            end
            push!(symmetries, current)
        end
    end

    return collect(symmetries)
end


    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
     Project No packages added to or removed from `~/.julia/environments/v1.12/Project.toml`
    Manifest No packages added to or removed from `~/.julia/environments/v1.12/Manifest.toml`


generate_index_symmetries (generic function with 1 method)

In [149]:
symmetries = generate_index_symmetries()
println("Total symmetries: ", length(symmetries))
for s in symmetries
    println(join(s, " "))
end


Total symmetries: 48
E001 E000 E101 E100 E011 E010 E111 E110
E100 E000 E101 E001 E110 E010 E111 E011
E111 E101 E110 E100 E011 E001 E010 E000
E001 E011 E000 E010 E101 E111 E100 E110
E101 E111 E100 E110 E001 E011 E000 E010
E010 E011 E000 E001 E110 E111 E100 E101
E011 E010 E111 E110 E001 E000 E101 E100
E101 E100 E111 E110 E001 E000 E011 E010
E010 E110 E000 E100 E011 E111 E001 E101
E010 E110 E011 E111 E000 E100 E001 E101
E001 E101 E000 E100 E011 E111 E010 E110
E110 E111 E010 E011 E100 E101 E000 E001
E000 E010 E100 E110 E001 E011 E101 E111
E110 E111 E100 E101 E010 E011 E000 E001
E001 E101 E011 E111 E000 E100 E010 E110
E100 E110 E000 E010 E101 E111 E001 E011
E110 E010 E111 E011 E100 E000 E101 E001
E001 E011 E101 E111 E000 E010 E100 E110
E000 E100 E010 E110 E001 E101 E011 E111
E100 E000 E110 E010 E101 E001 E111 E011
E101 E111 E001 E011 E100 E110 E000 E010
E000 E010 E001 E011 E100 E110 E101 E111
E100 E101 E000 E001 E110 E111 E010 E011
E011 E001 E010 E000 E111 E101 E110 E100
E011 E001 E111 E101

### Desigualdades do panda:

Reduced Inequalities:
-E111 <= 1

E000 <= 1

1. E000 -E001 -E010 -E011 -E100 -E101 -E110 -E111 <= 6

2. E000 +E001 +E010 -E011 -E100 -E101 -E110 -E111 <= 6

3. E000 -E001 -E010 +E011 -E100 +E101 -E110 -E111 <= 6

4. E000 +E001 -E010 -E011 -E100 -E101 +E110 -E111 <= 6

5. E000 +E001 +E010 -E011 -E100 +E101 +E110 -E111 <= 6

6. E000 +E001 +E010 -E011 +E100 -E101 -E110 +E111 <= 6

7. E000 +E001 +E010 +E011 +E100 -E101 -E110 -E111 <= 6

8. E000 +E001 +E010 +E011 +E100 +E101 +E110 -E111 <= 6

### Vamos testar as desigualdades

In [150]:
using QuantumNPA  # Load QuantumNPA package for NPA hierarchy computations

@dichotomic A[1:2] B[1:2] C[1:2];  # Define dichotomic observables A, B, C with 2 inputs each

#1. E000 -E001 -E010 +E011 -E100 +E101 -E110 -E111 <= 6
S = A[1]*B[1]*C[1] +A[1]*B[1]*C[2] +A[1]*B[2]*C[1] -A[1]*B[2]*C[2] - A[2]*B[1]*C[1] -
A[2]*B[1]*C[2] -A[2]*B[2]*C[1] -A[2]*B[2]*C[2] 

NPA_lvl4_cor = npa_max(S, "4")  # Compute the maximal quantum value of S using NPA hierarchy at level 4

5.997755573617901

In [151]:
using QuantumNPA  # Load QuantumNPA package for NPA hierarchy computations

@dichotomic A[1:2] B[1:2] C[1:2];  # Define dichotomic observables A, B, C with 2 inputs each

#2. E000 +E001 +E010 -E011 -E100 -E101 -E110 -E111 <= 6
S = A[1]*B[1]*C[1] +A[1]*B[1]*C[2] +A[1]*B[2]*C[1] -A[1]*B[2]*C[2] -
A[2]*B[1]*C[1] -A[2]*B[1]*C[2] -A[2]*B[2]*C[1] -A[2]*B[2]*C[2] 

NPA_lvl4_cor = npa_max(S, "4")  # Compute the maximal quantum value of S using NPA hierarchy at level 4

5.997755573617901

In [152]:
using QuantumNPA  # Load QuantumNPA package for NPA hierarchy computations

@dichotomic A[1:2] B[1:2] C[1:2];  # Define dichotomic observables A, B, C with 2 inputs each

#3. E000 -E001 -E010 +E011 -E100 +E101 -E110 -E111 <= 6
S = A[1]*B[1]*C[1] -A[1]*B[1]*C[2] -A[1]*B[2]*C[1] +A[1]*B[2]*C[2] -
A[2]*B[1]*C[1] +A[2]*B[1]*C[2] -A[2]*B[2]*C[1] -A[2]*B[2]*C[2] 

NPA_lvl4_cor = npa_max(S, "4")  # Compute the maximal quantum value of S using NPA hierarchy at level 4

5.997755573605945

In [153]:
using QuantumNPA  # Load QuantumNPA package for NPA hierarchy computations

@dichotomic A[1:2] B[1:2] C[1:2];  # Define dichotomic observables A, B, C with 2 inputs each

#4. E000 +E001 -E010 -E011 -E100 -E101 +E110 -E111 <= 6
S = A[1]*B[1]*C[1] +A[1]*B[1]*C[2] -A[1]*B[2]*C[1] -A[1]*B[2]*C[2] -
A[2]*B[1]*C[1] -A[2]*B[1]*C[2] +A[2]*B[2]*C[1] -A[2]*B[2]*C[2] 

NPA_lvl4_cor = npa_max(S, "4")  # Compute the maximal quantum value of S using NPA hierarchy at level 4

5.997755573595232

In [154]:
using QuantumNPA  # Load QuantumNPA package for NPA hierarchy computations

@dichotomic A[1:2] B[1:2] C[1:2];  # Define dichotomic observables A, B, C with 2 inputs each

#5. E000 +E001 +E010 -E011 -E100 +E101 +E110 -E111 <= 6
S = A[1]*B[1]*C[1] +A[1]*B[1]*C[2] +A[1]*B[2]*C[1] -A[1]*B[2]*C[2] -
A[2]*B[1]*C[1] +A[2]*B[1]*C[2] +A[2]*B[2]*C[1] -A[2]*B[2]*C[2] 

NPA_lvl4_cor = npa_max(S, "4")  # Compute the maximal quantum value of S using NPA hierarchy at level 4

5.997755573587398

In [155]:
using QuantumNPA  # Load QuantumNPA package for NPA hierarchy computations

@dichotomic A[1:2] B[1:2] C[1:2];  # Define dichotomic observables A, B, C with 2 inputs each

#6. E000 +E001 +E010 -E011 +E100 -E101 -E110 +E111 <= 6
S = A[1]*B[1]*C[1] +A[1]*B[1]*C[2] +A[1]*B[2]*C[1] -A[1]*B[2]*C[2] +
A[2]*B[1]*C[1] -A[2]*B[1]*C[2] -A[2]*B[2]*C[1] +A[2]*B[2]*C[2] 

NPA_lvl4_cor = npa_max(S, "4")  # Compute the maximal quantum value of S using NPA hierarchy at level 4

5.997755573606857

In [156]:
using QuantumNPA  # Load QuantumNPA package for NPA hierarchy computations

@dichotomic A[1:2] B[1:2] C[1:2];  # Define dichotomic observables A, B, C with 2 inputs each

#7. E000 +E001 +E010 +E011 +E100 -E101 -E110 -E111 <= 6
S = A[1]*B[1]*C[1] +A[1]*B[1]*C[2] +A[1]*B[2]*C[1] +A[1]*B[2]*C[2] +
A[2]*B[1]*C[1] -A[2]*B[1]*C[2] -A[2]*B[2]*C[1] -A[2]*B[2]*C[2] 

NPA_lvl4_cor = npa_max(S, "4")  # Compute the maximal quantum value of S using NPA hierarchy at level 4

5.997755573605607

In [157]:
using QuantumNPA  # Load QuantumNPA package for NPA hierarchy computations

@dichotomic A[1:2] B[1:2] C[1:2];  # Define dichotomic observables A, B, C with 2 inputs each

#8. E000 +E001 +E010 +E011 +E100 +E101 +E110 -E111 <= 6
S = A[1]*B[1]*C[1] +A[1]*B[1]*C[2] +A[1]*B[2]*C[1] +A[1]*B[2]*C[2] +
A[2]*B[1]*C[1] +A[2]*B[1]*C[2] +A[2]*B[2]*C[1] -A[2]*B[2]*C[2] 

NPA_lvl4_cor = npa_max(S, "4")  # Compute the maximal quantum value of S using NPA hierarchy at level 4

5.997755573582024